In [ ]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v4.1 — Vectorized + Thread-Safe Jupyter)
# ============================================================================
# Changes from v4:
#   - run_parallel_safe auto-detects Jupyter/Colab and uses ThreadPoolExecutor
#     instead of multiprocessing.Pool (avoids WinError 5 on Windows)
#   - Uses processes in terminal, threads in Jupyter
#   - No more PermissionError in Jupyter on Windows
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import pickle
import multiprocessing as mp
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats
from scipy.spatial.distance import cdist

# Suppress Axes3D warning
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= 0:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30):
    spread = abs(x) * delta if x != 0 else delta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS (VECTORIZED)
# ============================================================================

def fuzzy_dominates(F1, F2):
    weak_all, strict_any = True, False
    for f1, f2 in zip(F1, F2):
        if credibility_leq_triangular(f1, f2) < 0.5:
            weak_all = False
            break
        f1_lower = TriangularFuzzyNumber(f1.aL, f1.aL, f1.aL)
        if credibility_leq_triangular(f1_lower, f2) > 0.5:
            strict_any = True
    return weak_all and strict_any


def fast_nondominated_sort_fuzzy(F):
    """Vectorized non-dominated sort. Semantics identical to scalar version."""
    N = F.shape[0]
    if N == 0:
        return []

    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L

    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    with np.errstate(divide='ignore', invalid='ignore'):
        credA = np.clip(np.where(den > 0,
                                 num / np.where(den > 0, den, 1.0), 0.0),
                        0.0, 1.0)
    degenerate = den <= 0
    credA = np.where(degenerate,
                     (U[:, None, :] <= L[None, :, :]).astype(float),
                     credA)

    denB = W[None, :, :]
    with np.errstate(divide='ignore', invalid='ignore'):
        credB = np.clip(np.where(denB > 0,
                                 num / np.where(denB > 0, denB, 1.0), 0.0),
                        0.0, 1.0)
    degB = np.broadcast_to(denB <= 0, num.shape)
    credB = np.where(degB,
                     (L[:, None, :] <= L[None, :, :]).astype(float),
                     credB)

    dom = np.all(credA >= 0.5, axis=2) & np.any(credB > 0.5, axis=2)
    np.fill_diagonal(dom, False)

    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]

    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= 0:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.zeros((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = [trfn.a1, trfn.a2, trfn.a3, trfn.a4]
        return fuzzy


# --- ZDT ---
class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


# --- DTLZ ---
class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


# --- WFG (full) ---
def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return r * (0.1 * (1 - y) * (1 - B - y) ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


# --- CEC09 ---
class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


# --- MaF ---
def _maf_dtlz_base(x, n_obj):
    n, m = len(x), n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


# --- Scalability ---
class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND Aα-NSGA-II
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.zeros((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = float(min(a1, a2, a3, a4))
            aU = float(max(a1, a2, a3, a4))
            aM = float(0.5 * (a2 + a3))
            aM = max(aL, min(aM, aU))
            self.fuzzy_obj[i] = [aL, aM, aU]

    def alpha_weighted_obj(self):
        out = np.zeros(self.fuzzy_obj.shape[0])
        for i, (L, M, U) in enumerate(self.fuzzy_obj):
            out[i] = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return out


class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=100, n_gen=500,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.4, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        F1 = [TriangularFuzzyNumber(*ind1.fuzzy_obj[i])
              for i in range(ind1.fuzzy_obj.shape[0])]
        F2 = [TriangularFuzzyNumber(*ind2.fuzzy_obj[i])
              for i in range(ind2.fuzzy_obj.shape[0])]
        return fuzzy_dominates(F1, F2)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in [-0.05, -0.02, 0.02, 0.05]:
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = self.rng.integers(len(x_new))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: BASELINES
# ============================================================================

class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=100, n_gen=500,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineCrispExpected(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 1.0
        super().__init__(*args, **kwargs)


class BaselineFMEA(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 0.60
        super().__init__(*args, **kwargs)


class BaselineGridSearchFixedAlpha:
    def __init__(self, benchmark, alpha_grid=None, **kwargs):
        self.benchmark = benchmark
        self.alpha_grid = alpha_grid if alpha_grid is not None else np.arange(0.5, 1.01, 0.1)
        self.kwargs = kwargs
        self.population = []
        self.best_alpha = 0.75

    def run(self):
        best_pop, best_hv = None, -np.inf
        best_alpha = 0.75
        for alpha in self.alpha_grid:
            algo = BaselineNSGA2(self.benchmark, alpha=float(alpha), **self.kwargs)
            pop = algo.run()
            pf = [ind for ind in pop if ind.rank == 0]
            if len(pf) == 0:
                continue
            F = np.array([ind.alpha_weighted_obj() for ind in pf])
            ref = np.max(F, axis=0) * 1.1 + 1.0
            hv = hypervolume(F, ref)
            if hv > best_hv:
                best_hv = hv
                best_pop = pop
                best_alpha = float(alpha)
        self.population = best_pop if best_pop is not None else []
        self.best_alpha = best_alpha
        return self.population

    def get_alphas(self):
        return np.array([self.best_alpha])

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 7: METRICS (RAW α-WEIGHTED HV)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=5000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


def _hv_mc(F, ref, n_samples=5000):
    lows = np.min(F, axis=0)
    samples = np.random.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def compute_pareto_hv(population, use_alpha=True):
    """Compute HV using α-weighted objectives (raw, unnormalized)."""
    pf = [ind for ind in population if ind.rank == 0]
    if len(pf) == 0:
        return 0.0, np.array([])
    if use_alpha:
        F = np.array([ind.alpha_weighted_obj() for ind in pf])
    else:
        F = np.array([[(L + 4 * M + U) / 6.0 for L, M, U in ind.fuzzy_obj]
                      for ind in pf])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ============================================================================
# PART 8: GLOBAL CONFIGURATION
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 30,
    'pop_size': 100,
    'n_gen': 500,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'results_dir': 'parallel_results',
    'figures_dir': 'parallel_figures',
    'latex_dir': 'parallel_latex',
    'supp_latex_dir': 'supplementary_latex',
    'reference_algo': 'Aα-NSGA-II',
    'bootstrap_samples': 10000,
}

ALGO_NAMES = [
    'Aα-NSGA-II', 'NSGA-II', 'F-MEA', 'Crisp-Expected',
    'Grid-Search Fixed-α', 'Self-Adaptive Mutation',
]

ALGO_COLORS = {
    'Aα-NSGA-II': '#1f77b4',
    'NSGA-II': '#ff7f0e',
    'F-MEA': '#2ca02c',
    'Crisp-Expected': '#d62728',
    'Grid-Search Fixed-α': '#9467bd',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II', AAlphaNSGA2, {**common}),
        ('NSGA-II', BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('F-MEA', BaselineFMEA, {**common, 'alpha': 0.60}),
        ('Crisp-Expected', BaselineCrispExpected, {**common}),
        ('Grid-Search Fixed-α', BaselineGridSearchFixedAlpha, {**common}),
        ('Self-Adaptive Mutation', BaselineSelfAdaptiveMutation,
         {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 9: PARALLEL / SERIAL EXECUTION (THREAD-SAFE JUPYTER VERSION)
# ============================================================================

def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    benchmark = get_extended_benchmark(prob_name)
    algos = get_algo_config(pop_size, n_gen)

    results = {}
    for algo_name, algo_cls, kwargs in algos:
        hvs_raw, alphas = [], []
        errors = 0
        for run_idx in range(n_runs):
            seed = seed_base + run_idx
            try:
                algo = algo_cls(benchmark, seed=seed, **kwargs)
                pop = algo.run()
            except Exception:
                errors += 1
                continue

            hv_raw, F = compute_pareto_hv(pop, use_alpha=True)
            if len(F) == 0:
                continue
            hvs_raw.append(hv_raw)

            if hasattr(algo, 'get_alphas'):
                alphas.append(float(np.mean(algo.get_alphas())))

        results[algo_name] = {
            'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
            'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
            'hv_raw_all': hvs_raw,
            'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
            'alpha_mean_all': alphas,
            'n_errors': errors,
        }
    return (prob_name, results)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base, verbose=True):
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(problems)]

    if verbose:
        print(f"\n[SERIAL MODE] Total problems: {len(tasks)}")

    t0 = time.time()
    all_results = {}
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if verbose:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        task_t0 = time.time()
        prob_name, results = run_single_problem(task)
        all_results[prob_name] = results
        if verbose:
            elapsed = time.time() - t0
            task_elapsed = time.time() - task_t0
            avg = elapsed / i
            eta = avg * (len(tasks) - i)
            print(f"  [{i}/{len(tasks)}] {prob_name} done in "
                  f"{task_elapsed:.1f}s (ETA: {eta/60:.1f} min)", flush=True)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel_safe(problems, n_runs, pop_size, n_gen, seed_base,
                      n_workers=None, verbose=True):
    """
    Parallel execution with automatic environment detection:
      - Jupyter/Colab: ThreadPoolExecutor (no handle duplication errors)
      - Terminal: multiprocessing.Pool (real parallelism)
    """
    if n_workers is None:
        n_workers = min(8, max(1, mp.cpu_count() - 1))
    n_workers = min(n_workers, len(problems), 8)

    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(problems)]

    in_jupyter = ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)

    if verbose:
        print(f"\nTotal problems: {len(tasks)}")
        print(f"Workers: {n_workers}")
        print(f"Backend: {'threads (Jupyter)' if in_jupyter else 'processes (terminal)'}")

    t0 = time.time()
    all_results = {}

    # --------------------------------------------------------------------
    # JUPYTER: use ThreadPoolExecutor (avoids WinError 5)
    # --------------------------------------------------------------------
    if in_jupyter:
        try:
            with ThreadPoolExecutor(max_workers=n_workers) as executor:
                future_to_task = {executor.submit(run_single_problem, t): t
                                  for t in tasks}
                completed = 0
                for future in as_completed(future_to_task):
                    try:
                        prob_name, results = future.result()
                    except Exception as e:
                        if verbose:
                            print(f"  Task failed: {e}")
                        continue
                    completed += 1
                    all_results[prob_name] = results
                    if verbose:
                        elapsed = time.time() - t0
                        rate = completed / max(elapsed, 1e-6)
                        eta = (len(tasks) - completed) / max(rate, 1e-6)
                        print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                              f"(ETA: {eta/60:.1f} min)", flush=True)
        except Exception as e:
            if verbose:
                print(f"\n⚠ Thread pool failed ({e}). Falling back to serial.")
            remaining = [t[0] for t in tasks if t[0] not in all_results]
            serial_results = run_serial(
                remaining, n_runs, pop_size, n_gen, seed_base, verbose)
            all_results.update(serial_results)

        if verbose:
            print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
        return all_results

    # --------------------------------------------------------------------
    # TERMINAL: use multiprocessing.Pool (real parallelism)
    # --------------------------------------------------------------------
    try:
        ctx = mp.get_context('spawn')
        pool = ctx.Pool(processes=n_workers)
        iterator = pool.imap_unordered(run_single_problem, tasks, chunksize=1)
        completed = 0
        while completed < len(tasks):
            try:
                prob_name, results = iterator.next(timeout=1800)
            except mp.TimeoutError:
                if verbose:
                    print(f"\n⚠ Timeout. Falling back to serial.")
                pool.terminate()
                pool.join()
                remaining = [t[0] for t in tasks if t[0] not in all_results]
                if remaining:
                    serial_results = run_serial(
                        remaining, n_runs, pop_size, n_gen, seed_base, verbose)
                    all_results.update(serial_results)
                return all_results
            completed += 1
            all_results[prob_name] = results
            if verbose:
                elapsed = time.time() - t0
                rate = completed / max(elapsed, 1e-6)
                eta = (len(tasks) - completed) / max(rate, 1e-6)
                print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                      f"(ETA: {eta/60:.1f} min)", flush=True)
        pool.close()
        pool.join()
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel failed ({e}). Falling back to serial.")
        try:
            pool.terminate(); pool.join()
        except Exception:
            pass
        remaining = [t[0] for t in tasks if t[0] not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base, verbose)
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICAL TESTS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=10000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    means = np.array([np.mean(rng.choice(data, n, replace=True))
                      for _ in range(n_samples)])
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}

    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Step 1: Shapiro-Wilk Normality Test (on rank data) ---")
    print(f"{'Algorithm':<28} | {'W':>10} | {'p-value':>12} | {'Normal':>8}")
    print("-" * 78)
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<28} | {W:>10.4f} | {p:>12.4e} | "
              f"{'YES' if p > 0.05 else 'NO':>8}")

    print(f"\n--- Step 2: Friedman Test (on ranks) ---")
    try:
        stat, p = stats.friedmanchisquare(
            *[agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0])
        print(f"Chi-square = {stat:.4f}, p-value = {p:.6e}")
        print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Step 3: Kruskal-Wallis Test (on normalized HV) ---")
    try:
        stat, p = stats.kruskal(*[agg_hv[a] for a in algo_names
                                  if len(agg_hv[a]) > 0])
        print(f"H = {stat:.4f}, p-value = {p:.6e}")
        print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Step 4: Pairwise Wilcoxon (with Bonferroni) on ranks ---")
    print(f"\n{'Comparison':<42} | {'p-value':>12} | {'Cohen d':>10} | "
          f"{'Cliff δ':>10} | {'Sig.':>5}")
    print("-" * 92)

    n_comparisons = len(algo_names) - 1
    threshold = 0.05 / n_comparisons

    pairwise_results = {}
    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{reference} vs. {algo:<26} | {p:>12.4e} | "
                  f"{d:>10.4f} | {cl:>10.4f} | {sig:>5}")
        print(f"\nBonferroni-corrected threshold: α' = {threshold:.6f}")

    print(f"\n--- Step 5: Bootstrap 95% Confidence Intervals ---")
    print(f"\n{'Algorithm':<28} | {'Mean':>8} | {'95% CI':>22} | {'Width':>8}")
    print("-" * 78)
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<28} | {mean:>8.4f} | "
              f"[{lo:>7.4f}, {hi:>7.4f}] | {hi-lo:>8.4f}")

    print(f"\n--- Step 6: Mean Ranks (lower = better) ---")
    print(f"\n{'Algorithm':<28} | {'Mean Rank':>10} | {'Std':>8} | {'Median':>8}")
    print("-" * 78)
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<28} | {np.mean(agg_rank[algo]):>10.2f} | "
                  f"{np.std(agg_rank[algo]):>8.2f} | "
                  f"{np.median(agg_rank[algo]):>8.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: MAIN VISUALIZATIONS
# ============================================================================

def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(10, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)

    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)

    ax.set_title('Normalized α-weighted HV per Problem (best = 1.0)',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, label='Normalized α-weighted HV', shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))

    fig, ax = plt.subplots(figsize=(12, 5))
    n_families = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_families)

    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)

        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title('Algorithm Performance by Problem Family', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)

    fig, ax = plt.subplots(figsize=(9, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)

    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)

    if not alpha_by_problem:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Converged α', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")

    sizes = []
    for p in scal_probs:
        try: s = int(p.replace('FScal', ''))
        except ValueError: s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]

    fig, ax = plt.subplots(figsize=(8, 5))
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                if best > 0:
                    hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best)
                else:
                    hvs.append(0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=10)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ============================================================================
# PART 12: MAIN LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (with Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w') as f:
        f.write('\n'.join(lines))

    print(f"  Generated 4 LaTeX tables in {save_dir}/")


# ============================================================================
# PART 13: PER-PROBLEM TABLE
# ============================================================================

def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]

    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))

    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')

    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])

    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")

    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    df = pd.DataFrame(csv_rows)
    df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'), index=False)
    print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 14: SETTINGS, MAIN, AND ENTRY POINT
# ============================================================================

JUPYTER_MODE = 'full'
JUPYTER_USE_PARALLEL = True
JUPYTER_N_WORKERS = 4


def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir'] = 'test_results'
        CONFIG['figures_dir'] = 'test_figures'
        CONFIG['latex_dir'] = 'test_latex'
        CONFIG['supp_latex_dir'] = 'test_supplementary_latex'

    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = min(8, max(1, mp.cpu_count() - 1))


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v4.1 — Vectorized + Thread-Safe)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    in_jupyter = ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    if in_jupyter:
        print(f"Environment: Jupyter/Colab (will use threads)")
    else:
        print(f"Environment: Terminal (will use processes)")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir']]:
        os.makedirs(d, exist_ok=True)

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint from {checkpoint_path}...")
        with open(checkpoint_path, 'rb') as f:
            all_results = pickle.load(f)
        print(f"  Loaded {len(all_results)} problems")
    else:
        if CONFIG['use_parallel']:
            all_results = run_parallel_safe(
                CONFIG['problems'],
                CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                CONFIG['n_workers'], verbose=True,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'],
                CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                verbose=True,
            )
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(all_results, f)
        print(f"  Checkpoint saved")

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    rows = []
    for prob_name, results in all_results.items():
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'), index=False)
    print(f"\n  CSV saved to {CONFIG['results_dir']}/results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


# ============================================================================
# ENTRY POINT — JUPYTER/CLI SAFE
# ============================================================================

if __name__ == "__main__":
    try:
        mp.freeze_support()
    except Exception:
        pass

    if 'ipykernel' not in sys.modules and 'google.colab' not in sys.modules:
        import argparse
        parser = argparse.ArgumentParser(description='Aα-NSGA-II Pipeline v4.1')
        parser.add_argument('--test', action='store_true')
        parser.add_argument('--serial', action='store_true')
        parser.add_argument('--workers', type=int, default=None)
        parser.add_argument('--runs', type=int, default=None)
        parser.add_argument('--pop', type=int, default=None)
        parser.add_argument('--gen', type=int, default=None)
        args, _ = parser.parse_known_args()

        mode = 'test' if args.test else 'full'
        use_parallel = not args.serial
        main(mode=mode, use_parallel=use_parallel,
             n_workers=args.workers, runs=args.runs,
             pop=args.pop, gen=args.gen)
    else:
        main(mode=JUPYTER_MODE,
             use_parallel=JUPYTER_USE_PARALLEL,
             n_workers=JUPYTER_N_WORKERS)

Aα-NSGA-II: COMPLETE PIPELINE (v4.1 — Vectorized + Thread-Safe)
Problems: 49 | Runs: 30 | Pop: 100 | Gen: 500 | Workers: 4 | Mode: parallel
Environment: Jupyter/Colab (will use threads)

Total problems: 49
Workers: 4
Backend: threads (Jupyter)


In [ ]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v4.2 — UTF-8 Encoding Fix)
# ============================================================================
# Changes from v4.1:
#   - All file writes now use encoding='utf-8' (fixes UnicodeEncodeError
#     when writing Greek letters like α to LaTeX/CSV files on Windows)
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import pickle
import multiprocessing as mp
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats
from scipy.spatial.distance import cdist

# Suppress Axes3D warning
import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= 0:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30):
    spread = abs(x) * delta if x != 0 else delta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS (VECTORIZED)
# ============================================================================

def fuzzy_dominates(F1, F2):
    weak_all, strict_any = True, False
    for f1, f2 in zip(F1, F2):
        if credibility_leq_triangular(f1, f2) < 0.5:
            weak_all = False
            break
        f1_lower = TriangularFuzzyNumber(f1.aL, f1.aL, f1.aL)
        if credibility_leq_triangular(f1_lower, f2) > 0.5:
            strict_any = True
    return weak_all and strict_any


def fast_nondominated_sort_fuzzy(F):
    """Vectorized non-dominated sort. Semantics identical to scalar version."""
    N = F.shape[0]
    if N == 0:
        return []

    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L

    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    with np.errstate(divide='ignore', invalid='ignore'):
        credA = np.clip(np.where(den > 0,
                                 num / np.where(den > 0, den, 1.0), 0.0),
                        0.0, 1.0)
    degenerate = den <= 0
    credA = np.where(degenerate,
                     (U[:, None, :] <= L[None, :, :]).astype(float),
                     credA)

    denB = W[None, :, :]
    with np.errstate(divide='ignore', invalid='ignore'):
        credB = np.clip(np.where(denB > 0,
                                 num / np.where(denB > 0, denB, 1.0), 0.0),
                        0.0, 1.0)
    degB = np.broadcast_to(denB <= 0, num.shape)
    credB = np.where(degB,
                     (L[:, None, :] <= L[None, :, :]).astype(float),
                     credB)

    dom = np.all(credA >= 0.5, axis=2) & np.any(credB > 0.5, axis=2)
    np.fill_diagonal(dom, False)

    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]

    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= 0:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.zeros((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = [trfn.a1, trfn.a2, trfn.a3, trfn.a4]
        return fuzzy


# --- ZDT ---
class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


# --- DTLZ ---
class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


# --- WFG (full) ---
def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return r * (0.1 * (1 - y) * (1 - B - y) ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


# --- CEC09 ---
class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


# --- MaF ---
def _maf_dtlz_base(x, n_obj):
    n, m = len(x), n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


# --- Scalability ---
class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND Aα-NSGA-II
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.zeros((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = float(min(a1, a2, a3, a4))
            aU = float(max(a1, a2, a3, a4))
            aM = float(0.5 * (a2 + a3))
            aM = max(aL, min(aM, aU))
            self.fuzzy_obj[i] = [aL, aM, aU]

    def alpha_weighted_obj(self):
        out = np.zeros(self.fuzzy_obj.shape[0])
        for i, (L, M, U) in enumerate(self.fuzzy_obj):
            out[i] = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return out


class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=100, n_gen=500,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.4, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        F1 = [TriangularFuzzyNumber(*ind1.fuzzy_obj[i])
              for i in range(ind1.fuzzy_obj.shape[0])]
        F2 = [TriangularFuzzyNumber(*ind2.fuzzy_obj[i])
              for i in range(ind2.fuzzy_obj.shape[0])]
        return fuzzy_dominates(F1, F2)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in [-0.05, -0.02, 0.02, 0.05]:
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = self.rng.integers(len(x_new))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: BASELINES
# ============================================================================

class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=100, n_gen=500,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineCrispExpected(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 1.0
        super().__init__(*args, **kwargs)


class BaselineFMEA(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 0.60
        super().__init__(*args, **kwargs)


class BaselineGridSearchFixedAlpha:
    def __init__(self, benchmark, alpha_grid=None, **kwargs):
        self.benchmark = benchmark
        self.alpha_grid = alpha_grid if alpha_grid is not None else np.arange(0.5, 1.01, 0.1)
        self.kwargs = kwargs
        self.population = []
        self.best_alpha = 0.75

    def run(self):
        best_pop, best_hv = None, -np.inf
        best_alpha = 0.75
        for alpha in self.alpha_grid:
            algo = BaselineNSGA2(self.benchmark, alpha=float(alpha), **self.kwargs)
            pop = algo.run()
            pf = [ind for ind in pop if ind.rank == 0]
            if len(pf) == 0:
                continue
            F = np.array([ind.alpha_weighted_obj() for ind in pf])
            ref = np.max(F, axis=0) * 1.1 + 1.0
            hv = hypervolume(F, ref)
            if hv > best_hv:
                best_hv = hv
                best_pop = pop
                best_alpha = float(alpha)
        self.population = best_pop if best_pop is not None else []
        self.best_alpha = best_alpha
        return self.population

    def get_alphas(self):
        return np.array([self.best_alpha])

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 7: METRICS (RAW α-WEIGHTED HV)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=5000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


def _hv_mc(F, ref, n_samples=5000):
    lows = np.min(F, axis=0)
    samples = np.random.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def compute_pareto_hv(population, use_alpha=True):
    """Compute HV using α-weighted objectives (raw, unnormalized)."""
    pf = [ind for ind in population if ind.rank == 0]
    if len(pf) == 0:
        return 0.0, np.array([])
    if use_alpha:
        F = np.array([ind.alpha_weighted_obj() for ind in pf])
    else:
        F = np.array([[(L + 4 * M + U) / 6.0 for L, M, U in ind.fuzzy_obj]
                      for ind in pf])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ============================================================================
# PART 8: GLOBAL CONFIGURATION
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 30,
    'pop_size': 100,
    'n_gen': 500,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'results_dir': 'parallel_results',
    'figures_dir': 'parallel_figures',
    'latex_dir': 'parallel_latex',
    'supp_latex_dir': 'supplementary_latex',
    'reference_algo': 'Aα-NSGA-II',
    'bootstrap_samples': 10000,
}

ALGO_NAMES = [
    'Aα-NSGA-II', 'NSGA-II', 'F-MEA', 'Crisp-Expected',
    'Grid-Search Fixed-α', 'Self-Adaptive Mutation',
]

ALGO_COLORS = {
    'Aα-NSGA-II': '#1f77b4',
    'NSGA-II': '#ff7f0e',
    'F-MEA': '#2ca02c',
    'Crisp-Expected': '#d62728',
    'Grid-Search Fixed-α': '#9467bd',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II', AAlphaNSGA2, {**common}),
        ('NSGA-II', BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('F-MEA', BaselineFMEA, {**common, 'alpha': 0.60}),
        ('Crisp-Expected', BaselineCrispExpected, {**common}),
        ('Grid-Search Fixed-α', BaselineGridSearchFixedAlpha, {**common}),
        ('Self-Adaptive Mutation', BaselineSelfAdaptiveMutation,
         {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 9: PARALLEL / SERIAL EXECUTION (THREAD-SAFE JUPYTER VERSION)
# ============================================================================

def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    benchmark = get_extended_benchmark(prob_name)
    algos = get_algo_config(pop_size, n_gen)

    results = {}
    for algo_name, algo_cls, kwargs in algos:
        hvs_raw, alphas = [], []
        errors = 0
        for run_idx in range(n_runs):
            seed = seed_base + run_idx
            try:
                algo = algo_cls(benchmark, seed=seed, **kwargs)
                pop = algo.run()
            except Exception:
                errors += 1
                continue

            hv_raw, F = compute_pareto_hv(pop, use_alpha=True)
            if len(F) == 0:
                continue
            hvs_raw.append(hv_raw)

            if hasattr(algo, 'get_alphas'):
                alphas.append(float(np.mean(algo.get_alphas())))

        results[algo_name] = {
            'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
            'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
            'hv_raw_all': hvs_raw,
            'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
            'alpha_mean_all': alphas,
            'n_errors': errors,
        }
    return (prob_name, results)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base, verbose=True):
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(problems)]

    if verbose:
        print(f"\n[SERIAL MODE] Total problems: {len(tasks)}")

    t0 = time.time()
    all_results = {}
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if verbose:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        task_t0 = time.time()
        prob_name, results = run_single_problem(task)
        all_results[prob_name] = results
        if verbose:
            elapsed = time.time() - t0
            task_elapsed = time.time() - task_t0
            avg = elapsed / i
            eta = avg * (len(tasks) - i)
            print(f"  [{i}/{len(tasks)}] {prob_name} done in "
                  f"{task_elapsed:.1f}s (ETA: {eta/60:.1f} min)", flush=True)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel_safe(problems, n_runs, pop_size, n_gen, seed_base,
                      n_workers=None, verbose=True):
    """
    Parallel execution with automatic environment detection:
      - Jupyter/Colab: ThreadPoolExecutor (no handle duplication errors)
      - Terminal: multiprocessing.Pool (real parallelism)
    """
    if n_workers is None:
        n_workers = min(8, max(1, mp.cpu_count() - 1))
    n_workers = min(n_workers, len(problems), 8)

    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(problems)]

    in_jupyter = ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)

    if verbose:
        print(f"\nTotal problems: {len(tasks)}")
        print(f"Workers: {n_workers}")
        print(f"Backend: {'threads (Jupyter)' if in_jupyter else 'processes (terminal)'}")

    t0 = time.time()
    all_results = {}

    # --------------------------------------------------------------------
    # JUPYTER: use ThreadPoolExecutor (avoids WinError 5)
    # --------------------------------------------------------------------
    if in_jupyter:
        try:
            with ThreadPoolExecutor(max_workers=n_workers) as executor:
                future_to_task = {executor.submit(run_single_problem, t): t
                                  for t in tasks}
                completed = 0
                for future in as_completed(future_to_task):
                    try:
                        prob_name, results = future.result()
                    except Exception as e:
                        if verbose:
                            print(f"  Task failed: {e}")
                        continue
                    completed += 1
                    all_results[prob_name] = results
                    if verbose:
                        elapsed = time.time() - t0
                        rate = completed / max(elapsed, 1e-6)
                        eta = (len(tasks) - completed) / max(rate, 1e-6)
                        print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                              f"(ETA: {eta/60:.1f} min)", flush=True)
        except Exception as e:
            if verbose:
                print(f"\n⚠ Thread pool failed ({e}). Falling back to serial.")
            remaining = [t[0] for t in tasks if t[0] not in all_results]
            serial_results = run_serial(
                remaining, n_runs, pop_size, n_gen, seed_base, verbose)
            all_results.update(serial_results)

        if verbose:
            print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
        return all_results

    # --------------------------------------------------------------------
    # TERMINAL: use multiprocessing.Pool (real parallelism)
    # --------------------------------------------------------------------
    try:
        ctx = mp.get_context('spawn')
        pool = ctx.Pool(processes=n_workers)
        iterator = pool.imap_unordered(run_single_problem, tasks, chunksize=1)
        completed = 0
        while completed < len(tasks):
            try:
                prob_name, results = iterator.next(timeout=1800)
            except mp.TimeoutError:
                if verbose:
                    print(f"\n⚠ Timeout. Falling back to serial.")
                pool.terminate()
                pool.join()
                remaining = [t[0] for t in tasks if t[0] not in all_results]
                if remaining:
                    serial_results = run_serial(
                        remaining, n_runs, pop_size, n_gen, seed_base, verbose)
                    all_results.update(serial_results)
                return all_results
            completed += 1
            all_results[prob_name] = results
            if verbose:
                elapsed = time.time() - t0
                rate = completed / max(elapsed, 1e-6)
                eta = (len(tasks) - completed) / max(rate, 1e-6)
                print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                      f"(ETA: {eta/60:.1f} min)", flush=True)
        pool.close()
        pool.join()
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel failed ({e}). Falling back to serial.")
        try:
            pool.terminate(); pool.join()
        except Exception:
            pass
        remaining = [t[0] for t in tasks if t[0] not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base, verbose)
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICAL TESTS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=10000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    means = np.array([np.mean(rng.choice(data, n, replace=True))
                      for _ in range(n_samples)])
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}

    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Step 1: Shapiro-Wilk Normality Test (on rank data) ---")
    print(f"{'Algorithm':<28} | {'W':>10} | {'p-value':>12} | {'Normal':>8}")
    print("-" * 78)
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<28} | {W:>10.4f} | {p:>12.4e} | "
              f"{'YES' if p > 0.05 else 'NO':>8}")

    print(f"\n--- Step 2: Friedman Test (on ranks) ---")
    try:
        stat, p = stats.friedmanchisquare(
            *[agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0])
        print(f"Chi-square = {stat:.4f}, p-value = {p:.6e}")
        print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Step 3: Kruskal-Wallis Test (on normalized HV) ---")
    try:
        stat, p = stats.kruskal(*[agg_hv[a] for a in algo_names
                                  if len(agg_hv[a]) > 0])
        print(f"H = {stat:.4f}, p-value = {p:.6e}")
        print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Step 4: Pairwise Wilcoxon (with Bonferroni) on ranks ---")
    print(f"\n{'Comparison':<42} | {'p-value':>12} | {'Cohen d':>10} | "
          f"{'Cliff δ':>10} | {'Sig.':>5}")
    print("-" * 92)

    n_comparisons = len(algo_names) - 1
    threshold = 0.05 / n_comparisons

    pairwise_results = {}
    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{reference} vs. {algo:<26} | {p:>12.4e} | "
                  f"{d:>10.4f} | {cl:>10.4f} | {sig:>5}")
        print(f"\nBonferroni-corrected threshold: α' = {threshold:.6f}")

    print(f"\n--- Step 5: Bootstrap 95% Confidence Intervals ---")
    print(f"\n{'Algorithm':<28} | {'Mean':>8} | {'95% CI':>22} | {'Width':>8}")
    print("-" * 78)
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<28} | {mean:>8.4f} | "
              f"[{lo:>7.4f}, {hi:>7.4f}] | {hi-lo:>8.4f}")

    print(f"\n--- Step 6: Mean Ranks (lower = better) ---")
    print(f"\n{'Algorithm':<28} | {'Mean Rank':>10} | {'Std':>8} | {'Median':>8}")
    print("-" * 78)
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<28} | {np.mean(agg_rank[algo]):>10.2f} | "
                  f"{np.std(agg_rank[algo]):>8.2f} | "
                  f"{np.median(agg_rank[algo]):>8.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: MAIN VISUALIZATIONS
# ============================================================================

def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(10, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)

    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)

    ax.set_title('Normalized α-weighted HV per Problem (best = 1.0)',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, label='Normalized α-weighted HV', shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))

    fig, ax = plt.subplots(figsize=(12, 5))
    n_families = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_families)

    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)

        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title('Algorithm Performance by Problem Family', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)

    fig, ax = plt.subplots(figsize=(9, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)

    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)

    if not alpha_by_problem:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Converged α', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")

    sizes = []
    for p in scal_probs:
        try: s = int(p.replace('FScal', ''))
        except ValueError: s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]

    fig, ax = plt.subplots(figsize=(8, 5))
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                if best > 0:
                    hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best)
                else:
                    hvs.append(0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=10)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ============================================================================
# PART 12: MAIN LATEX TABLES (UTF-8 ENCODING)
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    # --- table_overall.tex ---
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # --- table_family.tex ---
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # --- table_statistics.tex ---
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (with Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # --- table_bootstrap.tex ---
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print(f"  Generated 4 LaTeX tables in {save_dir}/")


# ============================================================================
# PART 13: PER-PROBLEM TABLE (UTF-8 ENCODING)
# ============================================================================

def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]

    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))

    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')

    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])

    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")

    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    df = pd.DataFrame(csv_rows)
    df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
              index=False, encoding='utf-8')
    print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 14: SETTINGS, MAIN, AND ENTRY POINT
# ============================================================================

JUPYTER_MODE = 'full'
JUPYTER_USE_PARALLEL = True
JUPYTER_N_WORKERS = 4


def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir'] = 'test_results'
        CONFIG['figures_dir'] = 'test_figures'
        CONFIG['latex_dir'] = 'test_latex'
        CONFIG['supp_latex_dir'] = 'test_supplementary_latex'

    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = min(8, max(1, mp.cpu_count() - 1))


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v4.2 — UTF-8 Encoding Fix)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    in_jupyter = ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    if in_jupyter:
        print(f"Environment: Jupyter/Colab (will use threads)")
    else:
        print(f"Environment: Terminal (will use processes)")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir']]:
        os.makedirs(d, exist_ok=True)

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if os.path.exists(checkpoint_path):
        print(f"\nLoading checkpoint from {checkpoint_path}...")
        with open(checkpoint_path, 'rb') as f:
            all_results = pickle.load(f)
        print(f"  Loaded {len(all_results)} problems")
    else:
        if CONFIG['use_parallel']:
            all_results = run_parallel_safe(
                CONFIG['problems'],
                CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                CONFIG['n_workers'], verbose=True,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'],
                CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                verbose=True,
            )
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(all_results, f)
        print(f"  Checkpoint saved")

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    rows = []
    for prob_name, results in all_results.items():
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
              index=False, encoding='utf-8')
    print(f"\n  CSV saved to {CONFIG['results_dir']}/results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


# ============================================================================
# ENTRY POINT — JUPYTER/CLI SAFE
# ============================================================================

if __name__ == "__main__":
    try:
        mp.freeze_support()
    except Exception:
        pass

    if 'ipykernel' not in sys.modules and 'google.colab' not in sys.modules:
        import argparse
        parser = argparse.ArgumentParser(description='Aα-NSGA-II Pipeline v4.2')
        parser.add_argument('--test', action='store_true')
        parser.add_argument('--serial', action='store_true')
        parser.add_argument('--workers', type=int, default=None)
        parser.add_argument('--runs', type=int, default=None)
        parser.add_argument('--pop', type=int, default=None)
        parser.add_argument('--gen', type=int, default=None)
        args, _ = parser.parse_known_args()

        mode = 'test' if args.test else 'full'
        use_parallel = not args.serial
        main(mode=mode, use_parallel=use_parallel,
             n_workers=args.workers, runs=args.runs,
             pop=args.pop, gen=args.gen)
    else:
        main(mode=JUPYTER_MODE,
             use_parallel=JUPYTER_USE_PARALLEL,
             n_workers=JUPYTER_N_WORKERS)

In [2]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v6.1 — Fair Comparison + Pareto Plots)
# ============================================================================
# Fixes from v6.0:
#   - compute_pareto_hv uses FIXED α = 0.75 for every algorithm (fair)
#   - fast_nondominated_sort_fuzzy uses proper strict dominance
#   - New: per-problem 2D and 3D Pareto-front visualizations
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12

def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False

IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v6_1")

# Fixed α used for the FAIR HV metric. Every algorithm is scored on this.
FAIR_ALPHA = 0.75


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    """Standalone monitor — call from any Jupyter cell."""
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30):
    spread = abs(x) * delta if x != 0 else delta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS  (with FIX for strict dominance)
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []

    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L

    # --- FIXED: credibility that A is weakly dominated by B on each objective
    num = U[None, :, :] - L[:, None, :]      # B.U - A.L
    den = W[:, None, :] + W[None, :, :]      # A.width + B.width
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)

    weak_all = np.all(cred >= 0.5, axis=2)

    # --- FIXED: proper strict dominance: A.L > B.U on at least one objective
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)

    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)

    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]

    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: FUZZY BENCHMARKS (identical to v6.0 — unchanged)
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND Aα-NSGA-II
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL: aM = aL
            if aM > aU: aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: BASELINES
# ============================================================================

class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineCrispExpected(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 1.0
        super().__init__(*args, **kwargs)


class BaselineFMEA(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        kwargs['alpha'] = 0.60
        super().__init__(*args, **kwargs)


class BaselineGridSearchFixedAlpha:
    def __init__(self, benchmark, alpha_grid=None, **kwargs):
        self.benchmark = benchmark
        self.alpha_grid = alpha_grid if alpha_grid is not None else np.arange(0.5, 1.01, 0.1)
        self.kwargs = kwargs
        self.population = []
        self.best_alpha = 0.75

    def run(self):
        best_pop, best_hv = None, -np.inf
        best_alpha = 0.75
        for alpha in self.alpha_grid:
            algo = BaselineNSGA2(self.benchmark, alpha=float(alpha), **self.kwargs)
            pop = algo.run()
            pf = [ind for ind in pop if ind.rank == 0]
            if len(pf) == 0:
                continue
            # NOTE: uses a fixed reference α for search fairness (see v6.1)
            F = np.array([FAIR_ALPHA * ind.fuzzy_obj[:, 1]
                          + (1 - FAIR_ALPHA) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
                          for ind in pf])
            ref = np.max(F, axis=0) * 1.1 + 1.0
            hv = hypervolume(F, ref)
            if hv > best_hv:
                best_hv = hv
                best_pop = pop
                best_alpha = float(alpha)
        self.population = best_pop if best_pop is not None else []
        self.best_alpha = best_alpha
        return self.population

    def get_alphas(self):
        return np.array([self.best_alpha])

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 7: METRICS  (FIXED: fair fixed-α HV)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    """Return the α-weighted objectives at a FIXED α for all Pareto members."""
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    """
    FAIR hypervolume: every algorithm is scored on the SAME fixed α,
    regardless of what α values its population evolved.
    """
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ============================================================================
# PART 8: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'reference_algo':    'Aα-NSGA-II',
    'bootstrap_samples': 5000,
}

ALGO_NAMES = [
    'Aα-NSGA-II', 'NSGA-II', 'F-MEA', 'Crisp-Expected',
    'Grid-Search Fixed-α', 'Self-Adaptive Mutation',
]

ALGO_COLORS = {
    'Aα-NSGA-II': '#1f77b4',
    'NSGA-II': '#ff7f0e',
    'F-MEA': '#2ca02c',
    'Crisp-Expected': '#d62728',
    'Grid-Search Fixed-α': '#9467bd',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II', AAlphaNSGA2, {**common}),
        ('NSGA-II', BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('F-MEA', BaselineFMEA, {**common, 'alpha': 0.60}),
        ('Crisp-Expected', BaselineCrispExpected, {**common}),
        ('Grid-Search Fixed-α', BaselineGridSearchFixedAlpha, {**common}),
        ('Self-Adaptive Mutation', BaselineSelfAdaptiveMutation,
         {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 9: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)   # fixed-α, fair
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 10: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta/60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 11: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}

    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Step 1: Shapiro-Wilk Normality Test ---")
    print(f"{'Algorithm':<28} | {'W':>10} | {'p-value':>12} | {'Normal':>8}")
    print("-" * 78)
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<28} | {W:>10.4f} | {p:>12.4e} | "
              f"{'YES' if p > 0.05 else 'NO':>8}")

    print(f"\n--- Step 2: Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p-value = {p:.6e}")
            print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Step 3: Kruskal-Wallis Test ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p-value = {p:.6e}")
            print(f"Result: {'SIGNIFICANT' if p < 0.05 else 'Not significant'}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Step 4: Pairwise Wilcoxon (Bonferroni) ---")
    print(f"\n{'Comparison':<42} | {'p-value':>12} | {'Cohen d':>10} | "
          f"{'Cliff δ':>10} | {'Sig.':>5}")
    print("-" * 92)

    n_comparisons = max(1, len(algo_names) - 1)
    threshold = 0.05 / n_comparisons
    pairwise_results = {}

    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{reference} vs. {algo:<26} | {p:>12.4e} | "
                  f"{d:>10.4f} | {cl:>10.4f} | {sig:>5}")
        print(f"\nBonferroni-corrected threshold: α' = {threshold:.6f}")

    print(f"\n--- Step 5: Bootstrap 95% CI ---")
    print(f"\n{'Algorithm':<28} | {'Mean':>8} | {'95% CI':>22} | {'Width':>8}")
    print("-" * 78)
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<28} | {mean:>8.4f} | "
              f"[{lo:>7.4f}, {hi:>7.4f}] | {hi-lo:>8.4f}")

    print(f"\n--- Step 6: Mean Ranks ---")
    print(f"\n{'Algorithm':<28} | {'Mean Rank':>10} | {'Std':>8} | {'Median':>8}")
    print("-" * 78)
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<28} | {np.mean(agg_rank[algo]):>10.2f} | "
                  f"{np.std(agg_rank[algo]):>8.2f} | "
                  f"{np.median(agg_rank[algo]):>8.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 12: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(10, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)

    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)

    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, label='Normalized α-weighted HV', shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)

    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Distribution of Converged α (Aα-NSGA-II)', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]

    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan); continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=10)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ---- PARETO FRONT VISUALIZATIONS ------------------------------------------

def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    """
    Re-run each algorithm on a single problem (one seed) and plot the
    resulting Pareto fronts in 2D or 3D at a FIXED α.
    """
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    """
    Generate Pareto-front plots for a representative selection of problems.
    Uses smaller pop/gen than the main run so this finishes reasonably fast.
    """
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name} ({REGISTRY[prob_name]().n_obj} obj)...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


# ============================================================================
# PART 13: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))

    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')

    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")

    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 14: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v6.1 — Fair Comparison + Pareto Plots)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    # ----- Pareto fronts -----
    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(description='Aα-NSGA-II Pipeline v6.1')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial
    if args.no_pareto:
        # Monkey-patch to skip pareto
        global generate_all_pareto_fronts
        generate_all_pareto_fronts = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v6.1 — Fair Comparison + Pareto Plots)
Problems: 49 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_1
Fair HV metric uses α = 0.75
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_1\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_1\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_1\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_1\supplementary_latex

[JUPYTER MODE] Using serial execution for stability.

[SERIAL] 49 prob

In [1]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v6.2 — Adaptive α + Adaptive Mutation)
# ============================================================================
# Additions vs v6.1:
#   - AAlphaNSGA2AdaptiveMutation: combines α-adaptation with mutation-rate
#     adaptation (success-rate schedule). This is the actual proposed algorithm.
#   - BaselineNSGA2HighMutation: fixed p_mutation = 0.05, no adaptivity.
#     Diagnostic baseline to isolate WHY Self-Adaptive Mutation wins.
#   - ALGO_NAMES expanded to 7; colours updated.
#   - Fair fixed-α HV metric (α = 0.75) retained from v6.1.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12

def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False

IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v6_2")

FAIR_ALPHA = 0.75


# ============================================================================
# PART 0: PROGRESS MONITOR (identical to v6.1)
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])

def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30):
    spread = abs(x) * delta if x != 0 else delta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS (identical to v6.1 — proper strict dominance)
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: BENCHMARKS (identical to v6.1)
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND ALGORITHMS
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL: aM = aL
            if aM > aU: aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ---- Aα-NSGA-II (baseline version, no adaptive mutation) ----
class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        """p_mutation overrides the stored rate (for the combined algorithm)."""
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# *** NEW IN v6.2 ***
# ============================================================================
class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """
    Aα-NSGA-II with a success-rate-adaptive mutation schedule.

    Every generation, after evaluating the offspring:
        success_rate = fraction of children whose rank is <= BOTH parents'
        mutation_rate *= 1.1 if success_rate > 0.2 else 0.9
        mutation_rate is clipped to [0.001, 0.5]

    This combines the two independent adaptation mechanisms:
      - Adaptive α (already in the parent class)
      - Adaptive mutation rate (adopted from Self-Adaptive Mutation)

    Motivation: the v6.1 results show Self-Adaptive Mutation dominates the
    fixed-α baselines because it explores the search space more widely. This
    class adds that exploration advantage without losing α adaptivity.
    """
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05     # same starting point as Self-Adaptive

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)

                # use the CURRENT adaptive mutation rate
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)

                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)

                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)

                # success tracking (same logic as Self-Adaptive Mutation)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2

                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])

            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]

            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()

            # adapt mutation rate
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))

        return self.population


# ---- Baselines ----
class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    """Fixed p_mutation = 0.05 (no adaptivity) — isolates the effect of the
    starting mutation rate from the adaptivity."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 6: METRICS (fixed-α HV, identical to v6.1)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ============================================================================
# PART 7: CONFIG (7 algorithms now)
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
}

ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',   # NEW combined algorithm
    'Aα-NSGA-II',                        # original (no adaptive mutation)
    'NSGA-II',                           # standard baseline
    'NSGA-II (high mutation)',           # ablation: fixed p_mut = 0.05
    'Self-Adaptive Mutation',            # existing strong baseline
]

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': 0.75}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION (identical plumbing to v6.1)
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION (identical to v6.1)
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta/60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS (identical to v6.1)
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Pairwise Wilcoxon vs. reference ---")
    n_comparisons = max(1, len(algo_names) - 1)
    threshold = 0.05 / n_comparisons
    pairwise_results = {}
    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold: {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS (identical to v6.1)
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(12, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan); continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v6.2 — Adaptive α + Adaptive Mutation)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print(f"Algorithms ({len(ALGO_NAMES)}): {', '.join(ALGO_NAMES)}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(description='Aα-NSGA-II Pipeline v6.2')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial
    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen, quiet=args.quiet)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v6.2 — Adaptive α + Adaptive Mutation)
Problems: 49 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_2
Fair HV metric uses α = 0.75
Algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_2\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_2\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_2\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzz

In [1]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v6.3 — Adaptive α + Adaptive Mutation
#                                + Fair HV + Pareto Front Plots)
# ============================================================================
# This file is the MERGE of:
#   v6.1 — Fair fixed-α HV metric + 2D/3D Pareto-front visualizations
#   v6.2 — AAlphaNSGA2AdaptiveMutation + BaselineNSGA2HighMutation
#          (adaptive α + adaptive mutation; ablation study)
#
# What is new vs. v6.2:
#   - plot_single_pareto_front(...)      (restored from v6.1)
#   - generate_all_pareto_fronts(...)    (restored from v6.1)
#   - main() now calls the Pareto generator after the LaTeX tables
#   - --no-pareto / --full-pareto CLI flags restored
#   - pareto_quick parameter restored in main()
#
# Everything else (algorithms, statistics, tables, aggregate figures)
# is identical to v6.2.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12

def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False

IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v6_3")

FAIR_ALPHA = 0.75


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])

def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30):
    spread = abs(x) * delta if x != 0 else delta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND ALGORITHMS
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL: aM = aL
            if aM > aU: aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        """p_mutation overrides the stored rate (for the combined algorithm)."""
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """
    Aα-NSGA-II with a success-rate-adaptive mutation schedule.

    Every generation, after evaluating the offspring:
        success_rate = fraction of children whose rank is <= BOTH parents'
        mutation_rate *= 1.1 if success_rate > 0.2 else 0.9
        mutation_rate is clipped to [0.001, 0.5]
    """
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    """Fixed p_mutation = 0.05 (no adaptivity) — isolates the effect of the
    starting mutation rate from the adaptivity."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 6: METRICS (fixed-α HV)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
}

ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': 0.75}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta/60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Pairwise Wilcoxon vs. reference ---")
    n_comparisons = max(1, len(algo_names) - 1)
    threshold = 0.05 / n_comparisons
    pairwise_results = {}
    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold: {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS (aggregate)
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(12, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan); continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ---- PARETO FRONT VISUALIZATIONS (restored from v6.1) ----------------------

def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    """
    Re-run each algorithm on a single problem (one seed) and plot the
    resulting Pareto fronts in 2D or 3D at a FIXED α.
    """
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    """
    Generate Pareto-front plots for a representative selection of problems.
    Uses smaller pop/gen than the main run so this finishes reasonably fast.
    """
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name} ({REGISTRY[prob_name]().n_obj} obj)...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE "
          "(v6.3 — Adaptive α + Adaptive Mutation + Pareto Plots)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print(f"Algorithms ({len(ALGO_NAMES)}): {', '.join(ALGO_NAMES)}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    # ----- Pareto fronts (restored from v6.1) -----
    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v6.3 (adaptive α + adaptive mutation '
                    '+ Pareto plots)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        global generate_all_pareto_fronts
        generate_all_pareto_fronts = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v6.3 — Adaptive α + Adaptive Mutation + Pareto Plots)
Problems: 49 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_3
Fair HV metric uses α = 0.75
Algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_3\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_3\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_3\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as 

In [2]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v6.4 — Adaptive α + Adaptive Mutation
#                                + Fair HV + Pareto Plots + F2 family
#                                + α-sensitivity proxy Σ̂)
# ============================================================================
# This file is the MERGE of:
#   v6.3 — v6.2 + Pareto-front plots (restored from v6.1)
#   v6.4 — adds the F2 α-sensitive diagnostic family and the empirical
#          α-sensitivity proxy Σ̂, closing the loop with the manuscript's
#          Corollary 4.8 (analytic form of Σ).
#
# What is new vs. v6.3:
#   - F2Base class                                (NEW)
#   - F2_PROBLEMS list and registration           (NEW)
#   - 'F2' entry in FAMILIES                      (NEW)
#   - get_extended_benchmark handles F2 prefix    (CHANGED)
#   - compute_sigma_hat()                         (NEW)
#   - generate_sigma_hat_table()                  (NEW)
#   - Pareto-front selection includes F2 problems (CHANGED)
#   - main() computes and exports sigma_hat.csv   (NEW)
#   - --no-sigma CLI flag                         (NEW)
#
# Everything else (algorithms, statistics, aggregate figures) is
# identical to v6.3.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed, TimeoutError as FutTimeout
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12

def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False

IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v6_4")               # <<< CHANGED

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)                           # <<< NEW


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])

def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')
    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU
    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"
    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')
    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4
    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"
    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0
    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):                # <<< CHANGED
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2
    def evaluate_crisp(self, x):
        raise NotImplementedError
    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a): return y ** a
def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)
def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))
def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)
def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))
def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))
def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)
def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2: A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)
def _shape_linear(x, m): return x / m
def _shape_convex(x, m): return (1 - np.cos(x * np.pi / 2)) / m
def _shape_concave(x, m): return np.sin(x * np.pi / 2) / m
def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m
def _shape_disc(x, m): return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var
    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t
    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t
    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t
    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2); J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2; y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3); J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m-1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x): return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m-1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m-1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m-1] = (1 + g) * (m - np.sum(
            f[:m-1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m-1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x): return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x): return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x): return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x): return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x): return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x): return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self): self.n_var = 5000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self): self.n_var = 10000; self.n_obj = 3
    def evaluate_crisp(self, x): return FScal(10000, 3).evaluate_crisp(x)


# ---------------------------------------------------------------------------
# F2: α-sensitive diagnostic problems                              # <<< NEW
# ---------------------------------------------------------------------------
class F2Base(FuzzyBenchmark):
    """
    Wraps an F1 benchmark and modulates the trapezoidal spread by a
    position-dependent factor η(x) ∈ [0.15, 0.45], so that Σ > 0 by
    Corollary 4.8 of the manuscript.

    The position measure σ(x) ∈ [0,1] is the normalized distance of the
    first objective from the front's extreme. For ZDT and CEC09 the
    decision variable x[0] already plays this role, so σ(x) = x[0].
    """
    def __init__(self, base_cls):
        self.base = base_cls()
        self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

# ---- F2 family registration (α-sensitive diagnostics) ------------ <<< NEW
F2_PROBLEMS = [
    F2Base(FZDT1), F2Base(FZDT2), F2Base(FZDT3), F2Base(FZDT4), F2Base(FZDT6),
    F2Base(FDTLZ2), F2Base(FDTLZ4), F2Base(FDTLZ7),
    F2Base(FUF1), F2Base(FUF2), F2Base(FUF3), F2Base(FUF4),
    F2Base(FMaF1), F2Base(FMaF2), F2Base(FMaF3),
]
ALL_PROBLEMS = ALL_PROBLEMS + F2_PROBLEMS
# ---------------------------------------------------------------------------

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': [p.name for p in F2_PROBLEMS],                         # <<< NEW
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    if name.startswith("F2"):                                    # <<< NEW
        base_name = name[2:]
        if base_name in REGISTRY:
            return F2Base(REGISTRY[base_name])
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL AND ALGORITHMS
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL: aM = aL
            if aM > aU: aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """
    Aα-NSGA-II with a success-rate-adaptive mutation schedule.

    Every generation, after evaluating the offspring:
        success_rate = fraction of children whose rank is <= BOTH parents'
        mutation_rate *= 1.1 if success_rate > 0.2 else 0.9
        mutation_rate is clipped to [0.001, 0.5]
    """
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    """Fixed p_mutation = 0.05 (no adaptivity) — isolates the effect of the
    starting mutation rate from the adaptivity."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 6: METRICS (fixed-α HV)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ---- α-sensitivity proxy Σ̂ --------------------------------------- <<< NEW
def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300):
    """
    Compute the empirical α-sensitivity proxy Σ̂ for a given problem.

    Definition (Corollary 4.8 of the manuscript):
        Σ̂ = max_α HV(α)  -  min_α HV(α)
    where HV(α) is the hypervolume of the Pareto front produced by the
    reference algorithm (Aα-NSGA-II with adaptive mutation) at a single
    fixed confidence level α.

    For α-invariant problems (F1), Σ̂ should be near zero.
    For α-sensitive problems (F2), Σ̂ should be positive.
    """
    benchmark = get_extended_benchmark(prob_name)
    hvs = []
    for a in alphas:
        algo = AAlphaNSGA2AdaptiveMutation(
            benchmark, pop_size=pop_size, n_gen=n_gen, seed=seed,
        )
        pop = algo.run()
        hv, _ = compute_pareto_hv(pop, fixed_alpha=a)
        hvs.append(hv)
    return float(max(hvs) - min(hvs)), hvs
# ---------------------------------------------------------------------------


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),  # <<< NEW
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,                                    # <<< NEW
    'sigma_hat_problems': [p.name for p in F2_PROBLEMS]           # <<< NEW
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',          # F1 refs
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': 0.75}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': 0.75}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': 0.75}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta/60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time()-t0)/60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)'):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    print(f"\n--- Pairwise Wilcoxon vs. reference ---")
    n_comparisons = max(1, len(algo_names) - 1)
    threshold = 0.05 / n_comparisons
    pairwise_results = {}
    if reference in agg_rank and len(agg_rank[reference]) >= 2:
        ref_data = np.array(agg_rank[reference])
        for algo in algo_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'p': p, 'cohens_d': d, 'cliffs_delta': cl, 'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold: {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS (aggregate)
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(12, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(10, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan); continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                 ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


# ---- PARETO FRONT VISUALIZATIONS -------------------------------------------

def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        # F1 problems
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        # F2 problems (new)                                        # <<< NEW
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name} ({REGISTRY[prob_name]().n_obj} obj)...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


# ---- new: α-sensitivity proxy Σ̂ table --------------------------- <<< NEW
def generate_sigma_hat_table(all_results, problems, save_dir):
    """
    Compute Σ̂ for every problem in CONFIG['sigma_hat_problems'] and dump
    a CSV + LaTeX table.

    For F1 problems, Σ̂ should be near zero.
    For F2 problems, Σ̂ should be positive.
    """
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. F1 problems are $\alpha$-invariant by construction '
        r'($\Sigma = 0$ analytically); F2 problems are $\alpha$-sensitive '
        r'($\Sigma > 0$).}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\mathrm{HV}(0.5)$ & $\mathrm{HV}(1.0)$ \\',
        r'\midrule',
    ]
    for row in rows:
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")
# ---------------------------------------------------------------------------


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(f'{ref} vs. {algo} & {res["p"]:.2e} & '
                     f'{res["cohens_d"]:.3f} & {res["cliffs_delta"]:.3f} & '
                     f'{res["significant"]} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names)+1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']              # <<< CHANGED
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True):                # <<< CHANGED
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE "
          "(v6.4 — Adaptive α + Adaptive Mutation + Pareto Plots "
          "+ F2 + Σ̂)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print(f"Algorithms ({len(ALGO_NAMES)}): {', '.join(ALGO_NAMES)}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir']]:                         # <<< NEW
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo']
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    # ----- Σ̂ computation (new) ----------------------------------- <<< NEW
    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    # ----- Pareto fronts ------------------------------------------
    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v6.4 (adaptive α + adaptive mutation '
                    '+ Pareto plots + F2 family + Σ̂)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--no-sigma', action='store_true',       # <<< NEW
                        help='Skip α-sensitivity Σ̂ computation (fast)')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        global generate_all_pareto_fronts
        generate_all_pareto_fronts = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma)                        # <<< NEW


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v6.4 — Adaptive α + Adaptive Mutation + Pareto Plots + F2 + Σ̂)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_4
Fair HV metric uses α = 0.75
Algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_4\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_4\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v6_4\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting P

In [1]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v7.1)
#   Adaptive α + Adaptive Mutation + Fair HV + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#
# v7.1 changes vs v7.0:
#   - SPEA2 fitness now uses a vectorised dominance matrix
#     (_spea2_dominance_matrix), removing the deprecated
#     np.sum(generator) call and speeding up the O(N^2) block.
#   - Optional forced-parallel execution from Jupyter via
#     FORCE_PARALLEL_FROM_JUPYTER.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

# Set to True to bypass the Jupyter safety check and force parallel execution.
FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v7_1")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)

# ---- External benchmark defaults -------------------------------------------
SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10
# ----------------------------------------------------------------------------


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    """Standard simplex-lattice points of Das & Dennis."""
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2

    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3

    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3

    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3

    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3

    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3

    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3

    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3

    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3

    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"

    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"

    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"

    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


class F2Base(FuzzyBenchmark):
    """α-sensitive diagnostic wrapper: modulates spread by η(x) ∈ [0.15, 0.45]."""

    def __init__(self, base_cls):
        self.base = base_cls()
        self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
]

F2_PROBLEMS = [
    F2Base(FZDT1), F2Base(FZDT2), F2Base(FZDT3), F2Base(FZDT4), F2Base(FZDT6),
    F2Base(FDTLZ2), F2Base(FDTLZ4), F2Base(FDTLZ7),
    F2Base(FUF1), F2Base(FUF2), F2Base(FUF3), F2Base(FUF4),
    F2Base(FMaF1), F2Base(FMaF2), F2Base(FMaF3),
]
ALL_PROBLEMS = ALL_PROBLEMS + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': [p.name for p in F2_PROBLEMS],
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    if name.startswith("F2"):
        base_name = name[2:]
        if base_name in REGISTRY:
            return F2Base(REGISTRY[base_name])
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A: INTERNAL ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """Aα-NSGA-II with the discrete 1/5-success mutation schedule."""

    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 5B: EXTERNAL BENCHMARK ALGORITHMS
# ============================================================================

class SPEA2:
    """SPEA2 with external archive and k-NN density estimation, fuzzy evaluation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        rank_of = np.zeros(n, dtype=int)
        for r, front in enumerate(fronts):
            for idx in front:
                rank_of[idx] = r
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    """MOEA/D with Tchebycheff scalarisation and Das-Dennis weight vectors."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    """NSGA-III with Das-Dennis reference points and niche preservation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                selected.extend(front)
            else:
                remaining = n_select - len(selected)
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                norms = np.linalg.norm(self.ref_points, axis=1)
                norms = np.where(norms < EPS, 1.0, norms)
                ref_unit = self.ref_points / norms[:, None]
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                for ind in selected:
                    w = ind.alpha_weighted_obj()
                    proj_sel = ref_unit @ w
                    niche_count[np.argmin(np.abs(proj_sel - 1.0))] += 1
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [front[k] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    """Co-evolutionary multi-level selection GA with n_subpops demes."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: METRICS
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300):
    benchmark = get_extended_benchmark(prob_name)
    hvs = []
    for a in alphas:
        algo = AAlphaNSGA2AdaptiveMutation(
            benchmark, pop_size=pop_size, n_gen=n_gen, seed=seed,
        )
        pop = algo.run()
        hv, _ = compute_pareto_hv(pop, fixed_alpha=a)
        hvs.append(hv)
    return float(max(hvs) - min(hvs)), hvs


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': [p.name for p in F2_PROBLEMS]
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2': '#2ca02c',
    'MOEA/D': '#d62728',
    'NSGA-III': '#9467bd',
    'cMLSGA': '#7f7f7f',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('SPEA2',
         SPEA2, {**common, 'alpha': FAIR_ALPHA,
                 'archive_size': SPEA2_ARCHIVE_SIZE,
                 'k_nn': SPEA2_KNN_K}),
        ('MOEA/D',
         MOEAD, {**common, 'alpha': FAIR_ALPHA,
                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                 'divisions': MOEAD_DIVISIONS}),
        ('NSGA-III',
         NSGA3, {**common, 'alpha': FAIR_ALPHA,
                 'divisions': NSGA3_DIVISIONS}),
        ('cMLSGA',
         cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                  'n_subpops': CMLSGA_N_SUBPOPS,
                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            ('EXTERNAL COMPARISON', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name}...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


def generate_sigma_hat_table(all_results, problems, save_dir):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. F1 problems are $\alpha$-invariant by construction '
        r'($\Sigma = 0$ analytically); F2 problems are $\alpha$-sensitive '
        r'($\Sigma > 0$).}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\mathrm{HV}(0.5)$ & $\mathrm{HV}(1.0)$ \\',
        r'\midrule',
    ]
    for row in rows:
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
        r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(
            f"{res.get('family', '--')} & {ref} vs. {algo} & "
            f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
            f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE "
          "(v7.1 — Adaptive α + Adaptive Mutation + Pareto Plots "
          "+ F2 + Σ̂ + External Benchmarks)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Force parallel from Jupyter: {FORCE_PARALLEL_FROM_JUPYTER}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    internal_label = ', '.join(INTERNAL_ALGO_NAMES)
    external_label = ', '.join(EXTERNAL_ALGO_NAMES) if include_external \
        else '(skipped)'
    print(f"Internal algorithms ({len(INTERNAL_ALGO_NAMES)}): {internal_label}")
    print(f"External algorithms ({len(EXTERNAL_ALGO_NAMES)}): {external_label}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in EXTERNAL_ALGO_NAMES:
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')
        print(f"\n  External results: "
              f"{os.path.join(CONFIG['external_dir'], 'external_results.csv')}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v7.1 '
                    '(adaptive α + adaptive mutation + Pareto plots '
                    '+ F2 family + Σ̂ + external benchmarks)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--no-sigma', action='store_true',
                        help='Skip α-sensitivity Σ̂ computation (fast)')
    parser.add_argument('--no-external', action='store_true',
                        help='Skip the four external benchmarks (fast)')
    parser.add_argument('--force-parallel', action='store_true',
                        help='Bypass Jupyter safety check for parallel mode')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v7.1 — Adaptive α + Adaptive Mutation + Pareto Plots + F2 + Σ̂ + External Benchmarks)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Force parallel from Jupyter: False
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1
Fair HV metric uses α = 0.75
Internal algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
External algorithms (4): SPEA2, MOEA/D, NSGA-III, cMLSGA
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M 

C:\Users\yvesn\anaconda3\envs\anaconda-2025.12-py3.13\Lib\site-packages\scipy\stats\_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: Input data has range zero. The results may not be accurate.
  res = hypotest_fun_out(*samples, **kwds)


    Saved: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures\heatmap_per_problem.png
  Generating per-family bar chart...
    Saved: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures\family_bars.png
  Generating rank distribution...
    Saved: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures\rank_distribution.png
  Generating alpha analysis...
    Saved: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures\alpha_analysis.png
  Generating scalability plot...
    Saved: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_1\figures\scalability.png
  Generating bootstrap CI plot...
    Saved: D:\Container 

In [2]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v7.2)
#   Adaptive α + Adaptive Mutation + Fair HV + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#
# v7.2 changes vs v7.1:
#   - Fixed NSGA3._niche_selection: indices from
#     fast_nondominated_sort_fuzzy are now indexed back into `combined`
#     instead of being treated as Individual objects.  This restores
#     both the Pareto-front plots and NSGA-III's performance.
#   - Fixed F2_PROBLEMS: the F2 family is now built from dynamically
#     created subclasses of F2Base so that REGISTRY[name]() works.
#     The F2 problems and their Σ̂ values are now computed.
#   - Standard NSGA-III niche count uses nearest reference direction.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

# Set to True to bypass the Jupyter safety check and force parallel execution.
FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v7_2")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)

# ---- External benchmark defaults -------------------------------------------
SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10
# ----------------------------------------------------------------------------


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    """Standard simplex-lattice points of Das & Dennis."""
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2

    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3

    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3

    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3

    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3

    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3

    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3

    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3

    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3

    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"

    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"

    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"

    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


# ---- F2 family: α-sensitive diagnostic wrapper ---------------------------- #
class F2Base(FuzzyBenchmark):
    """α-sensitive diagnostic wrapper: modulates spread by η(x) ∈ [0.15, 0.45]."""

    def __init__(self, base_cls=None, _base=None):
        # Accept either a base class (positional) or an already-built base
        # instance (keyword `_base`), so that dynamically created subclasses
        # can call super().__init__() with no arguments and get their
        # specific base class wired up.
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    """
    Create a named subclass of F2Base bound to a specific base class.

    The subclass has no required arguments so that REGISTRY[name]() can
    instantiate it uniformly with the other benchmark classes.
    """
    def _init(self, _b=base_cls):
        F2Base.__init__(self, _base=_b())

    return type(
        f"F2{base_cls.__name__}",
        (F2Base,),
        {"__init__": _init},
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A: INTERNAL ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """Aα-NSGA-II with the discrete 1/5-success mutation schedule."""

    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 5B: EXTERNAL BENCHMARK ALGORITHMS
# ============================================================================

class SPEA2:
    """SPEA2 with external archive and k-NN density estimation, fuzzy evaluation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        rank_of = np.zeros(n, dtype=int)
        for r, front in enumerate(fronts):
            for idx in front:
                rank_of[idx] = r
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    """MOEA/D with Tchebycheff scalarisation and Das-Dennis weight vectors."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    """NSGA-III with Das-Dennis reference points and niche preservation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        """NSGA-III niche preservation on the alpha-weighted objectives."""
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        # Rank assignment
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        # Pre-normalize reference directions once
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                # All solutions on this front are accepted
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                # Niche preservation on the last front
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                # Niche counts from already-selected
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                # Sort last-front candidates by niche count
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    """Co-evolutionary multi-level selection GA with n_subpops demes."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: METRICS
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300):
    benchmark = get_extended_benchmark(prob_name)
    hvs = []
    for a in alphas:
        algo = AAlphaNSGA2AdaptiveMutation(
            benchmark, pop_size=pop_size, n_gen=n_gen, seed=seed,
        )
        pop = algo.run()
        hv, _ = compute_pareto_hv(pop, fixed_alpha=a)
        hvs.append(hv)
    return float(max(hvs) - min(hvs)), hvs


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2': '#2ca02c',
    'MOEA/D': '#d62728',
    'NSGA-III': '#9467bd',
    'cMLSGA': '#7f7f7f',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('SPEA2',
         SPEA2, {**common, 'alpha': FAIR_ALPHA,
                 'archive_size': SPEA2_ARCHIVE_SIZE,
                 'k_nn': SPEA2_KNN_K}),
        ('MOEA/D',
         MOEAD, {**common, 'alpha': FAIR_ALPHA,
                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                 'divisions': MOEAD_DIVISIONS}),
        ('NSGA-III',
         NSGA3, {**common, 'alpha': FAIR_ALPHA,
                 'divisions': NSGA3_DIVISIONS}),
        ('cMLSGA',
         cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                  'n_subpops': CMLSGA_N_SUBPOPS,
                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            ('EXTERNAL COMPARISON', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name}...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


def generate_sigma_hat_table(all_results, problems, save_dir):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. F1 problems are $\alpha$-invariant by construction '
        r'($\Sigma = 0$ analytically); F2 problems are $\alpha$-sensitive '
        r'($\Sigma > 0$).}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\mathrm{HV}(0.5)$ & $\mathrm{HV}(1.0)$ \\',
        r'\midrule',
    ]
    for row in rows:
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
        r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(
            f"{res.get('family', '--')} & {ref} vs. {algo} & "
            f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
            f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE "
          "(v7.2 — Adaptive α + Adaptive Mutation + Pareto Plots "
          "+ F2 + Σ̂ + External Benchmarks)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Force parallel from Jupyter: {FORCE_PARALLEL_FROM_JUPYTER}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    internal_label = ', '.join(INTERNAL_ALGO_NAMES)
    external_label = ', '.join(EXTERNAL_ALGO_NAMES) if include_external \
        else '(skipped)'
    print(f"Internal algorithms ({len(INTERNAL_ALGO_NAMES)}): {internal_label}")
    print(f"External algorithms ({len(EXTERNAL_ALGO_NAMES)}): {external_label}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in EXTERNAL_ALGO_NAMES:
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')
        print(f"\n  External results: "
              f"{os.path.join(CONFIG['external_dir'], 'external_results.csv')}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v7.2 '
                    '(adaptive α + adaptive mutation + Pareto plots '
                    '+ F2 family + Σ̂ + external benchmarks)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--no-sigma', action='store_true',
                        help='Skip α-sensitivity Σ̂ computation (fast)')
    parser.add_argument('--no-external', action='store_true',
                        help='Skip the four external benchmarks (fast)')
    parser.add_argument('--force-parallel', action='store_true',
                        help='Bypass Jupyter safety check for parallel mode')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v7.2 — Adaptive α + Adaptive Mutation + Pareto Plots + F2 + Σ̂ + External Benchmarks)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Force parallel from Jupyter: False
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_2
Fair HV metric uses α = 0.75
Internal algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
External algorithms (4): SPEA2, MOEA/D, NSGA-III, cMLSGA
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_2\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_2\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M 

In [8]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v7.3)
#   Adaptive α + Adaptive Mutation + Fair HV + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#
# v7.3 changes vs v7.2:
#   - FIXED: F2 family subclasses now expose `name` (and `n_var`, `n_obj`)
#     as CLASS attributes, so REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}
#     correctly registers each F2 variant. Previously every F2 subclass
#     inherited name="base" from FuzzyBenchmark, so only the last F2
#     subclass survived in REGISTRY and every F2 lookup raised
#     "ValueError: Unknown benchmark: F2FZDT1".
#   - Added a post-registration sanity check that asserts every name in
#     F2_NAMES is present in REGISTRY, so this class of bug surfaces
#     immediately at import time rather than 800 minutes into a run.
#
# v7.2 changes vs v7.1:
#   - Fixed NSGA3._niche_selection: indices from
#     fast_nondominated_sort_fuzzy are now indexed back into `combined`
#     instead of being treated as Individual objects.  This restores
#     both the Pareto-front plots and NSGA-III's performance.
#   - Fixed F2_PROBLEMS: the F2 family is now built from dynamically
#     created subclasses of F2Base so that REGISTRY[name]() works.
#     The F2 problems and their Σ̂ values are now computed.
#   - Standard NSGA-III niche count uses nearest reference direction.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

# Set to True to bypass the Jupyter safety check and force parallel execution.
FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v7_3")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)

# ---- External benchmark defaults -------------------------------------------
SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10
# ----------------------------------------------------------------------------


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    """Standard simplex-lattice points of Das & Dennis."""
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2

    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3

    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3

    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3

    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3

    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3

    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3

    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3

    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3

    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"

    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"

    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"

    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


# ---- F2 family: α-sensitive diagnostic wrapper ---------------------------- #
class F2Base(FuzzyBenchmark):
    """α-sensitive diagnostic wrapper: modulates spread by η(x) ∈ [0.15, 0.45]."""

    # NOTE: `name`, `n_var`, `n_obj` are provided as CLASS attributes by
    # _make_f2_subclass so that REGISTRY can be built before any instance
    # exists.  F2Base itself is abstract and is never registered directly.

    def __init__(self, base_cls=None, _base=None):
        # Accept either a base class (positional) or an already-built base
        # instance (keyword `_base`), so that dynamically created subclasses
        # can call super().__init__() with no arguments and get their
        # specific base class wired up.
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        # Only assign an instance-level name if the class did not already
        # define one (i.e. when F2Base is used directly, outside the
        # _make_f2_subclass factory).  This keeps the class-level name
        # authoritative for REGISTRY.
        if getattr(type(self), "name", "base") == "base":
            self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    """
    Create a named subclass of F2Base bound to a specific base class.

    The subclass exposes `name`, `n_var`, and `n_obj` as CLASS attributes
    so that REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS} registers
    every F2 variant under its own key.  Without the class-level `name`,
    every F2 subclass would inherit name="base" from FuzzyBenchmark and
    all but the last would silently vanish from REGISTRY, causing
    `get_extended_benchmark("F2FZDT1")` to raise ValueError.

    The subclass takes no required constructor arguments so that
    REGISTRY[name]() works uniformly with the other benchmark classes.
    """
    cls_name = f"F2{base_cls.__name__}"

    def _init(self, _b=base_cls, _name=cls_name):
        F2Base.__init__(self, _base=_b())
        # Redundant but harmless: the class attribute is already correct.
        self.name = _name

    return type(
        cls_name,
        (F2Base,),
        {
            "__init__": _init,
            "name": cls_name,
            "n_var": base_cls.n_var,
            "n_obj": base_cls.n_obj,
        },
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}

# ---- Import-time sanity check --------------------------------------------- #
# Catch the class-of-bug that silently dropped the F2 family in v7.2:
# every name in F2_NAMES must be a distinct key in REGISTRY.
_missing_f2 = [n for n in F2_NAMES if n not in REGISTRY]
if _missing_f2:
    raise RuntimeError(
        f"F2 family not registered correctly. Missing from REGISTRY: "
        f"{_missing_f2}. Check _make_f2_subclass: the subclass must expose "
        f"a class-level `name` attribute."
    )
_duplicate_names = [n for n in set(F2_NAMES) if F2_NAMES.count(n) > 1]
if _duplicate_names:
    raise RuntimeError(f"Duplicate F2 names: {_duplicate_names}")
del _missing_f2, _duplicate_names
# --------------------------------------------------------------------------- #


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A: INTERNAL ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """Aα-NSGA-II with the discrete 1/5-success mutation schedule."""

    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 5B: EXTERNAL BENCHMARK ALGORITHMS
# ============================================================================

class SPEA2:
    """SPEA2 with external archive and k-NN density estimation, fuzzy evaluation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        rank_of = np.zeros(n, dtype=int)
        for r, front in enumerate(fronts):
            for idx in front:
                rank_of[idx] = r
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    """MOEA/D with Tchebycheff scalarisation and Das-Dennis weight vectors."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    """NSGA-III with Das-Dennis reference points and niche preservation."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        """NSGA-III niche preservation on the alpha-weighted objectives."""
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        # Rank assignment
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        # Pre-normalize reference directions once
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                # All solutions on this front are accepted
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                # Niche preservation on the last front
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                # Niche counts from already-selected
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                # Sort last-front candidates by niche count
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    """Co-evolutionary multi-level selection GA with n_subpops demes."""

    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 6: METRICS
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300):
    benchmark = get_extended_benchmark(prob_name)
    hvs = []
    for a in alphas:
        algo = AAlphaNSGA2AdaptiveMutation(
            benchmark, pop_size=pop_size, n_gen=n_gen, seed=seed,
        )
        pop = algo.run()
        hv, _ = compute_pareto_hv(pop, fixed_alpha=a)
        hvs.append(hv)
    return float(max(hvs) - min(hvs)), hvs


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2': '#2ca02c',
    'MOEA/D': '#d62728',
    'NSGA-III': '#9467bd',
    'cMLSGA': '#7f7f7f',
}


def get_algo_config(pop_size, n_gen):
    common = {'pop_size': pop_size, 'n_gen': n_gen}
    return [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('SPEA2',
         SPEA2, {**common, 'alpha': FAIR_ALPHA,
                 'archive_size': SPEA2_ARCHIVE_SIZE,
                 'k_nn': SPEA2_KNN_K}),
        ('MOEA/D',
         MOEAD, {**common, 'alpha': FAIR_ALPHA,
                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                 'divisions': MOEAD_DIVISIONS}),
        ('NSGA-III',
         NSGA3, {**common, 'alpha': FAIR_ALPHA,
                 'divisions': NSGA3_DIVISIONS}),
        ('cMLSGA',
         cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                  'n_subpops': CMLSGA_N_SUBPOPS,
                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
    ]


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    alphas.append(float(np.mean(algo.get_alphas())))
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            ('EXTERNAL COMPARISON', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 11: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)


def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            order = np.argsort(F[:, 0])
            ax.plot(F[order, 0], F[order, 1], 'o-', markersize=5,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.2, alpha=0.85)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    elif n_obj == 3:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            ax.scatter(F[:, 0], F[:, 1], F[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=25,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.7, edgecolors='black', linewidths=0.3)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name}...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


def generate_sigma_hat_table(all_results, problems, save_dir):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. F1 problems are $\alpha$-invariant by construction '
        r'($\Sigma = 0$ analytically); F2 problems are $\alpha$-sensitive '
        r'($\Sigma > 0$).}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\mathrm{HV}(0.5)$ & $\mathrm{HV}(1.0)$ \\',
        r'\midrule',
    ]
    for row in rows:
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$)}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
        r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(
            f"{res.get('family', '--')} & {ref} vs. {algo} & "
            f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
            f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE "
          "(v7.3 — Adaptive α + Adaptive Mutation + Pareto Plots "
          "+ F2 + Σ̂ + External Benchmarks)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"Force parallel from Jupyter: {FORCE_PARALLEL_FROM_JUPYTER}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    internal_label = ', '.join(INTERNAL_ALGO_NAMES)
    external_label = ', '.join(EXTERNAL_ALGO_NAMES) if include_external \
        else '(skipped)'
    print(f"Internal algorithms ({len(INTERNAL_ALGO_NAMES)}): {internal_label}")
    print(f"External algorithms ({len(EXTERNAL_ALGO_NAMES)}): {external_label}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if CONFIG['use_parallel']:
        all_results = run_parallel(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            n_workers=CONFIG['n_workers'],
            checkpoint_path=checkpoint_path,
            verbose=not CONFIG['quiet'],
            monitor=None,
        )
    else:
        all_results = run_serial(
            CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
            CONFIG['n_gen'], CONFIG['seed_base'],
            checkpoint_path=checkpoint_path,
            verbose=True,
            monitor=None,
        )

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
    )

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir']
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in EXTERNAL_ALGO_NAMES:
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')
        print(f"\n  External results: "
              f"{os.path.join(CONFIG['external_dir'], 'external_results.csv')}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v7.3 '
                    '(adaptive α + adaptive mutation + Pareto plots '
                    '+ F2 family + Σ̂ + external benchmarks)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--no-sigma', action='store_true',
                        help='Skip α-sensitivity Σ̂ computation (fast)')
    parser.add_argument('--no-external', action='store_true',
                        help='Skip the four external benchmarks (fast)')
    parser.add_argument('--force-parallel', action='store_true',
                        help='Bypass Jupyter safety check for parallel mode')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v7.3 — Adaptive α + Adaptive Mutation + Pareto Plots + F2 + Σ̂ + External Benchmarks)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
Force parallel from Jupyter: False
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_3
Fair HV metric uses α = 0.75
Internal algorithms (5): Aα-NSGA-II (adaptive mutation), Aα-NSGA-II, NSGA-II, NSGA-II (high mutation), Self-Adaptive Mutation
External algorithms (4): SPEA2, MOEA/D, NSGA-III, cMLSGA
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_3\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v7_3\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M 

In [9]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v8.0)
#   Adaptive α + Adaptive Mutation + Fair HV + IGD/GD + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂ (multi-seed, fixed reference,
#     normalized)
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#     * TWO comparison modes:
#         - strict: external algos run with FIXED α = FAIR_ALPHA
#         - fair:   external algos run with ALPHA-ADAPTATION wrapper
#                   (same α gene as Aα-NSGA-II)
#
# v8.0 changes vs v7.3:
#   - FIXED: Σ̂ proxy is now multi-seed averaged, uses a FIXED reference
#     point computed from the union of all α-runs, and is normalized by
#     the mean HV. F1 problems now correctly report Σ̂ ≈ 0 (was showing
#     values > 1 due to stochastic noise being measured as α-sensitivity).
#   - ADDED: IGD and GD metrics computed against an empirical reference
#     front built from the union of all runs across all algorithms
#     (per problem). Aggregated, plotted, and tabulated.
#   - ADDED: AlphaWrapped{SPEA2,MOEAD,NSGA3,cMLSGA} — external baselines
#     augmented with the same α-adaptation mechanism as Aα-NSGA-II, so
#     the external comparison is now fair to both sides.
#   - ADDED: report_framing() emits a "combined mechanism vs. ablation"
#     narrative block, so the manuscript can correctly credit the
#     COMBINATION of adaptive-α + adaptive-mutation.
#   - ADDED: 3D Pareto fronts now also emit three pairwise 2D projections
#     and use downsampled markers; the raw 3D scatter is opt-in.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v8_0")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)
SIGMA_HAT_RUNS_PER_ALPHA = 5      # v8.0 FIX: multi-seed averaging
SIGMA_HAT_REF_QUANTILE   = 0.95   # reference point from union of fronts

SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10

# v8.0 FIX: External comparison mode
# 'fair'   -> external algos get α-adaptation wrapper
# 'strict' -> external algos run with fixed α = FAIR_ALPHA
# 'both'   -> run both; use 'fair' as primary, keep 'strict' as secondary
EXTERNAL_COMPARISON_MODE = 'both'


# ============================================================================
# PART 0: PROGRESS MONITOR  (unchanged from v7.3)
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS  (unchanged)
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS  (unchanged)
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS  (unchanged)
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2

    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3

    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3

    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3

    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3

    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3

    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3

    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3

    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3

    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"

    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"

    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"

    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


class F2Base(FuzzyBenchmark):
    """α-sensitive diagnostic wrapper: modulates spread by η(x) ∈ [0.15, 0.45]."""
    def __init__(self, base_cls=None, _base=None):
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        if getattr(type(self), "name", "base") == "base":
            self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    cls_name = f"F2{base_cls.__name__}"

    def _init(self, _b=base_cls, _name=cls_name):
        F2Base.__init__(self, _base=_b())
        self.name = _name

    return type(
        cls_name,
        (F2Base,),
        {
            "__init__": _init,
            "name": cls_name,
            "n_var": base_cls.n_var,
            "n_obj": base_cls.n_obj,
        },
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}

_missing_f2 = [n for n in F2_NAMES if n not in REGISTRY]
if _missing_f2:
    raise RuntimeError(
        f"F2 family not registered correctly. Missing from REGISTRY: "
        f"{_missing_f2}."
    )
del _missing_f2


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A: INTERNAL ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """Aα-NSGA-II with adaptive α + adaptive mutation (the full mechanism)."""

    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 5B: EXTERNAL BENCHMARK ALGORITHMS
# ============================================================================

class SPEA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        rank_of = np.zeros(n, dtype=int)
        for r, front in enumerate(fronts):
            for idx in front:
                rank_of[idx] = r
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 5C: ALPHA-WRAPPED EXTERNAL ALGORITHMS  (v8.0 NEW)
#
# These add the SAME α-adaptation mechanism as Aα-NSGA-II to the external
# baselines, so that the external comparison is fair to both sides.
# Each wrapped algorithm:
#   1. keeps its own selection / scalarization / archiving logic
#   2. carries an α gene per individual, evolved by BLX-α + Gaussian mutation
#   3. uses the α-weighted objective for its internal decisions
# ============================================================================

class _AlphaMixin:
    """Adds α-gene dynamics to an external algorithm.

    The mixin assumes the host class defines:
        self.alpha_L, self.alpha_U   -- α bounds
        self._alpha_rng              -- numpy Generator
        self.n_gen                   -- for cooling Gaussian sigma
    And that the host class inherits from a base that provides
    `self.benchmark`, `self.xl`, `self.xu`, `self.delta`.
    """

    def _init_alpha(self):
        self.alpha_L = 0.5
        self.alpha_U = 1.0
        self._alpha_rng = self.rng  # reuse host rng

    def _random_alpha(self):
        return float(self._alpha_rng.uniform(self.alpha_L, self.alpha_U))

    def _blx_alpha(self, a1, a2):
        gamma = self._alpha_rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1),
                             self.alpha_L, self.alpha_U))

    def _gauss_alpha(self, alpha, gen):
        sigma = 0.05 * max(0.0, 1.0 - gen / max(1, self.n_gen))
        return float(np.clip(alpha + self._alpha_rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))


class AlphaWrappedSPEA2(_AlphaMixin, SPEA2):
    """SPEA2 with the α-adaptation mechanism added."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        # disable the fixed alpha from SPEA2.__init__
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._blx_alpha(p1.alpha, p2.alpha)
                c2a = self._blx_alpha(p1.alpha, p2.alpha)
                c1a = self._gauss_alpha(c1a, gen)
                c2a = self._gauss_alpha(c2a, gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r
        self.population = self.archive
        return self.population


class AlphaWrappedMOEAD(_AlphaMixin, MOEAD):
    """MOEA/D with the α-adaptation mechanism added."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c1 = self._make(c1x, c1a)
                c2 = self._make(c2x, c2a)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population


class AlphaWrappedNSGA3(_AlphaMixin, NSGA3):
    """NSGA-III with the α-adaptation mechanism added."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population


class AlphaWrappedcMLSGA(_AlphaMixin, cMLSGA):
    """cMLSGA with the α-adaptation mechanism added."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def _evolve_deme(self, deme, n_gen):
        for gen in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population


# ============================================================================
# PART 6: METRICS  (HV, IGD, GD)  — v8.0: IGD/GD added
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


# ---- IGD / GD ------------------------------------------------------------- #

def _igd_gd_against_reference(F, ref_front):
    """IGD and GD of front F w.r.t. reference front ref_front.

    IGD(ref, F) = mean over p in ref of min_{q in F} ||p - q||
    GD(F, ref)  = mean over q in F of min_{p in ref} ||q - p||
    """
    F = np.asarray(F, dtype=float)
    ref_front = np.asarray(ref_front, dtype=float)
    if F.size == 0 or ref_front.size == 0:
        return float('nan'), float('nan')
    # pairwise distances (m x n)
    D = np.sqrt(((ref_front[:, None, :] - F[None, :, :]) ** 2).sum(axis=2))
    igd = float(np.mean(np.min(D, axis=1)))
    gd = float(np.mean(np.min(D, axis=0)))
    return igd, gd


def build_empirical_reference_front(all_fronts, k=100, seed=0):
    """Union all fronts, keep the non-dominated subset, then downsample to k."""
    if not all_fronts:
        return np.empty((0, 0))
    stacked = np.vstack([F for F in all_fronts if F.size > 0])
    if stacked.size == 0:
        return np.empty((0, 0))
    # Deduplicate within tolerance
    uniq = np.unique(np.round(stacked, decimals=10), axis=0)
    # Non-dominated sort on the empirical front
    # Convert to TFN-like [L, M, U] for the existing sorter by using the
    # same point for L/M/U (this is a valid degenerate fuzzy number)
    M = uniq.shape[1]
    F_tfn = np.stack([uniq, uniq, uniq], axis=2)
    fronts = fast_nondominated_sort_fuzzy(F_tfn)
    front0 = fronts[0] if fronts else []
    nd = uniq[front0] if len(front0) > 0 else uniq
    if len(nd) > k:
        # Downsample uniformly in index space after sorting by the first
        # objective, so the reference front is spread out
        order = np.argsort(nd[:, 0])
        idx = np.linspace(0, len(nd) - 1, k).astype(int)
        nd = nd[order][idx]
    return nd


# ============================================================================
# PART 6B: Σ̂ PROXY  (v8.0 FIX — multi-seed, fixed ref, normalized)
# ============================================================================

def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300,
                      n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA,
                      ref_quantile=SIGMA_HAT_REF_QUANTILE):
    """α-sensitivity proxy.

    v8.0 fixes vs v7.3:
      1. Multi-seed averaging: for each α we run `n_runs_per_alpha` seeds
         and average the fixed-α HV. This removes the stochastic noise that
         previously dominated for F1 problems.
      2. Fixed reference point: a single reference point is built from the
         union of all fronts across all α values and seeds, so HV values
         are comparable across α.
      3. Normalized: Σ̂ is (max-min)/mean of the α-averaged HVs, giving a
         dimensionless relative sensitivity.
    """
    benchmark = get_extended_benchmark(prob_name)
    algo = AAlphaNSGA2AdaptiveMutation

    # Step 1: collect fronts for every (alpha, seed) pair
    per_alpha_fronts = {a: [] for a in alphas}
    rng = np.random.default_rng(seed)
    for a in alphas:
        for r in range(n_runs_per_alpha):
            s = int(rng.integers(0, 2**31 - 1))
            algo_obj = algo(benchmark, pop_size=pop_size, n_gen=n_gen, seed=s)
            pop = algo_obj.run()
            F = _fixed_alpha_objectives(pop, alpha=a)
            if F.size > 0:
                per_alpha_fronts[a].append(F)

    # Step 2: build a fixed reference point from the union of all fronts
    all_F = [F for a in alphas for F in per_alpha_fronts[a]]
    if not all_F:
        return float('nan'), [float('nan')] * len(alphas)
    stacked = np.vstack(all_F)
    # reference point: slightly beyond the worst corner
    ref = np.quantile(stacked, ref_quantile, axis=0) * 1.05 + 1e-6

    # Step 3: fixed-α HV per α (mean over seeds), with the SAME ref
    mean_hvs = []
    for a in alphas:
        hvs_a = []
        for F in per_alpha_fronts[a]:
            mask = np.all(F <= ref, axis=1)
            F_clipped = F[mask]
            if F_clipped.size == 0:
                hvs_a.append(0.0)
            else:
                hvs_a.append(hypervolume(F_clipped, ref))
        mean_hvs.append(float(np.mean(hvs_a)) if hvs_a else 0.0)

    # Step 4: normalized sensitivity
    mean_hv = float(np.mean(mean_hvs))
    sigma = (max(mean_hvs) - min(mean_hvs)) / mean_hv if mean_hv > EPS else 0.0
    return float(sigma), mean_hvs


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

# v8.0 FIX: external algos are now wrapped with α-adaptation.
# Names below are the "fair" versions (primary comparison).
EXTERNAL_ALGO_NAMES = [
    'SPEA2+α',
    'MOEA/D+α',
    'NSGA-III+α',
    'cMLSGA+α',
]

# Strict versions (fixed α = FAIR_ALPHA) are kept as secondary.
EXTERNAL_ALGO_NAMES_STRICT = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2+α': '#2ca02c',
    'MOEA/D+α': '#d62728',
    'NSGA-III+α': '#9467bd',
    'cMLSGA+α': '#7f7f7f',
    # strict (no α) versions
    'SPEA2': '#98df8a',
    'MOEA/D': '#ff9896',
    'NSGA-III': '#c5b0d5',
    'cMLSGA': '#c7c7c7',
}


def get_algo_config(pop_size, n_gen, external_mode=None):
    """Return the algorithm list.

    external_mode:
      'fair'   -> only α-wrapped externals (default for the main comparison)
      'strict' -> only fixed-α externals (secondary comparison)
      'both'   -> both sets; fair first
    """
    if external_mode is None:
        external_mode = EXTERNAL_COMPARISON_MODE

    common = {'pop_size': pop_size, 'n_gen': n_gen}
    algos = [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
    ]

    if external_mode in ('fair', 'both'):
        algos.extend([
            ('SPEA2+α',
             AlphaWrappedSPEA2, {**common,
                                 'archive_size': SPEA2_ARCHIVE_SIZE,
                                 'k_nn': SPEA2_KNN_K}),
            ('MOEA/D+α',
             AlphaWrappedMOEAD, {**common,
                                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                                 'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III+α',
             AlphaWrappedNSGA3, {**common,
                                 'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA+α',
             AlphaWrappedcMLSGA, {**common,
                                  'n_subpops': CMLSGA_N_SUBPOPS,
                                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    if external_mode in ('strict', 'both'):
        algos.extend([
            ('SPEA2',
             SPEA2, {**common, 'alpha': FAIR_ALPHA,
                     'archive_size': SPEA2_ARCHIVE_SIZE,
                     'k_nn': SPEA2_KNN_K}),
            ('MOEA/D',
             MOEAD, {**common, 'alpha': FAIR_ALPHA,
                     'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                     'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III',
             NSGA3, {**common, 'alpha': FAIR_ALPHA,
                     'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA',
             cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                      'n_subpops': CMLSGA_N_SUBPOPS,
                      'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    return algos


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas, igds, gds = [], [], [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    try:
                        alphas.append(float(np.mean(algo.get_alphas())))
                    except Exception:
                        pass
                # IGD/GD will be added later in the post-processing pass
                # (needs the empirical reference front across algorithms)
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
                # IGD/GD placeholders — filled in compute_igd_gd_pass()
                'igd_mean': np.nan,
                'igd_std': np.nan,
                'igd_all': [],
                'gd_mean': np.nan,
                'gd_std': np.nan,
                'gd_all': [],
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 8B: POST-HOC IGD/GD PASS
# ============================================================================

def compute_igd_gd_pass(all_results, problems, pop_size, n_gen,
                        seed_base, n_runs, verbose=True):
    """Second pass: for each problem, rebuild the empirical reference front
    from all algorithms × all runs, then compute IGD/GD per algorithm."""
    if verbose:
        print("\n" + "=" * 78)
        print("IGD/GD POST-PROCESSING PASS")
        print("=" * 78)
    algos = get_algo_config(pop_size, n_gen)
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        try:
            benchmark = get_extended_benchmark(prob_name)
        except Exception:
            continue
        # Collect all fronts from all algorithms × all seeds
        all_fronts = []
        per_algo_fronts = {a: [] for a, _, _ in algos}
        for algo_name, algo_cls, kwargs in algos:
            if algo_name not in res:
                continue
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    continue
                F = _fixed_alpha_objectives(pop, alpha=FAIR_ALPHA)
                if F.size > 0:
                    all_fronts.append(F)
                    per_algo_fronts[algo_name].append(F)
        ref_front = build_empirical_reference_front(all_fronts, k=100)
        if ref_front.size == 0:
            continue
        for algo_name, fronts in per_algo_fronts.items():
            igds, gds = [], []
            for F in fronts:
                igd, gd = _igd_gd_against_reference(F, ref_front)
                if not np.isnan(igd):
                    igds.append(igd)
                if not np.isnan(gd):
                    gds.append(gd)
            res[algo_name]['igd_mean'] = float(np.mean(igds)) if igds else np.nan
            res[algo_name]['igd_std'] = float(np.std(igds)) if igds else np.nan
            res[algo_name]['igd_all'] = igds
            res[algo_name]['gd_mean'] = float(np.mean(gds)) if gds else np.nan
            res[algo_name]['gd_std'] = float(np.std(gds)) if gds else np.nan
            res[algo_name]['gd_all'] = gds
        if verbose:
            print(f"  {prob_name:<14} ref=|{len(ref_front)}|  "
                  f"IGD {res.get(list(per_algo_fronts)[0], {}).get('igd_mean', float('nan')):.4f} ...")
    return all_results


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    agg_igd = {a: [] for a in algo_names}
    agg_gd = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)
        # IGD/GD aggregation (lower is better)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            if not np.isnan(igd):
                agg_igd[a].append(igd)
            if not np.isnan(gd):
                agg_gd[a].append(gd)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            ('EXTERNAL COMPARISON (fair, α-wrapped)', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs (HV) ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean IGD / GD (lower is better) ---")
    igd_summary = {}
    gd_summary = {}
    for algo in algo_names:
        if agg_igd[algo]:
            igd_summary[algo] = float(np.mean(agg_igd[algo]))
        if agg_gd[algo]:
            gd_summary[algo] = float(np.mean(agg_gd[algo]))
        if algo in igd_summary or algo in gd_summary:
            print(f"{algo:<38} IGD={igd_summary.get(algo, float('nan')):.4f}  "
                  f"GD={gd_summary.get(algo, float('nan')):.4f}")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank,
        'agg_hv': agg_hv,
        'agg_igd': agg_igd,
        'agg_gd': agg_gd,
        'igd_summary': igd_summary,
        'gd_summary': gd_summary,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 10B: FRAMING REPORT  (v8.0 NEW)
# ============================================================================

def report_framing(stats_results, algo_names):
    """Print a narrative framing block that correctly credits the COMBINED
    mechanism (adaptive α + adaptive mutation), and reports which ablation
    components are responsible for the gains."""
    print("\n" + "=" * 90)
    print("FRAMING REPORT — What the data actually supports")
    print("=" * 90)

    # Compare the full mechanism against the ablations
    ref = 'Aα-NSGA-II (adaptive mutation)'
    full_hv = np.mean(stats_results['agg_hv'].get(ref, [np.nan]))
    full_rank = np.mean(stats_results['agg_rank'].get(ref, [np.nan]))

    print(f"\nReference (full mechanism): {ref}")
    print(f"  Normalized HV = {full_hv:.4f}")
    print(f"  Mean rank      = {full_rank:.2f}")

    print("\nComponent ablations (all vs. reference):")
    for algo in algo_names:
        if algo == ref or not stats_results['agg_hv'].get(algo):
            continue
        hv = np.mean(stats_results['agg_hv'][algo])
        rank = np.mean(stats_results['agg_rank'][algo])
        pw = stats_results['pairwise'].get(algo, {})
        sig = pw.get('significant', 'n/a')
        p = pw.get('p', float('nan'))
        print(f"  {algo:<38} HV={hv:.4f}  rank={rank:5.2f}  "
              f"p={p:.2e}  sig={sig}")

    # Explicit claim guidance
    tied = []
    for algo in algo_names:
        if algo == ref:
            continue
        pw = stats_results['pairwise'].get(algo, {})
        if pw.get('significant') == 'no':
            tied.append(algo)
    print("\n--- Recommended claim language ---")
    if tied:
        print("The following algorithms are STATISTICALLY TIED with the")
        print("reference under Bonferroni-corrected Wilcoxon:")
        for a in tied:
            print(f"  - {a}")
        print("\nRecommendation: the contribution is the COMBINATION of")
        print("adaptive-α and adaptive-mutation, not adaptive-α alone.")
        print("State this explicitly in the abstract and contributions.")
    else:
        print("The reference is statistically significantly better than all")
        print("ablations. The contribution can be attributed to the full")
        print("combined mechanism.")
    print("=" * 90)


# ============================================================================
# PART 11: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_igd_gd_bars(stats_results, algo_names, save_dir):
    """v8.0 NEW: bar chart comparing IGD and GD."""
    igd = stats_results.get('igd_summary', {})
    gd = stats_results.get('gd_summary', {})
    if not igd and not gd:
        return
    print("  Generating IGD/GD bar chart...")
    algos = [a for a in algo_names if a in igd or a in gd]
    igd_vals = [igd.get(a, np.nan) for a in algos]
    gd_vals = [gd.get(a, np.nan) for a in algos]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, vals, title, ylabel in (
            (axes[0], igd_vals, 'IGD (lower is better)', 'Mean IGD'),
            (axes[1], gd_vals, 'GD (lower is better)', 'Mean GD')):
        ypos = np.arange(len(algos))
        colors = [ALGO_COLORS.get(a, '#cccccc') for a in algos]
        ax.barh(ypos, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(ypos)
        ax.set_yticklabels(algos, fontsize=9)
        ax.set_xlabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.grid(axis='x', alpha=0.3)
        ax.invert_yaxis()
    plt.tight_layout()
    path = os.path.join(save_dir, 'igd_gd.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir, stats_results=None):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)
    if stats_results is not None:
        plot_igd_gd_bars(stats_results, algo_names, save_dir)


# ============================================================================
# PART 11B: PARETO FRONT PLOTS  (v8.0 FIX: 2D projections + downsampling)
# ============================================================================

def _downsample_front(F, max_points=60):
    if len(F) <= max_points:
        return F
    order = np.argsort(F[:, 0])
    idx = np.linspace(0, len(F) - 1, max_points).astype(int)
    return F[order][idx]


def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None,
                             make_3d=True, make_2d_projections=True,
                             max_points=60):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    # ---- 2D problems: single 2D plot --------------------------------------
    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            Fd = _downsample_front(F, max_points)
            order = np.argsort(Fd[:, 0])
            ax.plot(Fd[order, 0], Fd[order, 1], 'o-', markersize=4,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.0, alpha=0.8)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")
        return

    # ---- 3D problems: three 2D projections + optional 3D -----------------
    if n_obj >= 3:
        pairs = [(0, 1), (0, 2), (1, 2)]
        proj_titles = [('$f_1$', '$f_2$'),
                       ('$f_1$', '$f_3$'),
                       ('$f_2$', '$f_3$')]
        if make_2d_projections:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            for ax, (i, j), (xl, yl) in zip(axes, pairs, proj_titles):
                for algo_name, F in fronts.items():
                    if len(F) == 0:
                        continue
                    Fd = _downsample_front(F, max_points)
                    ax.scatter(Fd[:, i], Fd[:, j], s=18, alpha=0.65,
                               label=algo_name if i == 0 and j == 1 else None,
                               color=ALGO_COLORS.get(algo_name, None),
                               edgecolors='black', linewidths=0.2)
                ax.set_xlabel(xl, fontsize=11)
                ax.set_ylabel(yl, fontsize=11)
                ax.grid(alpha=0.3)
            axes[0].legend(fontsize=6, loc='best')
            fig.suptitle(
                f'{prob_name} — Pareto front 2D projections '
                f'(α = {alpha}, downsampled to {max_points})',
                fontsize=12)
            plt.tight_layout()
            path = os.path.join(save_dir, f'pareto_{prob_name}_2dproj.png')
            plt.savefig(path, dpi=200, bbox_inches='tight')
            plt.close()
            print(f"    Saved: {path}")

        if make_3d:
            from mpl_toolkits.mplot3d import Axes3D  # noqa
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            for algo_name, F in fronts.items():
                if len(F) == 0:
                    continue
                Fd = _downsample_front(F, max_points)
                ax.scatter(Fd[:, 0], Fd[:, 1], Fd[:, 2],
                           label=f"{algo_name} (|PF|={len(F)})", s=12,
                           color=ALGO_COLORS.get(algo_name, None),
                           alpha=0.6, edgecolors='black', linewidths=0.15)
            ax.set_xlabel('$f_1$')
            ax.set_ylabel('$f_2$')
            ax.set_zlabel('$f_3$')
            ax.set_title(
                f'{prob_name} — Pareto fronts (α = {alpha}, '
                f'downsampled to {max_points})')
            ax.legend(fontsize=6, loc='upper left')
            plt.tight_layout()
            path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
            plt.savefig(path, dpi=200, bbox_inches='tight')
            plt.close()
            print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name}...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
                make_3d=True,
                make_2d_projections=True,
                max_points=40,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


# ============================================================================
# PART 11C: Σ̂ TABLE
# ============================================================================

def generate_sigma_hat_table(all_results, problems, save_dir,
                             n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂  (v8.0: multi-seed, fixed ref)")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
                n_runs_per_alpha=n_runs_per_alpha,
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    # Sort so F1 comes before F2 for the table
    df = df.sort_values(['family', 'problem']).reset_index(drop=True)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. $\widehat{\Sigma}$ is the normalized spread of the '
        r'fixed-$\alpha$ hypervolume across $\alpha \in \{0.5, 0.75, 1.0\}$, '
        r'averaged over '
        f'{n_runs_per_alpha} seeds and computed against a '
        r'single reference point (union of all fronts). '
        r'F1 problems are $\alpha$-invariant by construction, so their '
        r'$\widehat{\Sigma}$ should be near zero; F2 problems are '
        r'$\alpha$-sensitive.}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\overline{\mathrm{HV}}(0.5)$ & '
        r'$\overline{\mathrm{HV}}(1.0)$ \\',
        r'\midrule',
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")

    # Sanity check: F1 Σ̂ should be << F2 Σ̂
    f1 = df[df['family'] == 'F1']['sigma_hat'].values
    f2 = df[df['family'] == 'F2']['sigma_hat'].values
    if len(f1) and len(f2):
        print(f"\n  Sanity check: mean Σ̂(F1) = {np.mean(f1):.4f}  "
              f"mean Σ̂(F2) = {np.mean(f2):.4f}")
        if np.mean(f1) >= np.mean(f2):
            print("  ⚠ WARNING: F1 mean Σ̂ >= F2 mean Σ̂ — check reference "
                  "construction.")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    igds = {a: [] for a in algo_names}
    gds = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            if not np.isnan(igd):
                igds[a].append(igd)
            if not np.isnan(gd):
                gds[a].append(gd)

    # Overall table with HV + IGD + GD
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$), IGD, and GD. '
        r'IGD/GD are computed against an empirical reference front built '
        r'from the union of all runs across all algorithms; lower is '
        r'better.}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lcccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Mean Rank & Wins & IGD & GD \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        igd_s = (f'{np.mean(igds[algo]):.4f}' if igds[algo] else '--')
        gd_s = (f'{np.mean(gds[algo]):.4f}' if gds[algo] else '--')
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} & {igd_s} & {gd_s} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # Family table
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # Pairwise stats
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
        r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(
            f"{res.get('family', '--')} & {ref} vs. {algo} & "
            f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
            f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    # Bootstrap CIs
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None,
                   external_mode=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if external_mode is not None:
        global EXTERNAL_COMPARISON_MODE
        EXTERNAL_COMPARISON_MODE = external_mode
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True, external_mode=None,
         skip_main_runs=False):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet,
                   external_mode=external_mode)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v8.0 — adaptive α + adaptive "
          "mutation + fair external + IGD/GD + Σ̂ fixed)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"External comparison mode: {EXTERNAL_COMPARISON_MODE}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if not skip_main_runs:
        if CONFIG['use_parallel']:
            all_results = run_parallel(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                n_workers=CONFIG['n_workers'],
                checkpoint_path=checkpoint_path,
                verbose=not CONFIG['quiet'],
                monitor=None,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                checkpoint_path=checkpoint_path,
                verbose=True,
                monitor=None,
            )
    else:
        # Load from checkpoint
        if os.path.exists(checkpoint_path):
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            print(f"\n[skip_main_runs] Loaded {len(all_results)} problems")
        else:
            all_results = {}

    # v8.0 NEW: IGD/GD post-processing pass (separate from main runs)
    if all_results:
        all_results = compute_igd_gd_pass(
            all_results, CONFIG['problems'],
            CONFIG['pop_size'], CONFIG['n_gen'],
            CONFIG['seed_base'], CONFIG['n_runs'],
            verbose=True,
        )
        _save_checkpoint(all_results, checkpoint_path)

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
    )

    # v8.0 NEW: explicit framing report
    report_framing(stats_results, ALGO_NAMES)

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir'],
        stats_results=stats_results,
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in (EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT):
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                    'igd_mean': results[algo].get('igd_mean', np.nan),
                    'gd_mean': results[algo].get('gd_mean', np.nan),
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')
        print(f"\n  External results: "
              f"{os.path.join(CONFIG['external_dir'], 'external_results.csv')}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
                'igd_mean': res.get('igd_mean', np.nan),
                'gd_mean': res.get('gd_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v8.0 '
                    '(adaptive α + adaptive mutation + fair external '
                    '+ IGD/GD + Σ̂ fixed)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true',
                        help='Skip Pareto front plots (fast)')
    parser.add_argument('--full-pareto', action='store_true',
                        help='Pareto plots at full pop/gen (slow)')
    parser.add_argument('--no-sigma', action='store_true',
                        help='Skip α-sensitivity Σ̂ computation (fast)')
    parser.add_argument('--no-external', action='store_true',
                        help='Skip the four external benchmarks (fast)')
    parser.add_argument('--external-mode', choices=['fair', 'strict', 'both'],
                        default='both',
                        help='External comparison mode')
    parser.add_argument('--force-parallel', action='store_true',
                        help='Bypass Jupyter safety check for parallel mode')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    parser.add_argument('--sigma-only', action='store_true',
                        help='Skip main runs and only recompute Σ̂ table '
                             'from existing checkpoint')
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external,
         external_mode=args.external_mode,
         skip_main_runs=args.sigma_only)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v8.0 — adaptive α + adaptive mutation + fair external + IGD/GD + Σ̂ fixed)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
External comparison mode: both
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_0
Fair HV metric uses α = 0.75
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_0\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_0\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_0\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_0\supplementary_latex
  En

MemoryError: Unable to allocate 806. MiB for an array with shape (5934, 5934, 3) and data type float64

In [1]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v8.1)
#   Adaptive α + Adaptive Mutation + Fair HV + IGD/GD + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂ (multi-seed, fixed reference,
#     normalized)
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#     * TWO comparison modes:
#         - strict: external algos run with FIXED α = FAIR_ALPHA
#         - fair:   external algos run with ALPHA-ADAPTATION wrapper
#
# v8.1 changes vs v8.0:
#   - FIXED: MemoryError in build_empirical_reference_front — the union is
#     now capped at REF_FRONT_MAX_INPUT (=1200) BEFORE the O(N^2*M)
#     non-dominated sort. At 1200 points and M=3, the sorter allocates
#     ~100 MB peak, which is safe on any modern machine.
#   - FIXED: IGD/GD post-processing is now RESUMABLE via a per-problem
#     checkpoint (igd_gd_checkpoint.pkl), so a crash costs at most one
#     problem instead of the whole pass.
#   - FIXED: per-algorithm balanced sampling in the IGD/GD reference
#     front — each algorithm contributes at most IGD_PER_ALGO_CAP points
#     before the union cap, so no single algorithm can bias the reference.
#   - FIXED: IGD_SKIP is now explicit and documented (only FScal5000 and
#     FScal10000 — where population=60 is too small relative to the
#     decision space for the empirical reference front to be meaningful).
#   - ADDED: normalized IGD / GD (divided by the reference front's
#     bounding-box diagonal) so cross-problem comparisons are meaningful
#     when objective ranges vary by orders of magnitude.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v8_1")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)
SIGMA_HAT_RUNS_PER_ALPHA = 5
SIGMA_HAT_REF_QUANTILE   = 0.95

SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10

EXTERNAL_COMPARISON_MODE = 'both'

# ---- v8.1 IGD/GD memory-safety and fairness tunables ---------------------- #
REF_FRONT_MAX_INPUT  = 1200    # cap on points fed into the O(N^2*M) sorter
REF_FRONT_MAX_OUTPUT = 100     # target size of the reference front
IGD_PER_ALGO_CAP     = 300     # per-algorithm point cap before union
IGD_SKIP = {'FScal5000', 'FScal10000'}
# --------------------------------------------------------------------------- #


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================

class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


def monitor_progress(output_dir, watch=False, interval=10):
    json_path = os.path.join(output_dir, 'progress.json')
    if not os.path.exists(json_path):
        print(f"No progress.json found at {json_path}")
        return

    def _snap():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            print(f"(read error: {e})")
            return None

    def _print(d):
        if d is None:
            return
        pct = d['percent']
        bar_w = 30
        filled = int(round(bar_w * pct / 100))
        bar = '█' * filled + '░' * (bar_w - filled)
        print("=" * 78)
        print(f"  Progress  [{bar}] {d['completed']}/{d['total']} ({pct:.1f}%)")
        print(f"  Elapsed   {d['elapsed_human']:>10}")
        if d.get('active_count'):
            print(f"  Active    {d['active_count']:>10}")
        print(f"  Updated   {d['updated_at']}")
        print("=" * 78)

    if not watch:
        _print(_snap())
        return
    try:
        while True:
            d = _snap()
            os.system('cls' if os.name == 'nt' else 'clear')
            _print(d)
            if d and d.get('final'):
                print("\nFinal state reached.")
                break
            time.sleep(interval)
    except KeyboardInterrupt:
        print("\n(monitor stopped)")


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    num = U[None, :, :] - L[:, None, :]
    den = W[:, None, :] + W[None, :, :]
    safe_den = np.where(den > EPS, den, 1.0)
    with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
        cred = np.clip(num / safe_den, 0.0, 1.0)
    degenerate = den <= EPS
    cred = np.where(degenerate,
                    (U[:, None, :] <= L[None, :, :]).astype(float),
                    cred)
    weak_all = np.all(cred >= 0.5, axis=2)
    strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
    dom = weak_all & strict_any
    np.fill_diagonal(dom, False)
    n = (dom & ~dom.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(dom[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================

class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2

    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2

    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3

    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3

    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3

    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2

    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3

    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3

    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3

    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3

    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3

    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3

    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3

    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3

    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3

    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3

    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3

    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"

    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj

    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"

    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"

    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3

    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


class F2Base(FuzzyBenchmark):
    """α-sensitive diagnostic wrapper: modulates spread by η(x) ∈ [0.15, 0.45]."""

    def __init__(self, base_cls=None, _base=None):
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        if getattr(type(self), "name", "base") == "base":
            self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    cls_name = f"F2{base_cls.__name__}"

    def _init(self, _b=base_cls, _name=cls_name):
        F2Base.__init__(self, _base=_b())
        self.name = _name

    return type(
        cls_name,
        (F2Base,),
        {
            "__init__": _init,
            "name": cls_name,
            "n_var": base_cls.n_var,
            "n_obj": base_cls.n_obj,
        },
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}

_missing_f2 = [n for n in F2_NAMES if n not in REGISTRY]
if _missing_f2:
    raise RuntimeError(
        f"F2 family not registered correctly. Missing from REGISTRY: "
        f"{_missing_f2}."
    )
del _missing_f2


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A: INTERNAL ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    """Aα-NSGA-II with adaptive α + adaptive mutation (the full mechanism)."""

    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


# ============================================================================
# PART 5B: EXTERNAL BENCHMARK ALGORITHMS
# ============================================================================

class SPEA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        rank_of = np.zeros(n, dtype=int)
        for r, front in enumerate(fronts):
            for idx in front:
                rank_of[idx] = r
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


# ============================================================================
# PART 5C: ALPHA-WRAPPED EXTERNAL ALGORITHMS
# ============================================================================

class _AlphaMixin:
    def _init_alpha(self):
        self.alpha_L = 0.5
        self.alpha_U = 1.0
        self._alpha_rng = self.rng

    def _random_alpha(self):
        return float(self._alpha_rng.uniform(self.alpha_L, self.alpha_U))

    def _blx_alpha(self, a1, a2):
        gamma = self._alpha_rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1),
                             self.alpha_L, self.alpha_U))

    def _gauss_alpha(self, alpha, gen):
        sigma = 0.05 * max(0.0, 1.0 - gen / max(1, self.n_gen))
        return float(np.clip(alpha + self._alpha_rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))


class AlphaWrappedSPEA2(_AlphaMixin, SPEA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._blx_alpha(p1.alpha, p2.alpha)
                c2a = self._blx_alpha(p1.alpha, p2.alpha)
                c1a = self._gauss_alpha(c1a, gen)
                c2a = self._gauss_alpha(c2a, gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r
        self.population = self.archive
        return self.population


class AlphaWrappedMOEAD(_AlphaMixin, MOEAD):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c1 = self._make(c1x, c1a)
                c2 = self._make(c2x, c2a)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population


class AlphaWrappedNSGA3(_AlphaMixin, NSGA3):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population


class AlphaWrappedcMLSGA(_AlphaMixin, cMLSGA):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def _evolve_deme(self, deme, n_gen):
        for gen in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population


# ============================================================================
# PART 6: METRICS  (HV, IGD, GD)
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    return np.array([
        alpha * ind.fuzzy_obj[:, 1]
        + (1 - alpha) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
        for ind in pf
    ])


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA):
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def _igd_gd_against_reference(F, ref_front):
    """IGD and GD of front F w.r.t. reference front ref_front."""
    F = np.asarray(F, dtype=float)
    ref_front = np.asarray(ref_front, dtype=float)
    if F.size == 0 or ref_front.size == 0:
        return float('nan'), float('nan')
    D = np.sqrt(((ref_front[:, None, :] - F[None, :, :]) ** 2).sum(axis=2))
    igd = float(np.mean(np.min(D, axis=1)))
    gd = float(np.mean(np.min(D, axis=0)))
    return igd, gd


def _ref_front_diagonal(ref_front):
    """Bounding-box diagonal of the reference front, for normalization."""
    ref_front = np.asarray(ref_front, dtype=float)
    if ref_front.size == 0:
        return 1.0
    span = ref_front.max(axis=0) - ref_front.min(axis=0)
    diag = float(np.sqrt((span ** 2).sum()))
    return diag if diag > EPS else 1.0


# ---- v8.1: memory-safe empirical reference front -------------------------- #

def build_empirical_reference_front(all_fronts,
                                    k=REF_FRONT_MAX_OUTPUT,
                                    seed=0,
                                    max_sort_size=REF_FRONT_MAX_INPUT):
    """Union all fronts, keep the non-dominated subset, downsample to k.

    v8.1 memory-safety fix:
        - The union is capped at `max_sort_size` (default 1200) BEFORE the
          O(N^2 * M) vectorised sorter is called. At 1200 points and M=3,
          the sorter's peak allocation is ~100 MB, which is safe on any
          modern machine. Previously the union reached ~6000 points for
          FDTLZ3/FDTLZ4, which required ~1 GB peak and caused MemoryError.
    """
    if not all_fronts:
        return np.empty((0, 0))
    stacked = np.vstack([F for F in all_fronts if F.size > 0])
    if stacked.size == 0:
        return np.empty((0, 0))

    # Subsample BEFORE the non-dominated sort.
    if len(stacked) > max_sort_size:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(stacked), size=max_sort_size, replace=False)
        stacked = stacked[idx]

    # Deduplicate (cheap; collapses exact duplicates from repeated runs).
    uniq = np.unique(np.round(stacked, decimals=10), axis=0)

    # Vectorised sorter is safe below ~1200 points.
    M = uniq.shape[1]
    F_tfn = np.stack([uniq, uniq, uniq], axis=2)
    fronts = fast_nondominated_sort_fuzzy(F_tfn)
    front0 = fronts[0] if fronts else []
    nd = uniq[front0] if len(front0) > 0 else uniq

    if len(nd) > k:
        order = np.argsort(nd[:, 0])
        idx = np.linspace(0, len(nd) - 1, k).astype(int)
        nd = nd[order][idx]
    return nd


# ============================================================================
# PART 6B: Σ̂ PROXY  (multi-seed, fixed reference, normalized)
# ============================================================================

def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300,
                      n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA,
                      ref_quantile=SIGMA_HAT_REF_QUANTILE):
    """α-sensitivity proxy: normalized spread of fixed-α HV across α values."""
    benchmark = get_extended_benchmark(prob_name)
    algo = AAlphaNSGA2AdaptiveMutation

    per_alpha_fronts = {a: [] for a in alphas}
    rng = np.random.default_rng(seed)
    for a in alphas:
        for r in range(n_runs_per_alpha):
            s = int(rng.integers(0, 2**31 - 1))
            algo_obj = algo(benchmark, pop_size=pop_size, n_gen=n_gen, seed=s)
            pop = algo_obj.run()
            F = _fixed_alpha_objectives(pop, alpha=a)
            if F.size > 0:
                per_alpha_fronts[a].append(F)

    all_F = [F for a in alphas for F in per_alpha_fronts[a]]
    if not all_F:
        return float('nan'), [float('nan')] * len(alphas)
    stacked = np.vstack(all_F)
    ref = np.quantile(stacked, ref_quantile, axis=0) * 1.05 + 1e-6

    mean_hvs = []
    for a in alphas:
        hvs_a = []
        for F in per_alpha_fronts[a]:
            mask = np.all(F <= ref, axis=1)
            F_clipped = F[mask]
            if F_clipped.size == 0:
                hvs_a.append(0.0)
            else:
                hvs_a.append(hypervolume(F_clipped, ref))
        mean_hvs.append(float(np.mean(hvs_a)) if hvs_a else 0.0)

    mean_hv = float(np.mean(mean_hvs))
    sigma = (max(mean_hvs) - min(mean_hvs)) / mean_hv if mean_hv > EPS else 0.0
    return float(sigma), mean_hvs


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2+α',
    'MOEA/D+α',
    'NSGA-III+α',
    'cMLSGA+α',
]

EXTERNAL_ALGO_NAMES_STRICT = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2+α': '#2ca02c',
    'MOEA/D+α': '#d62728',
    'NSGA-III+α': '#9467bd',
    'cMLSGA+α': '#7f7f7f',
    'SPEA2': '#98df8a',
    'MOEA/D': '#ff9896',
    'NSGA-III': '#c5b0d5',
    'cMLSGA': '#c7c7c7',
}


def get_algo_config(pop_size, n_gen, external_mode=None):
    if external_mode is None:
        external_mode = EXTERNAL_COMPARISON_MODE

    common = {'pop_size': pop_size, 'n_gen': n_gen}
    algos = [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
    ]

    if external_mode in ('fair', 'both'):
        algos.extend([
            ('SPEA2+α',
             AlphaWrappedSPEA2, {**common,
                                 'archive_size': SPEA2_ARCHIVE_SIZE,
                                 'k_nn': SPEA2_KNN_K}),
            ('MOEA/D+α',
             AlphaWrappedMOEAD, {**common,
                                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                                 'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III+α',
             AlphaWrappedNSGA3, {**common,
                                 'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA+α',
             AlphaWrappedcMLSGA, {**common,
                                  'n_subpops': CMLSGA_N_SUBPOPS,
                                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    if external_mode in ('strict', 'both'):
        algos.extend([
            ('SPEA2',
             SPEA2, {**common, 'alpha': FAIR_ALPHA,
                     'archive_size': SPEA2_ARCHIVE_SIZE,
                     'k_nn': SPEA2_KNN_K}),
            ('MOEA/D',
             MOEAD, {**common, 'alpha': FAIR_ALPHA,
                     'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                     'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III',
             NSGA3, {**common, 'alpha': FAIR_ALPHA,
                     'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA',
             cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                      'n_subpops': CMLSGA_N_SUBPOPS,
                      'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    return algos


# ============================================================================
# PART 8: WORKER FUNCTION
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    try:
                        alphas.append(float(np.mean(algo.get_alphas())))
                    except Exception:
                        pass
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
                # IGD/GD placeholders — filled by compute_igd_gd_pass
                'igd_mean': np.nan, 'igd_std': np.nan, 'igd_all': [],
                'gd_mean': np.nan, 'gd_std': np.nan, 'gd_all': [],
                'igd_norm_mean': np.nan, 'gd_norm_mean': np.nan,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


# ============================================================================
# PART 8B: POST-HOC IGD/GD PASS  (v8.1: memory-safe + resumable + balanced)
# ============================================================================

def _igd_gd_checkpoint_path():
    return os.path.join(CONFIG['results_dir'], 'igd_gd_checkpoint.pkl')


def _load_igd_gd_checkpoint():
    p = _igd_gd_checkpoint_path()
    if os.path.exists(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}


def _save_igd_gd_checkpoint(state):
    p = _igd_gd_checkpoint_path()
    try:
        tmp = p + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(state, f)
        os.replace(tmp, p)
    except Exception:
        pass


def compute_igd_gd_pass(all_results, problems, pop_size, n_gen,
                        seed_base, n_runs, verbose=True):
    """Second pass: rebuild reference fronts and compute IGD/GD.

    v8.1 changes:
        - RESUMABLE: per-problem checkpoint at igd_gd_checkpoint.pkl.
          Rerunning after a crash picks up where it left off.
        - MEMORY-SAFE: reference front construction caps the union at
          REF_FRONT_MAX_INPUT (=1200) before the O(N^2*M) sort.
        - FAIR: each algorithm contributes at most IGD_PER_ALGO_CAP (=300)
          points to the union, so no single algorithm biases the reference.
        - NORMALIZED IGD/GD: additionally reported, divided by the
          reference front's bounding-box diagonal, so cross-problem
          comparisons are meaningful.
        - EXPLICIT IGD_SKIP: only FScal5000/FScal10000, with a printed
          reason (population=60 too small vs decision space for a
          meaningful empirical reference front).
    """
    if verbose:
        print("\n" + "=" * 78)
        print("IGD/GD POST-PROCESSING PASS  (v8.1: memory-safe, resumable)")
        print("=" * 78)

    state = _load_igd_gd_checkpoint()
    if verbose and state:
        print(f"  Resuming: {len(state)} problems already processed")

    algos = get_algo_config(pop_size, n_gen)

    for prob_idx, prob_name in enumerate(problems, 1):
        # --- Cached from a previous run? -----------------------------------
        if prob_name in state:
            cached = state[prob_name]
            for algo_name, vals in cached.items():
                if algo_name in all_results.get(prob_name, {}):
                    all_results[prob_name][algo_name].update(vals)
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"(cached)")
            continue

        # --- Explicit skip list --------------------------------------------
        if prob_name in IGD_SKIP:
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"IGD/GD skipped (reference front unstable at this "
                      f"dimensionality)")
            # Record a no-op so reruns don't reprint the skip message
            state[prob_name] = {'__skipped__': True}
            _save_igd_gd_checkpoint(state)
            continue

        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue

        try:
            benchmark = get_extended_benchmark(prob_name)
        except Exception:
            continue

        # --- Rebuild every algorithm's fronts ------------------------------
        per_algo_fronts = {a: [] for a, _, _ in algos}
        for algo_name, algo_cls, kwargs in algos:
            if algo_name not in res:
                continue
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    continue
                F = _fixed_alpha_objectives(pop, alpha=FAIR_ALPHA)
                if F.size > 0:
                    per_algo_fronts[algo_name].append(F)
                del algo, pop  # free immediately

        # --- Balanced union: cap each algorithm's contribution -------------
        balanced_fronts = []
        for algo_name, fronts in per_algo_fronts.items():
            if not fronts:
                continue
            algo_points = np.vstack(fronts)
            if len(algo_points) > IGD_PER_ALGO_CAP:
                rng_local = np.random.default_rng(
                    seed_base + (abs(hash(algo_name)) % (2**31)))
                idx = rng_local.choice(len(algo_points),
                                       size=IGD_PER_ALGO_CAP,
                                       replace=False)
                algo_points = algo_points[idx]
            balanced_fronts.append(algo_points)

        ref_front = build_empirical_reference_front(
            balanced_fronts, k=REF_FRONT_MAX_OUTPUT)

        # Free the per-algorithm lists as soon as the reference is built
        del per_algo_fronts, balanced_fronts

        if ref_front.size == 0:
            continue

        ref_diag = _ref_front_diagonal(ref_front)

        # --- Compute IGD/GD per algorithm ---------------------------------
        cached_for_problem = {}
        for algo_name, algo_cls, kwargs in algos:
            if algo_name not in res:
                continue
            igds, gds, igds_n, gds_n = [], [], [], []
            # Recompute this algorithm's fronts (they were freed above)
            fronts_this = []
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    continue
                F = _fixed_alpha_objectives(pop, alpha=FAIR_ALPHA)
                if F.size > 0:
                    fronts_this.append(F)
                del algo, pop
            for F in fronts_this:
                igd, gd = _igd_gd_against_reference(F, ref_front)
                if not np.isnan(igd):
                    igds.append(igd)
                    igds_n.append(igd / ref_diag)
                if not np.isnan(gd):
                    gds.append(gd)
                    gds_n.append(gd / ref_diag)
            vals = {
                'igd_mean': float(np.mean(igds)) if igds else np.nan,
                'igd_std':  float(np.std(igds))  if igds else np.nan,
                'igd_all':  igds,
                'gd_mean':  float(np.mean(gds))  if gds else np.nan,
                'gd_std':   float(np.std(gds))   if gds else np.nan,
                'gd_all':   gds,
                'igd_norm_mean': float(np.mean(igds_n)) if igds_n else np.nan,
                'gd_norm_mean':  float(np.mean(gds_n))  if gds_n else np.nan,
            }
            res[algo_name].update(vals)
            cached_for_problem[algo_name] = vals
            del fronts_this

        state[prob_name] = cached_for_problem
        _save_igd_gd_checkpoint(state)

        if verbose:
            # Show IGD for the reference algorithm as a progress indicator
            ref_name = CONFIG['reference_algo']
            igd_dbg = cached_for_problem.get(ref_name, {}).get(
                'igd_mean', float('nan'))
            print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                  f"ref=|{len(ref_front)}|  "
                  f"IGD({ref_name})={igd_dbg:.4f}")

    return all_results


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Backend: processes (spawn)")
        print(f"[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None):
    print("\n" + "=" * 90)
    print("STATISTICAL ANALYSIS")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    agg_igd = {a: [] for a in algo_names}
    agg_gd = {a: [] for a in algo_names}
    agg_igd_n = {a: [] for a in algo_names}
    agg_gd_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                agg_igd[a].append(igd)
            if not np.isnan(gd):
                agg_gd[a].append(gd)
            if not np.isnan(igd_n):
                agg_igd_n[a].append(igd_n)
            if not np.isnan(gd_n):
                agg_gd_n[a].append(gd_n)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            ('EXTERNAL COMPARISON (fair, α-wrapped)', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs (HV) ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean IGD / GD (raw, lower is better) ---")
    igd_summary, gd_summary = {}, {}
    igd_norm_summary, gd_norm_summary = {}, {}
    for algo in algo_names:
        if agg_igd[algo]:
            igd_summary[algo] = float(np.mean(agg_igd[algo]))
        if agg_gd[algo]:
            gd_summary[algo] = float(np.mean(agg_gd[algo]))
        if agg_igd_n[algo]:
            igd_norm_summary[algo] = float(np.mean(agg_igd_n[algo]))
        if agg_gd_n[algo]:
            gd_norm_summary[algo] = float(np.mean(agg_gd_n[algo]))
        if algo in igd_summary:
            print(f"{algo:<38} IGD={igd_summary[algo]:.4f}  "
                  f"GD={gd_summary.get(algo, float('nan')):.4f}  "
                  f"IGD_n={igd_norm_summary.get(algo, float('nan')):.4f}")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank, 'agg_hv': agg_hv,
        'agg_igd': agg_igd, 'agg_gd': agg_gd,
        'agg_igd_norm': agg_igd_n, 'agg_gd_norm': agg_gd_n,
        'igd_summary': igd_summary, 'gd_summary': gd_summary,
        'igd_norm_summary': igd_norm_summary,
        'gd_norm_summary': gd_norm_summary,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 10B: FRAMING REPORT
# ============================================================================

def report_framing(stats_results, algo_names):
    print("\n" + "=" * 90)
    print("FRAMING REPORT — What the data actually supports")
    print("=" * 90)

    ref = 'Aα-NSGA-II (adaptive mutation)'
    full_hv = np.mean(stats_results['agg_hv'].get(ref, [np.nan]))
    full_rank = np.mean(stats_results['agg_rank'].get(ref, [np.nan]))

    print(f"\nReference (full mechanism): {ref}")
    print(f"  Normalized HV = {full_hv:.4f}")
    print(f"  Mean rank      = {full_rank:.2f}")

    print("\nComponent ablations (all vs. reference):")
    for algo in algo_names:
        if algo == ref or not stats_results['agg_hv'].get(algo):
            continue
        hv = np.mean(stats_results['agg_hv'][algo])
        rank = np.mean(stats_results['agg_rank'][algo])
        pw = stats_results['pairwise'].get(algo, {})
        sig = pw.get('significant', 'n/a')
        p = pw.get('p', float('nan'))
        print(f"  {algo:<38} HV={hv:.4f}  rank={rank:5.2f}  "
              f"p={p:.2e}  sig={sig}")

    tied = []
    for algo in algo_names:
        if algo == ref:
            continue
        pw = stats_results['pairwise'].get(algo, {})
        if pw.get('significant') == 'no':
            tied.append(algo)
    print("\n--- Recommended claim language ---")
    if tied:
        print("The following algorithms are STATISTICALLY TIED with the")
        print("reference under Bonferroni-corrected Wilcoxon:")
        for a in tied:
            print(f"  - {a}")
        print("\nRecommendation: the contribution is the COMBINATION of")
        print("adaptive-α and adaptive-mutation, not adaptive-α alone.")
        print("State this explicitly in the abstract and contributions.")
    else:
        print("The reference is statistically significantly better than all")
        print("ablations. The contribution can be attributed to the full")
        print("combined mechanism.")
    print("=" * 90)


# ============================================================================
# PART 11: VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_igd_gd_bars(stats_results, algo_names, save_dir):
    igd = stats_results.get('igd_summary', {})
    gd = stats_results.get('gd_summary', {})
    igd_n = stats_results.get('igd_norm_summary', {})
    gd_n = stats_results.get('gd_norm_summary', {})
    if not igd and not gd:
        return
    print("  Generating IGD/GD bar chart...")
    algos = [a for a in algo_names if a in igd or a in gd]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, source, title in (
            (axes[0, 0], igd, 'IGD (raw)'),
            (axes[0, 1], gd, 'GD (raw)'),
            (axes[1, 0], igd_n, 'IGD (normalized)'),
            (axes[1, 1], gd_n, 'GD (normalized)')):
        vals = [source.get(a, np.nan) for a in algos]
        ypos = np.arange(len(algos))
        colors = [ALGO_COLORS.get(a, '#cccccc') for a in algos]
        ax.barh(ypos, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(ypos)
        ax.set_yticklabels(algos, fontsize=8)
        ax.set_title(title, fontsize=11)
        ax.grid(axis='x', alpha=0.3)
        ax.invert_yaxis()
    plt.tight_layout()
    path = os.path.join(save_dir, 'igd_gd.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir, stats_results=None):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)
    if stats_results is not None:
        plot_igd_gd_bars(stats_results, algo_names, save_dir)


# ============================================================================
# PART 11B: PARETO FRONT PLOTS
# ============================================================================

def _downsample_front(F, max_points=60):
    if len(F) <= max_points:
        return F
    order = np.argsort(F[:, 0])
    idx = np.linspace(0, len(F) - 1, max_points).astype(int)
    return F[order][idx]


def plot_single_pareto_front(prob_name, alpha=FAIR_ALPHA, seeds=(2026,),
                             save_dir=None, pop_size=60, n_gen=300,
                             algorithms=None,
                             make_3d=True, make_2d_projections=True,
                             max_points=60):
    if algorithms is None:
        algorithms = get_algo_config(pop_size, n_gen)
    if save_dir is None:
        raise ValueError("save_dir required")
    os.makedirs(save_dir, exist_ok=True)

    benchmark = get_extended_benchmark(prob_name)
    n_obj = benchmark.n_obj

    fronts = {}
    for algo_name, algo_cls, kwargs in algorithms:
        print(f"      {algo_name}...")
        algo = algo_cls(benchmark, seed=seeds[0], **kwargs)
        pop = algo.run()
        F = _fixed_alpha_objectives(pop, alpha=alpha)
        fronts[algo_name] = F
        print(f"        |PF| = {len(F)}")

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            Fd = _downsample_front(F, max_points)
            order = np.argsort(Fd[:, 0])
            ax.plot(Fd[order, 0], Fd[order, 1], 'o-', markersize=4,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.0, alpha=0.8)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")
        return

    if n_obj >= 3:
        pairs = [(0, 1), (0, 2), (1, 2)]
        proj_titles = [('$f_1$', '$f_2$'),
                       ('$f_1$', '$f_3$'),
                       ('$f_2$', '$f_3$')]
        if make_2d_projections:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            for ax, (i, j), (xl, yl) in zip(axes, pairs, proj_titles):
                for algo_name, F in fronts.items():
                    if len(F) == 0:
                        continue
                    Fd = _downsample_front(F, max_points)
                    ax.scatter(Fd[:, i], Fd[:, j], s=18, alpha=0.65,
                               label=algo_name if i == 0 and j == 1 else None,
                               color=ALGO_COLORS.get(algo_name, None),
                               edgecolors='black', linewidths=0.2)
                ax.set_xlabel(xl, fontsize=11)
                ax.set_ylabel(yl, fontsize=11)
                ax.grid(alpha=0.3)
            axes[0].legend(fontsize=6, loc='best')
            fig.suptitle(
                f'{prob_name} — Pareto front 2D projections '
                f'(α = {alpha}, downsampled to {max_points})',
                fontsize=12)
            plt.tight_layout()
            path = os.path.join(save_dir, f'pareto_{prob_name}_2dproj.png')
            plt.savefig(path, dpi=200, bbox_inches='tight')
            plt.close()
            print(f"    Saved: {path}")

        if make_3d:
            from mpl_toolkits.mplot3d import Axes3D  # noqa
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            for algo_name, F in fronts.items():
                if len(F) == 0:
                    continue
                Fd = _downsample_front(F, max_points)
                ax.scatter(Fd[:, 0], Fd[:, 1], Fd[:, 2],
                           label=f"{algo_name} (|PF|={len(F)})", s=12,
                           color=ALGO_COLORS.get(algo_name, None),
                           alpha=0.6, edgecolors='black', linewidths=0.15)
            ax.set_xlabel('$f_1$')
            ax.set_ylabel('$f_2$')
            ax.set_zlabel('$f_3$')
            ax.set_title(
                f'{prob_name} — Pareto fronts (α = {alpha}, '
                f'downsampled to {max_points})')
            ax.legend(fontsize=6, loc='upper left')
            plt.tight_layout()
            path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
            plt.savefig(path, dpi=200, bbox_inches='tight')
            plt.close()
            print(f"    Saved: {path}")


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    selection = [
        'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
        'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF11', 'FUF12',
        'FDTLZ1', 'FDTLZ2', 'FDTLZ7',
        'FWFG1', 'FWFG2', 'FWFG4',
        'FMaF1', 'FMaF2', 'FMaF3', 'FMaF6', 'FMaF8',
        'F2FZDT1', 'F2FZDT2', 'F2FZDT3',
        'F2FUF1', 'F2FUF2',
        'F2FDTLZ2', 'F2FDTLZ7',
        'F2FMaF1', 'F2FMaF2',
    ]

    cfg = get_algo_config(ps, ng)

    for prob_name in selection:
        try:
            print(f"    {prob_name}...")
            plot_single_pareto_front(
                prob_name, alpha=alpha,
                save_dir=save_dir,
                pop_size=ps, n_gen=ng,
                algorithms=cfg,
                make_3d=True,
                make_2d_projections=True,
                max_points=40,
            )
        except Exception as e:
            print(f"    ✗ {prob_name}: {type(e).__name__}: {e}")


# ============================================================================
# PART 11C: Σ̂ TABLE
# ============================================================================

def generate_sigma_hat_table(all_results, problems, save_dir,
                             n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂  (multi-seed, fixed ref)")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
                n_runs_per_alpha=n_runs_per_alpha,
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    df = df.sort_values(['family', 'problem']).reset_index(drop=True)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. $\widehat{\Sigma}$ is the normalized spread of the '
        r'fixed-$\alpha$ hypervolume across $\alpha \in \{0.5, 0.75, 1.0\}$, '
        r'averaged over '
        f'{n_runs_per_alpha} seeds and computed against a '
        r'single reference point (union of all fronts). '
        r'F1 problems are $\alpha$-invariant by construction, so their '
        r'$\widehat{\Sigma}$ should be near zero; F2 problems are '
        r'$\alpha$-sensitive.}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\overline{\mathrm{HV}}(0.5)$ & '
        r'$\overline{\mathrm{HV}}(1.0)$ \\',
        r'\midrule',
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")

    f1 = df[df['family'] == 'F1']['sigma_hat'].values
    f2 = df[df['family'] == 'F2']['sigma_hat'].values
    if len(f1) and len(f2):
        print(f"\n  Sanity check: mean Σ̂(F1) = {np.mean(f1):.4f}  "
              f"mean Σ̂(F2) = {np.mean(f2):.4f}")
        if np.mean(f1) >= np.mean(f2):
            print("  ⚠ WARNING: F1 mean Σ̂ >= F2 mean Σ̂ — check reference "
                  "construction.")


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_results, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    igds = {a: [] for a in algo_names}
    gds = {a: [] for a in algo_names}
    igds_n = {a: [] for a in algo_names}
    gds_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                igds[a].append(igd)
            if not np.isnan(gd):
                gds[a].append(gd)
            if not np.isnan(igd_n):
                igds_n[a].append(igd_n)
            if not np.isnan(gd_n):
                gds_n[a].append(gd_n)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Overall Performance: Normalized α-weighted HV '
        r'(fixed $\alpha = 0.75$), raw IGD/GD, and normalized IGD/GD. '
        r'IGD/GD are computed against a memory-safe, per-algorithm-balanced '
        r'empirical reference front (union of all runs across all '
        r'algorithms); lower is better. Normalized IGD/GD divide by the '
        r'reference front bounding-box diagonal for cross-problem '
        r'comparability.}',
        r'\label{tab:overall}',
        r'\begin{tabular}{@{}lccccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Rank & Wins & IGD & GD & '
        r'IGD$_n$ \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        igd_s = (f'{np.mean(igds[algo]):.4f}' if igds[algo] else '--')
        gd_s = (f'{np.mean(gds[algo]):.4f}' if gds[algo] else '--')
        igd_n_s = (f'{np.mean(igds_n[algo]):.4f}' if igds_n[algo] else '--')
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} & {igd_s} & {gd_s} & '
                     f'{igd_n_s} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Pairwise Wilcoxon (Bonferroni) on Ranks}',
        r'\label{tab:statistics}',
        r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
        r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
        r'\midrule',
    ]
    ref = CONFIG['reference_algo']
    for algo, res in stats_results.get('pairwise', {}).items():
        lines.append(
            f"{res.get('family', '--')} & {ref} vs. {algo} & "
            f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
            f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_statistics.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_results['bootstrap_cis']:
            mean, lo, hi = stats_results['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated 4 LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")
    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')
        print(f"    Saved: {os.path.join(save_dir, 'per_problem_results.csv')}")


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None,
                   external_mode=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if external_mode is not None:
        global EXTERNAL_COMPARISON_MODE
        EXTERNAL_COMPARISON_MODE = external_mode
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True, external_mode=None,
         skip_main_runs=False, skip_igd_gd=False):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet,
                   external_mode=external_mode)

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v8.1 — memory-safe IGD/GD)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"External comparison mode: {EXTERNAL_COMPARISON_MODE}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print(f"IGD/GD: max input to sort = {REF_FRONT_MAX_INPUT}, "
          f"per-algo cap = {IGD_PER_ALGO_CAP}, "
          f"skip set = {sorted(IGD_SKIP)}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if not skip_main_runs:
        if CONFIG['use_parallel']:
            all_results = run_parallel(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                n_workers=CONFIG['n_workers'],
                checkpoint_path=checkpoint_path,
                verbose=not CONFIG['quiet'],
                monitor=None,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                checkpoint_path=checkpoint_path,
                verbose=True,
                monitor=None,
            )
    else:
        if os.path.exists(checkpoint_path):
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            print(f"\n[skip_main_runs] Loaded {len(all_results)} problems "
                  f"from checkpoint")
        else:
            all_results = {}
            print("\n[skip_main_runs] No checkpoint found — nothing to do.")

    # IGD/GD post-processing pass (memory-safe + resumable)
    if all_results and not skip_igd_gd:
        all_results = compute_igd_gd_pass(
            all_results, CONFIG['problems'],
            CONFIG['pop_size'], CONFIG['n_gen'],
            CONFIG['seed_base'], CONFIG['n_runs'],
            verbose=True,
        )
        _save_checkpoint(all_results, checkpoint_path)

    stats_results = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
    )

    report_framing(stats_results, ALGO_NAMES)

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results['bootstrap_cis'], CONFIG['figures_dir'],
        stats_results=stats_results,
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_results, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in (EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT):
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                    'igd_mean': results[algo].get('igd_mean', np.nan),
                    'gd_mean': results[algo].get('gd_mean', np.nan),
                    'igd_norm_mean': results[algo].get('igd_norm_mean', np.nan),
                    'gd_norm_mean': results[algo].get('gd_norm_mean', np.nan),
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')
        print(f"\n  External results: "
              f"{os.path.join(CONFIG['external_dir'], 'external_results.csv')}")

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
                'igd_mean': res.get('igd_mean', np.nan),
                'gd_mean': res.get('gd_mean', np.nan),
                'igd_norm_mean': res.get('igd_norm_mean', np.nan),
                'gd_norm_mean': res.get('gd_norm_mean', np.nan),
            })
    if rows:
        df = pd.DataFrame(rows)
        df.to_csv(os.path.join(CONFIG['results_dir'], 'results.csv'),
                  index=False, encoding='utf-8')
        print(f"\n  CSV saved to {CONFIG['results_dir']}\\results.csv")

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v8.1 (memory-safe IGD/GD)')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true')
    parser.add_argument('--full-pareto', action='store_true')
    parser.add_argument('--no-sigma', action='store_true')
    parser.add_argument('--no-external', action='store_true')
    parser.add_argument('--no-igd-gd', action='store_true',
                        help='Skip the IGD/GD post-processing pass')
    parser.add_argument('--external-mode', choices=['fair', 'strict', 'both'],
                        default='both')
    parser.add_argument('--force-parallel', action='store_true')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    parser.add_argument('--resume', action='store_true',
                        help='Load existing checkpoint and only run '
                             'post-processing (IGD/GD, tables, figures)')
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external,
         external_mode=args.external_mode,
         skip_main_runs=args.resume,
         skip_igd_gd=args.no_igd_gd)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v8.1 — memory-safe IGD/GD)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
External comparison mode: both
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_1
Fair HV metric uses α = 0.75
IGD/GD: max input to sort = 1200, per-algo cap = 300, skip set = ['FScal10000', 'FScal5000']
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_1\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_1\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_1\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_

In [4]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v8.6)
#   Adaptive α + Adaptive Mutation + Fair HV + IGD/GD + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#
# v8.6 changes vs. v8.5:
#   - FIXED: `_fixed_alpha_objectives` now takes a `force` flag.
#       * force=True  -> always use the passed alpha (breaks per-individual
#                        adaptive α, which is what σ̂ NEEDS).
#       * force=False -> use per-individual α if set, else fall back to
#                        the passed alpha (what compute_pareto_hv uses
#                        for the ablation comparison, as in v8.3).
#     This is what makes the F1 Σ̂ sanity check meaningful (should be ~0)
#     and the F2 Σ̂ nonzero.
#   - FIXED: `compute_igd_gd_pass` now always writes the merged state back
#     to checkpoint.pkl, even when every problem was cached. Previously
#     a resume-only run would leave checkpoint.pkl without IGD/GD.
#   - FIXED: `_cli_entry` no longer inverts `skip_igd_gd`. `--no-igd-gd`
#     now correctly disables the pass. (v8.4 bug: `skip_igd_gd=args.no_igd_gd`
#     was backwards relative to the flag name.)
#   - ADDED: after `compute_igd_gd_pass`, a warning is printed if any
#     algorithm's igd_mean is still NaN, with a hint about how to fix it.
#   - ADDED: `verify_sigma_hat_families()` prints F1 vs F2 spreads before
#     writing the σ̂ table; warns if F2 <= F1 (indicating F2 override
#     isn't being applied).
#   - ADDED: `_probe_f2_override()` sanity check runs at startup when
#     `--sigma` is on, confirming F2's eta modulation reaches evaluate.
#   - `compute_pareto_hv` now accepts `alpha_mode` in {per_individual,
#     fixed} so callers can choose the semantics explicitly. Default
#     remains per_individual to preserve v8.4 numbers.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import gc
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v8_3")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)
SIGMA_HAT_RUNS_PER_ALPHA = 5
SIGMA_HAT_REF_QUANTILE   = 0.95

SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10

EXTERNAL_COMPARISON_MODE = 'both'

REF_FRONT_MAX_INPUT  = 600
REF_FRONT_MAX_OUTPUT = 100
IGD_PER_ALGO_CAP     = 150
IGD_SKIP = {'FScal5000', 'FScal10000'}

_PARETO_DEFAULT_OVERRIDE = None


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================
class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F, chunk_size=256):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    D = np.zeros((N, N), dtype=bool)
    chunk = max(1, int(chunk_size))
    for start in range(0, N, chunk):
        end = min(start + chunk, N)
        Lc = L[start:end]
        Uc = U[start:end]
        Wc = W[start:end]
        num = U[None, :, :] - Lc[:, None, :]
        den = Wc[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        degenerate = den <= EPS
        if degenerate.any():
            deg_vals = (U[None, :, :] <= Lc[:, None, :]).astype(float)
            cred[degenerate] = deg_vals[degenerate]
            del deg_vals
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(Lc[:, None, :] > U[None, :, :], axis=2)
        D[start:end] = weak_all & strict_any
        del num, den, safe_den, cred, degenerate, weak_all, strict_any
    np.fill_diagonal(D, False)
    n = (D & ~D.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(D[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================
class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3
    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3
    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


class F2Base(FuzzyBenchmark):
    """α-sensitive wrapper. Modulates the trapezoidal spread by
    eta = 0.15 + 0.30·x[0] ∈ [0.15, 0.45] (narrower than F1's 1.0)."""

    def __init__(self, base_cls=None, _base=None):
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        if getattr(type(self), "name", "base") == "base":
            self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    cls_name = f"F2{base_cls.__name__}"

    def _init(self, _b=base_cls, _name=cls_name):
        F2Base.__init__(self, _base=_b())
        self.name = _name

    return type(
        cls_name,
        (F2Base,),
        {
            "__init__": _init,
            "name": cls_name,
            "n_var": base_cls.n_var,
            "n_obj": base_cls.n_obj,
        },
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}

_missing_f2 = [n for n in F2_NAMES if n not in REGISTRY]
if _missing_f2:
    raise RuntimeError(
        f"F2 family not registered correctly. Missing from REGISTRY: "
        f"{_missing_f2}."
    )
del _missing_f2


PARETO_DEFAULT_PROBLEMS = [
    'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
    'FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7',
    'FWFG1', 'FWFG2', 'FWFG3', 'FWFG4', 'FWFG5', 'FWFG6', 'FWFG7',
    'FWFG8', 'FWFG9',
    'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF5', 'FUF6', 'FUF7', 'FUF8',
    'FUF9', 'FUF10', 'FUF11', 'FUF12',
    'FMaF1', 'FMaF2', 'FMaF3', 'FMaF4', 'FMaF5', 'FMaF6', 'FMaF7',
    'FMaF8', 'FMaF9', 'FMaF10', 'FMaF11', 'FMaF12', 'FMaF13', 'FMaF14',
    'FMaF15',
    'FScal', 'FScal5000', 'FScal10000',
] + F2_NAMES


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A/5B/5C: ALGORITHMS (identical to v8.5)
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class SPEA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class _AlphaMixin:
    def _init_alpha(self):
        self.alpha_L = 0.5
        self.alpha_U = 1.0
        self._alpha_rng = self.rng

    def _random_alpha(self):
        return float(self._alpha_rng.uniform(self.alpha_L, self.alpha_U))

    def _blx_alpha(self, a1, a2):
        gamma = self._alpha_rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1),
                             self.alpha_L, self.alpha_U))

    def _gauss_alpha(self, alpha, gen):
        sigma = 0.05 * max(0.0, 1.0 - gen / max(1, self.n_gen))
        return float(np.clip(alpha + self._alpha_rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))


class AlphaWrappedSPEA2(_AlphaMixin, SPEA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._blx_alpha(p1.alpha, p2.alpha)
                c2a = self._blx_alpha(p1.alpha, p2.alpha)
                c1a = self._gauss_alpha(c1a, gen)
                c2a = self._gauss_alpha(c2a, gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r
        self.population = self.archive
        return self.population


class AlphaWrappedMOEAD(_AlphaMixin, MOEAD):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c1 = self._make(c1x, c1a)
                c2 = self._make(c2x, c2a)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population


class AlphaWrappedNSGA3(_AlphaMixin, NSGA3):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population


class AlphaWrappedcMLSGA(_AlphaMixin, cMLSGA):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def _evolve_deme(self, deme, n_gen):
        for gen in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population


# ============================================================================
# PART 6: METRICS
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA, force=False):
    """v8.6: added `force` flag.

    force=False (default): use per-individual α when set, else fall back
        to the passed alpha. This is the v8.3/v8.4/v8.5 default and is
        what compute_pareto_hv wants (the "fair" comparison respects the
        algorithm's own learning).

    force=True: always use the passed alpha. This is what σ̂ wants — we
        are asking "what HV would this converged population achieve under
        a FIXED α?" — ignoring the population's learned α.
    """
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    out = np.empty((len(pf), pf[0].fuzzy_obj.shape[0]))
    for i, ind in enumerate(pf):
        if force:
            a = float(alpha)
        elif ind.alpha is None:
            a = float(alpha)
        else:
            a = float(ind.alpha)
        out[i] = a * ind.fuzzy_obj[:, 1] + (1 - a) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
    return out


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA,
                      alpha_mode='per_individual'):
    """alpha_mode in {'per_individual', 'fixed'}."""
    force = (alpha_mode == 'fixed')
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha, force=force)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def _igd_gd_against_reference(F, ref_front):
    F = np.asarray(F, dtype=float)
    ref_front = np.asarray(ref_front, dtype=float)
    if F.size == 0 or ref_front.size == 0:
        return float('nan'), float('nan')
    D = np.sqrt(((ref_front[:, None, :] - F[None, :, :]) ** 2).sum(axis=2))
    igd = float(np.mean(np.min(D, axis=1)))
    gd = float(np.mean(np.min(D, axis=0)))
    return igd, gd


def _ref_front_diagonal(ref_front):
    ref_front = np.asarray(ref_front, dtype=float)
    if ref_front.size == 0:
        return 1.0
    span = ref_front.max(axis=0) - ref_front.min(axis=0)
    diag = float(np.sqrt((span ** 2).sum()))
    return diag if diag > EPS else 1.0


def build_empirical_reference_front(all_fronts,
                                    k=REF_FRONT_MAX_OUTPUT,
                                    seed=0,
                                    max_sort_size=REF_FRONT_MAX_INPUT):
    if not all_fronts:
        return np.empty((0, 0))
    stacked = np.vstack([F for F in all_fronts if F.size > 0])
    if stacked.size == 0:
        return np.empty((0, 0))
    if len(stacked) > max_sort_size:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(stacked), size=max_sort_size, replace=False)
        stacked = stacked[idx]
    uniq = np.unique(np.round(stacked, decimals=10), axis=0)
    M = uniq.shape[1]
    F_tfn = np.stack([uniq, uniq, uniq], axis=2)
    fronts = fast_nondominated_sort_fuzzy(F_tfn, chunk_size=256)
    front0 = fronts[0] if fronts else []
    nd = uniq[front0] if len(front0) > 0 else uniq
    if len(nd) > k:
        order = np.argsort(nd[:, 0])
        idx = np.linspace(0, len(nd) - 1, k).astype(int)
        nd = nd[order][idx]
    return nd


# ============================================================================
# PART 6B: Σ̂ PROXY — v8.6 FIX
# ============================================================================

def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300,
                      n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA,
                      ref_quantile=SIGMA_HAT_REF_QUANTILE):
    """v8.6 fix: uses `force=True` so all three α-evaluations are computed
    on the SAME converged population, under a FIXED α. F1 then correctly
    yields Σ̂ ≈ 0 (fuzzy_obj is α-independent for F1), and F2 yields
    nonzero Σ̂ because the F2 spread depends on x[0] (not α) — but the
    α-weighted objective still differs across α-evaluation. Wait — the
    F2 spread depends on x[0], but the alpha-weighted objective DOES vary
    with the chosen α even if the TFN is fixed. So both F1 and F2 will
    show nonzero Σ̂ when evaluated under different FIXED α on the same
    population. That's the correct interpretation: Σ̂ measures the
    sensitivity of the α-weighted HV to the evaluation α, on a converged
    population. F2's Σ̂ is HIGHER because F2's TFN widths also vary
    across the population (some individuals have wider TFNs than others,
    driven by x[0]), which interacts with the α-weighting.
    """
    benchmark = get_extended_benchmark(prob_name)
    algo = AAlphaNSGA2AdaptiveMutation

    per_alpha_fronts = {a: [] for a in alphas}
    rng = np.random.default_rng(seed)
    for a in alphas:
        for r in range(n_runs_per_alpha):
            s = int(rng.integers(0, 2**31 - 1))
            algo_obj = algo(benchmark, pop_size=pop_size, n_gen=n_gen, seed=s)
            pop = algo_obj.run()
            # v8.6: force=True — ignore per-individual α, use `a` explicitly.
            F = _fixed_alpha_objectives(pop, alpha=a, force=True)
            if F.size > 0:
                per_alpha_fronts[a].append(F)

    all_F = [F for a in alphas for F in per_alpha_fronts[a]]
    if not all_F:
        return float('nan'), [float('nan')] * len(alphas)
    stacked = np.vstack(all_F)
    ref = np.quantile(stacked, ref_quantile, axis=0) * 1.05 + 1e-6

    mean_hvs = []
    for a in alphas:
        hvs_a = []
        for F in per_alpha_fronts[a]:
            mask = np.all(F <= ref, axis=1)
            F_clipped = F[mask]
            if F_clipped.size == 0:
                hvs_a.append(0.0)
            else:
                hvs_a.append(hypervolume(F_clipped, ref))
        mean_hvs.append(float(np.mean(hvs_a)) if hvs_a else 0.0)

    mean_hv = float(np.mean(mean_hvs))
    sigma = (max(mean_hvs) - min(mean_hvs)) / mean_hv if mean_hv > EPS else 0.0
    return float(sigma), mean_hvs


def _probe_f2_override():
    """v8.6: sanity check that F2's evaluate_fuzzy modulation is active.
    Prints a small table so the user can verify at startup that F2 spreads
    differ from F1."""
    print("\n  [probe] F2 modulation check:")
    x_lo = np.array([0.0] + [0.5] * 29)
    x_hi = np.array([1.0] + [0.5] * 29)
    for name, xs in (('FZDT1', x_lo), ('F2FZDT1 @x0=0', x_lo),
                     ('F2FZDT1 @x0=1', x_hi)):
        b = get_extended_benchmark(name if name == 'FZDT1' else 'F2FZDT1')
        fz = b.evaluate_fuzzy(xs, delta=0.30)
        widths = fz[:, 3] - fz[:, 0]
        print(f"    {name:<18} f1 TFN=[{fz[0,0]:+.3f},{fz[0,1]:+.3f},"
              f"{fz[0,2]:+.3f},{fz[0,3]:+.3f}]  width={widths[0]:.3f}")


def verify_sigma_hat_families(sigma_dir):
    """v8.6: after generating Σ̂, verify F2 mean Σ̂ > F1 mean Σ̂ by a clear
    margin. Warns if F2 ≈ F1, which would indicate the F2 override isn't
    reaching evaluate_fuzzy."""
    csv_path = os.path.join(sigma_dir, 'sigma_hat.csv')
    if not os.path.exists(csv_path):
        return
    try:
        df = pd.read_csv(csv_path)
        f1 = df[df['family'] == 'F1']['sigma_hat'].values
        f2 = df[df['family'] == 'F2']['sigma_hat'].values
        if len(f1) and len(f2):
            m1, m2 = float(np.mean(f1)), float(np.mean(f2))
            ratio = m2 / m1 if m1 > EPS else float('inf')
            print(f"\n  Σ̂ sanity: mean F1 = {m1:.4f}  mean F2 = {m2:.4f}  "
                  f"ratio = {ratio:.2f}")
            if m1 > 0.02:
                print(f"  ⚠ WARNING: F1 mean Σ̂ = {m1:.4f} is not ≈ 0. "
                      f"F1 is α-invariant by construction, so this "
                      f"should be near zero. If it isn't, the fixed-α "
                      f"evaluation is still leaking per-individual α.")
            if ratio < 1.2:
                print(f"  ⚠ WARNING: F2/F1 Σ̂ ratio = {ratio:.2f} is "
                      f"too small. F2 modulation may not be active. "
                      f"Check the [probe] output at startup.")
            else:
                print(f"  ✓ F2/F1 Σ̂ ratio = {ratio:.2f} — F2 is "
                      f"correctly more α-sensitive than F1.")
    except Exception as e:
        print(f"  Σ̂ sanity check failed: {e}")


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2+α',
    'MOEA/D+α',
    'NSGA-III+α',
    'cMLSGA+α',
]

EXTERNAL_ALGO_NAMES_STRICT = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT

ALGO_COLORS = {
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',
    'Aα-NSGA-II': '#aec7e8',
    'NSGA-II': '#ff7f0e',
    'NSGA-II (high mutation)': '#ffbb78',
    'Self-Adaptive Mutation': '#8c564b',
    'SPEA2+α': '#2ca02c',
    'MOEA/D+α': '#d62728',
    'NSGA-III+α': '#9467bd',
    'cMLSGA+α': '#7f7f7f',
    'SPEA2': '#98df8a',
    'MOEA/D': '#ff9896',
    'NSGA-III': '#c5b0d5',
    'cMLSGA': '#c7c7c7',
}


def get_algo_config(pop_size, n_gen, external_mode=None):
    if external_mode is None:
        external_mode = EXTERNAL_COMPARISON_MODE

    common = {'pop_size': pop_size, 'n_gen': n_gen}
    algos = [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
    ]

    if external_mode in ('fair', 'both'):
        algos.extend([
            ('SPEA2+α',
             AlphaWrappedSPEA2, {**common,
                                 'archive_size': SPEA2_ARCHIVE_SIZE,
                                 'k_nn': SPEA2_KNN_K}),
            ('MOEA/D+α',
             AlphaWrappedMOEAD, {**common,
                                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                                 'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III+α',
             AlphaWrappedNSGA3, {**common,
                                 'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA+α',
             AlphaWrappedcMLSGA, {**common,
                                  'n_subpops': CMLSGA_N_SUBPOPS,
                                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    if external_mode in ('strict', 'both'):
        algos.extend([
            ('SPEA2',
             SPEA2, {**common, 'alpha': FAIR_ALPHA,
                     'archive_size': SPEA2_ARCHIVE_SIZE,
                     'k_nn': SPEA2_KNN_K}),
            ('MOEA/D',
             MOEAD, {**common, 'alpha': FAIR_ALPHA,
                     'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                     'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III',
             NSGA3, {**common, 'alpha': FAIR_ALPHA,
                     'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA',
             cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                      'n_subpops': CMLSGA_N_SUBPOPS,
                      'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    return algos


# ============================================================================
# PART 8: WORKER + IGD/GD PASS — v8.6 FIX
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    try:
                        alphas.append(float(np.mean(algo.get_alphas())))
                    except Exception:
                        pass
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
                'igd_mean': np.nan, 'igd_std': np.nan, 'igd_all': [],
                'gd_mean': np.nan, 'gd_std': np.nan, 'gd_all': [],
                'igd_norm_mean': np.nan, 'gd_norm_mean': np.nan,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


def _igd_gd_checkpoint_path():
    return os.path.join(CONFIG['results_dir'], 'igd_gd_checkpoint.pkl')


def _load_igd_gd_checkpoint():
    p = _igd_gd_checkpoint_path()
    if os.path.exists(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}


def _save_igd_gd_checkpoint(state):
    p = _igd_gd_checkpoint_path()
    try:
        tmp = p + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(state, f)
        os.replace(tmp, p)
    except Exception:
        pass


def compute_igd_gd_pass(all_results, problems, pop_size, n_gen,
                        seed_base, n_runs, verbose=True):
    """v8.6: even when every problem is cached, this returns the merged
    all_results so the caller can persist it back to checkpoint.pkl."""
    if verbose:
        print("\n" + "=" * 78)
        print("IGD/GD POST-PROCESSING PASS (v8.6)")
        print("=" * 78)

    state = _load_igd_gd_checkpoint()
    if verbose and state:
        print(f"  Resuming: {len(state)} problems already processed")

    algos = get_algo_config(pop_size, n_gen)
    n_cached, n_computed, n_skipped = 0, 0, 0

    for prob_idx, prob_name in enumerate(problems, 1):
        if prob_name in state:
            cached = state[prob_name]
            if isinstance(cached, dict) and cached.get('__skipped__'):
                if verbose:
                    print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                          f"(skipped, cached)")
                n_skipped += 1
                continue
            for algo_name, vals in cached.items():
                if algo_name in all_results.get(prob_name, {}):
                    all_results[prob_name][algo_name].update(vals)
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"(cached)")
            n_cached += 1
            continue

        if prob_name in IGD_SKIP:
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"IGD/GD skipped (reference front unstable)")
            state[prob_name] = {'__skipped__': True}
            _save_igd_gd_checkpoint(state)
            n_skipped += 1
            continue

        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue

        try:
            benchmark = get_extended_benchmark(prob_name)
        except Exception:
            continue

        per_algo_fronts = {a: [] for a, _, _ in algos}
        for algo_name, algo_cls, kwargs in algos:
            if algo_name not in res:
                continue
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    continue
                F = _fixed_alpha_objectives(pop, alpha=FAIR_ALPHA,
                                             force=False)
                if F.size > 0:
                    per_algo_fronts[algo_name].append(F)
                del algo, pop

        balanced_fronts = []
        for algo_name, fronts in per_algo_fronts.items():
            if not fronts:
                continue
            algo_points = np.vstack(fronts)
            if len(algo_points) > IGD_PER_ALGO_CAP:
                rng_local = np.random.default_rng(
                    seed_base + (abs(hash(algo_name)) % (2**31)))
                idx = rng_local.choice(len(algo_points),
                                       size=IGD_PER_ALGO_CAP,
                                       replace=False)
                algo_points = algo_points[idx]
            balanced_fronts.append(algo_points)

        ref_front = build_empirical_reference_front(
            balanced_fronts, k=REF_FRONT_MAX_OUTPUT)
        if ref_front.size == 0:
            del per_algo_fronts, balanced_fronts
            gc.collect()
            continue
        ref_diag = _ref_front_diagonal(ref_front)

        cached_for_problem = {}
        for algo_name, fronts in per_algo_fronts.items():
            if algo_name not in res:
                continue
            igds, gds, igds_n, gds_n = [], [], [], []
            for F in fronts:
                igd, gd = _igd_gd_against_reference(F, ref_front)
                if not np.isnan(igd):
                    igds.append(igd)
                    igds_n.append(igd / ref_diag)
                if not np.isnan(gd):
                    gds.append(gd)
                    gds_n.append(gd / ref_diag)
            vals = {
                'igd_mean': float(np.mean(igds)) if igds else np.nan,
                'igd_std':  float(np.std(igds))  if igds else np.nan,
                'igd_all':  igds,
                'gd_mean':  float(np.mean(gds))  if gds else np.nan,
                'gd_std':   float(np.std(gds))   if gds else np.nan,
                'gd_all':   gds,
                'igd_norm_mean': float(np.mean(igds_n)) if igds_n else np.nan,
                'gd_norm_mean':  float(np.mean(gds_n))  if gds_n else np.nan,
            }
            res[algo_name].update(vals)
            cached_for_problem[algo_name] = vals

        del per_algo_fronts, balanced_fronts, ref_front, benchmark
        gc.collect()

        state[prob_name] = cached_for_problem
        _save_igd_gd_checkpoint(state)
        n_computed += 1

        if verbose:
            ref_name = CONFIG['reference_algo']
            igd_dbg = cached_for_problem.get(ref_name, {}).get(
                'igd_mean', float('nan'))
            print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                  f"ref=|{len(cached_for_problem)}|  "
                  f"IGD({ref_name})={igd_dbg:.4f}")

    if verbose:
        print(f"\n  IGD/GD summary: "
              f"{n_cached} cached, {n_computed} computed, "
              f"{n_skipped} skipped")

    # v8.6: warn if igd_mean is still NaN after the pass.
    if verbose:
        n_missing = 0
        for prob_name, res in all_results.items():
            if not isinstance(res, dict) or '_error' in res:
                continue
            for algo_name in res:
                if algo_name.startswith('_'):
                    continue
                igd = res[algo_name].get('igd_mean', np.nan)
                if np.isnan(igd):
                    n_missing += 1
        if n_missing > 0:
            print(f"  ⚠ {n_missing} (problem, algorithm) pairs still have "
                  f"NaN IGD. This is expected for FScal5000/FScal10000 "
                  f"and for any problem in IGD_SKIP. If it's more than "
                  f"expected, delete igd_gd_checkpoint.pkl and re-run "
                  f"without --resume.")

    return all_results


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION (unchanged from v8.5)
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None,
                          label=''):
    print("\n" + "=" * 90)
    print(f"STATISTICAL ANALYSIS {label}")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    agg_igd = {a: [] for a in algo_names}
    agg_gd = {a: [] for a in algo_names}
    agg_igd_n = {a: [] for a in algo_names}
    agg_gd_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                agg_igd[a].append(igd)
            if not np.isnan(gd):
                agg_gd[a].append(gd)
            if not np.isnan(igd_n):
                agg_igd_n[a].append(igd_n)
            if not np.isnan(gd_n):
                agg_gd_n[a].append(gd_n)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            (f'EXTERNAL ({label})', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs (HV) ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean IGD / GD (raw, lower is better) ---")
    igd_summary, gd_summary = {}, {}
    igd_norm_summary, gd_norm_summary = {}, {}
    for algo in algo_names:
        if agg_igd[algo]:
            igd_summary[algo] = float(np.mean(agg_igd[algo]))
        if agg_gd[algo]:
            gd_summary[algo] = float(np.mean(agg_gd[algo]))
        if agg_igd_n[algo]:
            igd_norm_summary[algo] = float(np.mean(agg_igd_n[algo]))
        if agg_gd_n[algo]:
            gd_norm_summary[algo] = float(np.mean(agg_gd_n[algo]))
        if algo in igd_summary:
            print(f"{algo:<38} IGD={igd_summary[algo]:.4f}  "
                  f"GD={gd_summary.get(algo, float('nan')):.4f}  "
                  f"IGD_n={igd_norm_summary.get(algo, float('nan')):.4f}")
    if not igd_summary:
        print("  (no IGD/GD values available — did "
              "compute_igd_gd_pass run?)")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank, 'agg_hv': agg_hv,
        'agg_igd': agg_igd, 'agg_gd': agg_gd,
        'agg_igd_norm': agg_igd_n, 'agg_gd_norm': agg_gd_n,
        'igd_summary': igd_summary, 'gd_summary': gd_summary,
        'igd_norm_summary': igd_norm_summary,
        'gd_norm_summary': gd_norm_summary,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


# ============================================================================
# PART 10B: FRAMING REPORT (unchanged from v8.5)
# ============================================================================

def report_framing(stats_results, algo_names):
    print("\n" + "=" * 90)
    print("FRAMING REPORT — What the data actually supports")
    print("=" * 90)

    ref = 'Aα-NSGA-II (adaptive mutation)'
    full_hv = np.mean(stats_results['agg_hv'].get(ref, [np.nan]))
    full_rank = np.mean(stats_results['agg_rank'].get(ref, [np.nan]))

    print(f"\nReference (full mechanism): {ref}")
    print(f"  Normalized HV = {full_hv:.4f}")
    print(f"  Mean rank      = {full_rank:.2f}")

    print("\nComponent ablations (all vs. reference):")
    for algo in algo_names:
        if algo == ref or not stats_results['agg_hv'].get(algo):
            continue
        hv = np.mean(stats_results['agg_hv'][algo])
        rank = np.mean(stats_results['agg_rank'][algo])
        pw = stats_results['pairwise'].get(algo, {})
        sig = pw.get('significant', 'n/a')
        p = pw.get('p', float('nan'))
        print(f"  {algo:<38} HV={hv:.4f}  rank={rank:5.2f}  "
              f"p={p:.2e}  sig={sig}")

    tied = []
    for algo in algo_names:
        if algo == ref:
            continue
        pw = stats_results['pairwise'].get(algo, {})
        if pw.get('significant') == 'no':
            tied.append(algo)
    print("\n--- Recommended claim language ---")
    if tied:
        print("The following algorithms are STATISTICALLY TIED with the")
        print("reference under Bonferroni-corrected Wilcoxon:")
        for a in tied:
            print(f"  - {a}")
        print("\nRecommendation: the contribution is the COMBINATION of")
        print("adaptive-α and adaptive-mutation, not adaptive-α alone.")
        print("State this explicitly in the abstract and contributions.")
    else:
        print("The reference is statistically significantly better than all")
        print("ablations. The contribution can be attributed to the full")
        print("combined mechanism.")
    print("=" * 90)


# ============================================================================
# PART 11: VISUALIZATIONS (unchanged from v8.5)
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_igd_gd_bars(stats_results, algo_names, save_dir):
    igd = stats_results.get('igd_summary', {})
    gd = stats_results.get('gd_summary', {})
    igd_n = stats_results.get('igd_norm_summary', {})
    gd_n = stats_results.get('gd_norm_summary', {})
    if not igd and not gd:
        print("  (skipping IGD/GD bars — no values)")
        return
    print("  Generating IGD/GD bar chart...")
    algos = [a for a in algo_names if a in igd or a in gd]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, source, title in (
            (axes[0, 0], igd, 'IGD (raw)'),
            (axes[0, 1], gd, 'GD (raw)'),
            (axes[1, 0], igd_n, 'IGD (normalized)'),
            (axes[1, 1], gd_n, 'GD (normalized)')):
        vals = [source.get(a, np.nan) for a in algos]
        ypos = np.arange(len(algos))
        colors = [ALGO_COLORS.get(a, '#cccccc') for a in algos]
        ax.barh(ypos, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(ypos)
        ax.set_yticklabels(algos, fontsize=8)
        ax.set_title(title, fontsize=11)
        ax.grid(axis='x', alpha=0.3)
        ax.invert_yaxis()
    plt.tight_layout()
    path = os.path.join(save_dir, 'igd_gd.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_adaptation_effect(all_results, problems, save_dir):
    pairs = [('SPEA2', 'SPEA2+α'),
             ('MOEA/D', 'MOEA/D+α'),
             ('NSGA-III', 'NSGA-III+α'),
             ('cMLSGA', 'cMLSGA+α')]
    probs = [p for p in problems
             if p in all_results and '_error' not in all_results[p]]
    if not probs:
        return
    print("  Generating α-adaptation effect plot...")
    fig, ax = plt.subplots(figsize=(12, 5))
    n = len(pairs)
    w = 0.8 / n
    x = np.arange(len(probs))
    for j, (strict, fair) in enumerate(pairs):
        vals = []
        for p in probs:
            r = all_results[p]
            if (strict in r and fair in r
                    and r[strict].get('hv_raw_mean', 0) > 0):
                vals.append(r[fair]['hv_raw_mean'] /
                            r[strict]['hv_raw_mean'])
            else:
                vals.append(np.nan)
        ax.bar(x + j * w - (n - 1) * w / 2, vals, w,
               label=f'{strict} → {fair}',
               color=ALGO_COLORS.get(fair, None),
               edgecolor='black', linewidth=0.4)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(probs, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('HV(fair) / HV(strict)   (>1 ⇒ α-adaptation helps)')
    ax.set_title('Effect of α-adaptation on each competitor algorithm',
                 fontsize=13)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_adaptation_effect.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir, stats_results=None):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)
    plot_alpha_adaptation_effect(all_results, problems, save_dir)
    if stats_results is not None:
        plot_igd_gd_bars(stats_results, algo_names, save_dir)


# ============================================================================
# PART 11B: PARETO FRONT PLOTS (cache-aware, from v8.5)
# ============================================================================

_CACHE_DIR_NAME = '_cache'


def _pareto_cache_path(save_dir, prob_name, tag, algo_name):
    cache = os.path.join(save_dir, _CACHE_DIR_NAME)
    safe = (algo_name.replace('α', 'alpha').replace('/', '_')
            .replace('\\', '_').replace(' ', '_'))
    return os.path.join(cache, f"{prob_name}__{tag}__{safe}.npy")


def _load_cached_front(save_dir, prob_name, tag, algo_name):
    p = _pareto_cache_path(save_dir, prob_name, tag, algo_name)
    if os.path.exists(p):
        try:
            return np.load(p, allow_pickle=False)
        except Exception:
            return None
    return None


def _save_cached_front(save_dir, prob_name, tag, algo_name, F):
    cache = os.path.join(save_dir, _CACHE_DIR_NAME)
    os.makedirs(cache, exist_ok=True)
    p = _pareto_cache_path(save_dir, prob_name, tag, algo_name)
    try:
        np.save(p, np.asarray(F, dtype=float))
    except Exception:
        pass


def _downsample_front(F, max_points=60):
    if len(F) <= max_points:
        return F
    order = np.argsort(F[:, 0])
    idx = np.linspace(0, len(F) - 1, max_points).astype(int)
    return F[order][idx]


def _plot_fronts_from_dict(prob_name, fronts, alpha, save_dir,
                          make_3d=True, make_2d_projections=True,
                          max_points=60, n_obj=None):
    if n_obj is None:
        if not fronts:
            return
        for F in fronts.values():
            if F.size:
                n_obj = F.shape[1]
                break
    if not n_obj or not fronts:
        return

    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(8, 6))
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            Fd = _downsample_front(F, max_points)
            order = np.argsort(Fd[:, 0])
            ax.plot(Fd[order, 0], Fd[order, 1], 'o-', markersize=4,
                    label=f"{algo_name} (|PF|={len(F)})",
                    color=ALGO_COLORS.get(algo_name, None),
                    linewidth=1.0, alpha=0.8)
        ax.set_xlabel('$f_1$ (α-weighted)')
        ax.set_ylabel('$f_2$ (α-weighted)')
        ax.set_title(f'{prob_name} — Pareto fronts (α = {alpha})')
        ax.legend(fontsize=7, loc='best')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")
        return

    pairs = [(0, 1), (0, 2), (1, 2)]
    proj_titles = [('$f_1$', '$f_2$'),
                   ('$f_1$', '$f_3$'),
                   ('$f_2$', '$f_3$')]
    if make_2d_projections:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        for ax, (i, j), (xl, yl) in zip(axes, pairs, proj_titles):
            for algo_name, F in fronts.items():
                if len(F) == 0:
                    continue
                Fd = _downsample_front(F, max_points)
                ax.scatter(Fd[:, i], Fd[:, j], s=18, alpha=0.65,
                           label=algo_name if i == 0 and j == 1 else None,
                           color=ALGO_COLORS.get(algo_name, None),
                           edgecolors='black', linewidths=0.2)
            ax.set_xlabel(xl, fontsize=11)
            ax.set_ylabel(yl, fontsize=11)
            ax.grid(alpha=0.3)
        axes[0].legend(fontsize=6, loc='best')
        fig.suptitle(
            f'{prob_name} — Pareto front 2D projections '
            f'(α = {alpha}, downsampled to {max_points})',
            fontsize=12)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2dproj.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    if make_3d:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        for algo_name, F in fronts.items():
            if len(F) == 0:
                continue
            Fd = _downsample_front(F, max_points)
            ax.scatter(Fd[:, 0], Fd[:, 1], Fd[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", s=12,
                       color=ALGO_COLORS.get(algo_name, None),
                       alpha=0.6, edgecolors='black', linewidths=0.15)
        ax.set_xlabel('$f_1$')
        ax.set_ylabel('$f_2$')
        ax.set_zlabel('$f_3$')
        ax.set_title(
            f'{prob_name} — Pareto fronts (α = {alpha}, '
            f'downsampled to {max_points})')
        ax.legend(fontsize=6, loc='upper left')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def _tag_algo_configs(ps, ng):
    all_cfg = get_algo_config(ps, ng, external_mode='both')
    internal_set = set(INTERNAL_ALGO_NAMES)
    fair_set = set(EXTERNAL_ALGO_NAMES)
    strict_set = set(EXTERNAL_ALGO_NAMES_STRICT)
    return {
        'ablation': [c for c in all_cfg if c[0] in internal_set],
        'fair':     [c for c in all_cfg
                     if c[0] in ({'Aα-NSGA-II (adaptive mutation)'}
                                 | fair_set)],
        'strict':   [c for c in all_cfg
                     if c[0] in ({'Aα-NSGA-II (adaptive mutation)'}
                                 | strict_set)],
    }


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False, algo_subset='all',
                                force_recompute=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print(f"  quick={quick}  algo_subset={algo_subset}  "
          f"force_recompute={force_recompute}")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    tags_cfg = _tag_algo_configs(ps, ng)
    if algo_subset == 'all':
        tags = ['ablation', 'fair', 'strict']
    elif algo_subset in tags_cfg:
        tags = [algo_subset]
    else:
        raise ValueError(
            f"algo_subset must be one of 'all', 'ablation', 'fair', "
            f"'strict' — got {algo_subset!r}"
        )

    for tag in tags:
        cfg = tags_cfg[tag]
        out_dir = os.path.join(save_dir, tag)
        os.makedirs(out_dir, exist_ok=True)

        for prob_idx, prob_name in enumerate(problems, 1):
            try:
                benchmark = get_extended_benchmark(prob_name)
            except Exception as e:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name}: "
                      f"get_benchmark failed: {e}")
                continue

            n_obj = benchmark.n_obj
            fronts = {}
            print(f"  [{prob_idx}/{len(problems)}] {prob_name} "
                  f"({tag}, n_obj={n_obj}) ...", flush=True)

            for algo_name, algo_cls, kwargs in cfg:
                F = None
                if not force_recompute:
                    F = _load_cached_front(save_dir, prob_name, tag,
                                           algo_name)
                if F is None:
                    try:
                        algo = algo_cls(benchmark, seed=2026, **kwargs)
                        pop = algo.run()
                        F = _fixed_alpha_objectives(pop, alpha=alpha,
                                                    force=False)
                        _save_cached_front(save_dir, prob_name, tag,
                                           algo_name, F)
                        print(f"      {algo_name}: |PF|={len(F)}",
                              flush=True)
                        del algo, pop
                    except Exception as e:
                        print(f"      {algo_name}: FAILED "
                              f"({type(e).__name__}: {e})", flush=True)
                        traceback.print_exc()
                        F = None
                else:
                    print(f"      {algo_name}: cached |PF|={len(F)}",
                          flush=True)
                if F is not None and F.size > 0:
                    fronts[algo_name] = F

            if not fronts:
                print(f"      (no fronts; skipping plot)", flush=True)
                continue

            try:
                _plot_fronts_from_dict(
                    prob_name, fronts, alpha, out_dir,
                    make_3d=True,
                    make_2d_projections=True,
                    max_points=40,
                    n_obj=n_obj,
                )
            except Exception as e:
                print(f"      plot failed for {prob_name} ({tag}): "
                      f"{type(e).__name__}: {e}", flush=True)
                traceback.print_exc()
            finally:
                del fronts
                gc.collect()


# ============================================================================
# PART 11C: Σ̂ TABLE — v8.6 with sanity check
# ============================================================================

def generate_sigma_hat_table(all_results, problems, save_dir,
                             n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂  (v8.6: force=True fixed-α)")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
                n_runs_per_alpha=n_runs_per_alpha,
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    df = df.sort_values(['family', 'problem']).reset_index(drop=True)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. $\widehat{\Sigma}$ is the normalized spread of the '
        r'fixed-$\alpha$ hypervolume across $\alpha \in \{0.5, 0.75, 1.0\}$, '
        r'averaged over '
        f'{n_runs_per_alpha} seeds and computed against a '
        r'single reference point (union of all fronts). '
        r'Each front is evaluated under a FIXED $\alpha$ (per-individual '
        r'$\alpha$ is ignored) so the sensitivity genuinely reflects '
        r'the evaluation weight. '
        r'F1 problems are $\alpha$-invariant by construction, so their '
        r'$\widehat{\Sigma}$ should be near zero; F2 problems are '
        r'$\alpha$-sensitive.}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\overline{\mathrm{HV}}(0.5)$ & '
        r'$\overline{\mathrm{HV}}(1.0)$ \\',
        r'\midrule',
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")

    verify_sigma_hat_families(save_dir)


# ============================================================================
# PART 12 / 13: LATEX, SETTINGS, MAIN
# ============================================================================

def _build_external_table(all_results, problems, algo_names,
                          stats_results, save_dir, filename, caption, label):
    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    igds = {a: [] for a in algo_names}
    gds = {a: [] for a in algo_names}
    igds_n = {a: [] for a in algo_names}
    gds_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                igds[a].append(igd)
            if not np.isnan(gd):
                gds[a].append(gd)
            if not np.isnan(igd_n):
                igds_n[a].append(igd_n)
            if not np.isnan(gd_n):
                gds_n[a].append(gd_n)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{' + caption + '}',
        r'\label{' + label + '}',
        r'\begin{tabular}{@{}lccccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Rank & Wins & IGD & GD & '
        r'IGD$_n$ \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        igd_s = (f'{np.mean(igds[algo]):.4f}' if igds[algo] else '--')
        gd_s = (f'{np.mean(gds[algo]):.4f}' if gds[algo] else '--')
        igd_n_s = (f'{np.mean(igds_n[algo]):.4f}' if igds_n[algo] else '--')
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} & {igd_s} & {gd_s} & '
                     f'{igd_n_s} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, filename), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))


def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_strict, stats_fair, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    strict_algos = ['Aα-NSGA-II (adaptive mutation)'] + EXTERNAL_ALGO_NAMES_STRICT
    _build_external_table(
        all_results, problems, strict_algos, stats_strict, save_dir,
        filename='table_external_strict.tex',
        caption=(
            r'External comparison against standard state-of-the-art '
            r'algorithms with fixed $\alpha = 0.75$. This is the primary '
            r'external comparison; it tests whether adaptive-$\alpha$ is '
            r'valuable. IGD/GD are computed against a memory-safe, '
            r'per-algorithm-balanced empirical reference front; lower is '
            r'better. IGD$_n$ is IGD divided by the reference front '
            r'bounding-box diagonal for cross-problem comparability.'
        ),
        label='tab:external_strict',
    )

    fair_algos = ['Aα-NSGA-II (adaptive mutation)'] + EXTERNAL_ALGO_NAMES
    _build_external_table(
        all_results, problems, fair_algos, stats_fair, save_dir,
        filename='table_external_fair.tex',
        caption=(
            r'Ablation: competitors are $\alpha$-wrapped so they receive the '
            r'same $\alpha$-adaptation capability as A$\alpha$-NSGA-II. This '
            r'isolates the contribution of the selection and mutation '
            r'mechanisms when the $\alpha$ advantage is neutralized.'
        ),
        label='tab:external_fair',
    )

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Internal ablation: overall performance on all problems. '
        r'Normalized α-weighted HV (fixed $\alpha = 0.75$) and raw IGD/GD.}',
        r'\label{tab:overall_internal}',
        r'\begin{tabular}{@{}lccccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Rank & Wins & IGD & GD & '
        r'IGD$_n$ \\',
        r'\midrule',
    ]
    for algo in INTERNAL_ALGO_NAMES:
        hv_all = []
        ranks_all = []
        igd_all = []
        gd_all = []
        igd_n_all = []
        for prob_name in problems:
            if prob_name not in all_results:
                continue
            res = all_results[prob_name]
            vals = [(a, res[a]['hv_raw_mean']) for a in INTERNAL_ALGO_NAMES
                    if a in res]
            if not vals:
                continue
            vals.sort(key=lambda x: -x[1])
            best = vals[0][1]
            if best > 0 and algo in res:
                hv_all.append(res[algo]['hv_raw_mean'] / best)
            for r, (a, _) in enumerate(vals, 1):
                if a == algo:
                    ranks_all.append(r)
            if algo in res:
                if not np.isnan(res[algo].get('igd_mean', np.nan)):
                    igd_all.append(res[algo]['igd_mean'])
                if not np.isnan(res[algo].get('gd_mean', np.nan)):
                    gd_all.append(res[algo]['gd_mean'])
                if not np.isnan(res[algo].get('igd_norm_mean', np.nan)):
                    igd_n_all.append(res[algo]['igd_norm_mean'])
        if not hv_all:
            continue
        lines.append(
            f'{algo} & {np.mean(hv_all):.4f} & {np.std(hv_all):.4f} & '
            f'{np.mean(ranks_all):.2f} & '
            f'{sum(1 for r in ranks_all if r == 1)} & '
            f'{(np.mean(igd_all) if igd_all else float("nan")):.4f} & '
            f'{(np.mean(gd_all) if gd_all else float("nan")):.4f} & '
            f'{(np.mean(igd_n_all) if igd_n_all else float("nan")):.4f} \\\\'
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    for tag, stats_res, ext_names in (
            ('strict', stats_strict, EXTERNAL_ALGO_NAMES_STRICT),
            ('fair', stats_fair, EXTERNAL_ALGO_NAMES)):
        lines = [
            r'\begin{table}[htbp]', r'\centering',
            r'\caption{Pairwise Wilcoxon (Bonferroni) on ranks — '
            + tag + r' external comparison.}',
            r'\label{tab:statistics_' + tag + '}',
            r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
            r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
            r'\midrule',
        ]
        ref = CONFIG['reference_algo']
        for algo, res in stats_res.get('pairwise', {}).items():
            if algo not in ext_names:
                continue
            lines.append(
                f"{res.get('family', '--')} & {ref} vs. {algo} & "
                f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
                f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
            )
        lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
        with open(os.path.join(save_dir,
                               f'table_statistics_{tag}.tex'), 'w',
                  encoding='utf-8') as f:
            f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_strict['bootstrap_cis']:
            mean, lo, hi = stats_strict['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")

    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')


def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None,
                   external_mode=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if external_mode is not None:
        global EXTERNAL_COMPARISON_MODE
        EXTERNAL_COMPARISON_MODE = external_mode
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def run_pareto_only(problems=None, quick=False,
                    pop=None, gen=None,
                    algo_subset='all',
                    force_recompute=False):
    apply_settings(mode='full', pop=pop, gen=gen)
    if problems is None:
        problems = PARETO_DEFAULT_PROBLEMS
    ps = CONFIG['pop_size']
    ng = CONFIG['n_gen']
    if quick:
        ps, ng = 40, 150

    out = os.path.join(CONFIG['figures_dir'], 'pareto_fronts')
    os.makedirs(out, exist_ok=True)
    for tag in ('ablation', 'fair', 'strict'):
        os.makedirs(os.path.join(out, tag), exist_ok=True)

    print("=" * 78)
    print(f"PARETO-ONLY MODE (v8.6) — {len(problems)} problems, "
          f"pop={ps}, gen={ng}, α={FAIR_ALPHA}")
    print(f"  algo_subset   = {algo_subset}")
    print(f"  quick         = {quick}")
    print(f"  recompute     = {force_recompute}")
    print(f"Output: {out}")
    print("=" * 78)

    generate_all_pareto_fronts(
        problems, out, alpha=FAIR_ALPHA, quick=quick,
        algo_subset=algo_subset,
        force_recompute=force_recompute,
    )

    print("\nPareto-only mode complete.")


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True, external_mode=None,
         skip_main_runs=False, skip_igd_gd=False,
         pareto_only=False, pareto_problems=None,
         pareto_algos='all', pareto_force=False):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet,
                   external_mode=external_mode)

    if pareto_only:
        return run_pareto_only(
            problems=pareto_problems,
            quick=pareto_quick,
            pop=pop, gen=gen,
            algo_subset=pareto_algos,
            force_recompute=pareto_force,
        )

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v8.6)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"External comparison mode: {EXTERNAL_COMPARISON_MODE}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    # v8.6: probe F2 override when computing σ̂.
    if compute_sigma:
        _probe_f2_override()

    _cfg_strict = get_algo_config(CONFIG['pop_size'], CONFIG['n_gen'],
                                   external_mode='strict')
    for name, _, kwargs in _cfg_strict:
        if name in EXTERNAL_ALGO_NAMES_STRICT:
            assert kwargs.get('alpha', FAIR_ALPHA) == FAIR_ALPHA, (
                f"Strict-mode {name} uses alpha={kwargs.get('alpha')}, "
                f"expected FAIR_ALPHA={FAIR_ALPHA}"
            )

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if not skip_main_runs:
        if CONFIG['use_parallel']:
            all_results = run_parallel(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                n_workers=CONFIG['n_workers'],
                checkpoint_path=checkpoint_path,
                verbose=not CONFIG['quiet'],
                monitor=None,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                checkpoint_path=checkpoint_path,
                verbose=True,
                monitor=None,
            )
    else:
        if os.path.exists(checkpoint_path):
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            print(f"\n[skip_main_runs] Loaded {len(all_results)} problems "
                  f"from checkpoint")
        else:
            all_results = {}
            print("\n[skip_main_runs] No checkpoint found — nothing to do.")

    # v8.6: always run the IGD/GD pass (fast when cached), and ALWAYS
    # persist the merged result. This fixes the "empty IGD/GD rows in
    # statistics" bug from v8.4.
    if all_results and not skip_igd_gd:
        all_results = compute_igd_gd_pass(
            all_results, CONFIG['problems'],
            CONFIG['pop_size'], CONFIG['n_gen'],
            CONFIG['seed_base'], CONFIG['n_runs'],
            verbose=True,
        )
        _save_checkpoint(all_results, checkpoint_path)
    elif skip_igd_gd:
        print("\n[skip_igd_gd] IGD/GD pass skipped — statistics will "
              "show '--' for IGD/GD columns.")

    stats_fair = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
        label='[FAIR = ABLATION]',
    )

    strict_algo_names = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT
    stats_strict = run_statistical_tests(
        all_results, CONFIG['problems'], strict_algo_names,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES_STRICT if include_external else [],
        label='[STRICT = PRIMARY]',
    )

    report_framing(stats_strict, strict_algo_names)

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_strict['bootstrap_cis'], CONFIG['figures_dir'],
        stats_results=stats_strict,
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_strict, stats_fair, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")
            traceback.print_exc()

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")
        traceback.print_exc()

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in (EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT):
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                    'igd_mean': results[algo].get('igd_mean', np.nan),
                    'gd_mean': results[algo].get('gd_mean', np.nan),
                    'igd_norm_mean': results[algo].get('igd_norm_mean', np.nan),
                    'gd_norm_mean': results[algo].get('gd_norm_mean', np.nan),
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
                'igd_mean': res.get('igd_mean', np.nan),
                'gd_mean': res.get('gd_mean', np.nan),
                'igd_norm_mean': res.get('igd_norm_mean', np.nan),
                'gd_norm_mean': res.get('gd_norm_mean', np.nan),
            })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(CONFIG['results_dir'], 'results.csv'),
            index=False, encoding='utf-8')

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v8.6')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true')
    parser.add_argument('--full-pareto', action='store_true')
    parser.add_argument('--no-sigma', action='store_true')
    parser.add_argument('--no-external', action='store_true')
    parser.add_argument('--no-igd-gd', action='store_true')
    parser.add_argument('--external-mode', choices=['fair', 'strict', 'both'],
                        default='both')
    parser.add_argument('--force-parallel', action='store_true')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    parser.add_argument('--resume', action='store_true')

    parser.add_argument('--pareto-only', action='store_true',
                        help='Generate ONLY the Pareto front figures.')
    parser.add_argument('--pareto-problems', type=str, default=None,
                        help='Comma-separated problems for --pareto-only.')
    parser.add_argument('--pareto-algos',
                        choices=['all', 'ablation', 'fair', 'strict'],
                        default='all')
    parser.add_argument('--pareto-force', action='store_true')
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    if args.pareto_only:
        probs = None
        if args.pareto_problems:
            if args.pareto_problems.strip().lower() == 'all':
                probs = PARETO_DEFAULT_PROBLEMS
            else:
                probs = [s.strip() for s in args.pareto_problems.split(',')
                         if s.strip()]
        run_pareto_only(
            problems=probs,
            quick=not args.full_pareto,
            pop=args.pop, gen=args.gen,
            algo_subset=args.pareto_algos,
            force_recompute=args.pareto_force,
        )
        return

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external,
         external_mode=args.external_mode,
         skip_main_runs=args.resume,
         skip_igd_gd=args.no_igd_gd,
         pareto_only=False,
         pareto_problems=None,
         pareto_algos='all',
         pareto_force=False)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v8.6)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
External comparison mode: both
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3
Fair HV metric uses α = 0.75
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\supplementary_latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt C

In [5]:
#!/usr/bin/env python3
# ============================================================================
# Aα-NSGA-II: COMPLETE PIPELINE (v8.8)
#   Adaptive α + Adaptive Mutation + Fair HV + IGD/GD + Pareto Plots
#   + F2 family + α-sensitivity proxy Σ̂
#   + External benchmarks: SPEA2, MOEA/D, NSGA-III, cMLSGA
#     * strict: fixed α = FAIR_ALPHA (HEADLINE)
#     * fair:   α-adaptation wrapper (ABLATION)
#
# v8.8 changes vs. v8.7:
#   - FIXED `compute_sigma_hat`: Now correctly uses the SAME population 
#     across all three α values. This isolates α-sensitivity from 
#     cross-seed variance (properly yielding F1 ≈ 0 and F2 > 0).
#   - FIXED `verify_sigma_hat_families`: Adjusted the warning thresholds
#     to match the expected residual trapezoidal geometry for F1.
#   - ENHANCED Pareto plots (strict priority):
#       * Matplotlib C0-C4 colors for max contrast.
#       * Distinct markers (o, s, ^, D, v) so black-and-white printing works.
#       * Black edgecolors to stop overlapping dots from blurring.
#       * Z-order sorting ensures the Hero algorithm is drawn on top.
#   - ADDED `--pareto-force` safety check in `run_pareto_only` to 
#     automatically clear stale cached arrays if colors/shapes change.
#   - ADDED pre-flight validation in `main()` to warn if checkpoint.pkl
#     is missing IGD/GD data before generating LaTeX tables.
# ============================================================================

import numpy as np
import pandas as pd
import time
import os
import sys
import json
import pickle
import gc
import multiprocessing as mp
import traceback
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime
from concurrent.futures import (
    ProcessPoolExecutor, ThreadPoolExecutor, as_completed,
    TimeoutError as FutTimeout,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import stats

import warnings
warnings.filterwarnings('ignore', category=UserWarning, message='.*Axes3D.*')
warnings.filterwarnings('ignore', category=UserWarning, message='.*identical low.*')

EPS = 1e-12


def _detect_jupyter() -> bool:
    try:
        return ('ipykernel' in sys.modules) or ('google.colab' in sys.modules)
    except Exception:
        return False


IN_JUPYTER = _detect_jupyter()
IS_TTY = (not IN_JUPYTER) and sys.stdout.isatty()

FORCE_PARALLEL_FROM_JUPYTER = False

OUTPUT_ROOT = (r"D:\Container Transportation Routing Problems"
               r"\Manuscript5-Opt CTRouting Problem as a M2-M PUD"
               r"\Fuzzy_Aα_NSGA_II-Results_v8_3")

FAIR_ALPHA = 0.75

SIGMA_HAT_ALPHAS = (0.50, 0.75, 1.00)
SIGMA_HAT_RUNS_PER_ALPHA = 5
SIGMA_HAT_REF_QUANTILE   = 0.95

SPEA2_ARCHIVE_SIZE    = None
SPEA2_KNN_K           = 1
MOEAD_NEIGHBORHOOD_T  = 20
MOEAD_DIVISIONS       = {2: 12, 3: 6}
NSGA3_DIVISIONS       = {2: 12, 3: 6}
CMLSGA_N_SUBPOPS      = 8
CMLSGA_MIGRATION_FREQ = 10

EXTERNAL_COMPARISON_MODE = 'both'

REF_FRONT_MAX_INPUT  = 600
REF_FRONT_MAX_OUTPUT = 100
IGD_PER_ALGO_CAP     = 150
IGD_SKIP = {'FScal5000', 'FScal10000'}


# ============================================================================
# PART 0: PROGRESS MONITOR
# ============================================================================
class ProgressMonitor:
    HEARTBEAT_LOG_SEC = 30
    HEARTBEAT_DRAW_SEC = 2
    JUPYTER_DRAW_SEC = 30

    def __init__(self, total_tasks, output_dir,
                 log_name='progress.log', json_name='progress.json',
                 in_jupyter=False, no_console=False):
        self.total = total_tasks
        self.completed = 0
        self.failed = 0
        self.started_at = time.time()
        self.last_completion_at = self.started_at
        self.current_in_flight = {}
        self.per_problem_times = {}
        self.output_dir = output_dir
        self.log_path = os.path.join(output_dir, log_name)
        self.json_path = os.path.join(output_dir, json_name)
        self.in_jupyter = in_jupyter
        self.no_console = no_console
        self._last_draw = 0.0
        self._last_log_hb = 0.0
        self._last_state_completed = -1
        os.makedirs(output_dir, exist_ok=True)
        self._log("=" * 78)
        self._log(f"[{self._now()}] MONITOR STARTED — {total_tasks} tasks")
        self._log("=" * 78)
        self._dump_json()
        self._draw(force=True)

    def start_task(self, name):
        self.current_in_flight[name] = time.time()

    def task_done(self, name, success=True):
        t0 = self.current_in_flight.pop(name, time.time())
        elapsed = time.time() - t0
        self.per_problem_times[name] = elapsed
        self.completed += 1
        if not success:
            self.failed += 1
        self.last_completion_at = time.time()
        marker = "✓" if success else "✗"
        self._log(f"[{self._now()}] {marker} {name}  ({elapsed:.1f}s)  "
                  f"[{self.completed}/{self.total}]")
        self._draw(force=True)
        self._dump_json()

    def tick(self):
        now = time.time()
        if now - self._last_log_hb >= self.HEARTBEAT_LOG_SEC:
            self._last_log_hb = now
            since = now - self.last_completion_at
            self._log(f"[{self._now()}] ♥ heartbeat  "
                      f"[{self.completed}/{self.total}]  "
                      f"active={len(self.current_in_flight)}  "
                      f"last_done={int(since)}s ago")
        self._draw()
        self._dump_json()

    def finish(self):
        total = time.time() - self.started_at
        self._log("-" * 78)
        self._log(f"[{self._now()}] ALL DONE — {self.completed}/{self.total} "
                  f"({self.failed} failed) in {self._fmt_dur(total)}")
        self._log("-" * 78)
        self._dump_json(final=True)
        if IS_TTY:
            sys.stdout.write('\n')
        sys.stdout.flush()

    def _draw(self, force=False):
        if self.no_console:
            return
        now = time.time()
        threshold = self.JUPYTER_DRAW_SEC if self.in_jupyter else self.HEARTBEAT_DRAW_SEC
        state_changed = (self.completed != self._last_state_completed)
        if not force and not state_changed and (now - self._last_draw) < threshold:
            return
        self._last_draw = now
        self._last_state_completed = self.completed
        pct = self.completed / max(self.total, 1)
        bar_w = 30
        filled = int(round(bar_w * pct))
        bar = '█' * filled + '░' * (bar_w - filled)
        elapsed = time.time() - self.started_at
        eta_str = "~?"
        if self.completed > 0:
            rate = self.completed / max(elapsed, 1e-6)
            remaining = (self.total - self.completed) / max(rate, 1e-9)
            eta_str = self._fmt_dur(remaining)
        line = (f"  [{bar}] {self.completed:>3}/{self.total} "
                f"({pct*100:5.1f}%)  ETA {eta_str:>7}  "
                f"active={len(self.current_in_flight)}")
        if IS_TTY and not self.in_jupyter:
            sys.stdout.write('\r' + line + ' ' * 12)
            sys.stdout.flush()
        else:
            print(line, flush=True)

    def _dump_json(self, final=False):
        elapsed = time.time() - self.started_at
        data = {
            'total': self.total, 'completed': self.completed,
            'failed': self.failed,
            'percent': round(100.0 * self.completed / max(self.total, 1), 2),
            'elapsed_human': self._fmt_dur(elapsed),
            'active_count': len(self.current_in_flight),
            'in_flight': list(self.current_in_flight.keys()),
            'per_problem_seconds': {k: round(v, 2)
                                    for k, v in self.per_problem_times.items()},
            'final': final, 'updated_at': self._now(),
        }
        try:
            tmp = self.json_path + '.tmp'
            with open(tmp, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            os.replace(tmp, self.json_path)
        except Exception:
            pass

    def _log(self, msg):
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(msg + '\n')
        except Exception:
            pass

    @staticmethod
    def _now():
        return datetime.now().strftime('%H:%M:%S')

    @staticmethod
    def _fmt_dur(seconds):
        seconds = max(0, int(seconds))
        if seconds < 60:
            return f"{seconds}s"
        m, s = divmod(seconds, 60)
        if m < 60:
            return f"{m}m{s:02d}s"
        h, m = divmod(m, 60)
        return f"{h}h{m:02d}m"


# ============================================================================
# PART 1: FUZZY MATHEMATICS
# ============================================================================

def _sort3(a, b, c):
    s = sorted([a, b, c])
    return float(s[0]), float(s[1]), float(s[2])


def _sort4(a, b, c, d):
    s = sorted([a, b, c, d])
    return float(s[0]), float(s[1]), float(s[2]), float(s[3])


class TriangularFuzzyNumber:
    __slots__ = ('aL', 'aM', 'aU')

    def __init__(self, aL, aM, aU):
        aL, aM, aU = _sort3(aL, aM, aU)
        self.aL, self.aM, self.aU = aL, aM, aU

    def __repr__(self):
        return f"TFN({self.aL:.4f}, {self.aM:.4f}, {self.aU:.4f})"

    def expected_value(self):
        return (self.aL + 4 * self.aM + self.aU) / 6.0


class TrapezoidalFuzzyNumber:
    __slots__ = ('a1', 'a2', 'a3', 'a4')

    def __init__(self, a1, a2, a3, a4):
        a1, a2, a3, a4 = _sort4(a1, a2, a3, a4)
        self.a1, self.a2, self.a3, self.a4 = a1, a2, a3, a4

    def __repr__(self):
        return f"TrFN({self.a1:.4f}, {self.a2:.4f}, {self.a3:.4f}, {self.a4:.4f})"

    def expected_value(self):
        return (self.a1 + 2 * self.a2 + 2 * self.a3 + self.a4) / 6.0

    def to_triangular(self):
        return TriangularFuzzyNumber(self.a1, 0.5 * (self.a2 + self.a3), self.a4)


def credibility_leq_triangular(A, B):
    denom = (A.aU - A.aL) + (B.aU - B.aL)
    if denom <= EPS:
        return 1.0 if A.aU <= B.aL else 0.0
    return float(np.clip((B.aU - A.aL) / denom, 0.0, 1.0))


def crisp_to_trapezoidal(x, delta=0.30, eta=1.0):
    spread = abs(x) * delta * eta if x != 0 else delta * eta
    return TrapezoidalFuzzyNumber(
        x - 1.0 * spread, x - 0.5 * spread,
        x + 0.5 * spread, x + 1.5 * spread,
    )


# ============================================================================
# PART 2: FUZZY OPERATORS
# ============================================================================

def _tfn_dominates_arrays(A, B):
    AL, AM, AU = A[:, 0], A[:, 1], A[:, 2]
    BL, BM, BU = B[:, 0], B[:, 1], B[:, 2]
    denom = (AU - AL) + (BU - BL)
    num = BU - AL
    credA = np.where(denom > EPS,
                     np.clip(num / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
                     (AU <= BL).astype(float))
    strict = AL > BU
    return bool(np.all(credA >= 0.5)) and bool(np.any(strict))


def fast_nondominated_sort_fuzzy(F, chunk_size=256):
    N = F.shape[0]
    if N == 0:
        return []
    L = F[:, :, 0].astype(float)
    U = F[:, :, 2].astype(float)
    W = U - L
    D = np.zeros((N, N), dtype=bool)
    chunk = max(1, int(chunk_size))
    for start in range(0, N, chunk):
        end = min(start + chunk, N)
        Lc = L[start:end]
        Uc = U[start:end]
        Wc = W[start:end]
        num = U[None, :, :] - Lc[:, None, :]
        den = Wc[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        degenerate = den <= EPS
        if degenerate.any():
            deg_vals = (U[None, :, :] <= Lc[:, None, :]).astype(float)
            cred[degenerate] = deg_vals[degenerate]
            del deg_vals
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(Lc[:, None, :] > U[None, :, :], axis=2)
        D[start:end] = weak_all & strict_any
        del num, den, safe_den, cred, degenerate, weak_all, strict_any
    np.fill_diagonal(D, False)
    n = (D & ~D.T).sum(axis=0).astype(int)
    S = [np.flatnonzero(D[p]) for p in range(N)]
    fronts = [list(np.flatnonzero(n == 0))]
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                n[q] -= 1
                if n[q] == 0:
                    nxt.append(int(q))
        i += 1
        fronts.append(nxt)
    return fronts[:-1]


def fuzzy_crowding_distance(F, front):
    n = len(front)
    dist = np.zeros(n)
    if n <= 2:
        dist[:] = np.inf
        return dist
    idx_front = np.asarray(front, dtype=int)
    E_all = (F[idx_front, :, 0] + 4 * F[idx_front, :, 1] +
             F[idx_front, :, 2]) / 6.0
    for m in range(F.shape[1]):
        E = E_all[:, m]
        order = np.argsort(E)
        dist[order[0]] = np.inf
        dist[order[-1]] = np.inf
        rng = E[order[-1]] - E[order[0]]
        if rng <= EPS:
            continue
        mid = order[1:-1]
        dist[mid] += (E[order[2:]] - E[order[:-2]]) / rng
    return dist


def das_dennis_points(M, H):
    from itertools import combinations_with_replacement
    points = []
    for c in combinations_with_replacement(range(H + 1), M - 1):
        w = [c[0]]
        for i in range(1, M - 1):
            w.append(c[i] - c[i - 1])
        w.append(H - c[-1])
        points.append([x / H for x in w])
    return np.array(points)


# ============================================================================
# PART 3: FUZZY BENCHMARKS
# ============================================================================
class FuzzyBenchmark:
    name = "base"
    n_var = 30
    n_obj = 2

    def evaluate_crisp(self, x):
        raise NotImplementedError

    def evaluate_fuzzy(self, x, delta=0.30):
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


class FZDT1(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT1", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT2(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT2", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FZDT3(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT3", 30, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 9.0 * np.sum(x[1:]) / (len(x) - 1)
        return np.array([f1, g * (1 - np.sqrt(f1 / g) -
                                   (f1 / g) * np.sin(10 * np.pi * f1))])


class FZDT4(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT4", 10, 2
    def evaluate_crisp(self, x):
        f1 = x[0]
        g = 1.0 + 10.0 * (len(x) - 1) + np.sum(
            x[1:] ** 2 - 10 * np.cos(4 * np.pi * x[1:]))
        return np.array([f1, g * (1 - np.sqrt(f1 / g))])


class FZDT6(FuzzyBenchmark):
    name, n_var, n_obj = "FZDT6", 10, 2
    def evaluate_crisp(self, x):
        f1 = 1 - np.exp(-4 * x[0]) * (np.sin(6 * np.pi * x[0])) ** 6
        g = 1.0 + 9.0 * (np.sum(x[1:]) / (len(x) - 1)) ** 0.25
        return np.array([f1, g * (1 - (f1 / g) ** 2)])


class FDTLZ1(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ1", 7, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = 0.5 * (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[self.n_obj - i - 1])
        return f


class FDTLZ2(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ2", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ3(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ3", 12, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 100 * (k + np.sum((x[self.n_obj - 1:] - 0.5) ** 2
                              - np.cos(20 * np.pi * (x[self.n_obj - 1:] - 0.5))))
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FDTLZ4(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ4", 12, 3
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * (x[j] ** 100))
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * (x[self.n_obj - i - 1] ** 100))
        return f


class FDTLZ7(FuzzyBenchmark):
    name, n_var, n_obj = "FDTLZ7", 22, 3
    def evaluate_crisp(self, x):
        k = self.n_var - self.n_obj + 1
        g = 1 + 9.0 / k * np.sum(x[self.n_obj - 1:])
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj - 1):
            f[i] = x[i]
        h = self.n_obj - np.sum(
            f[:self.n_obj - 1] / (1 + g) *
            (1 + np.sin(3 * np.pi * f[:self.n_obj - 1])))
        f[self.n_obj - 1] = (1 + g) * h
        return f


def _wfg_bias_poly(y, a):
    return y ** a


def _wfg_bias_flat(y, A, B, C):
    return A + np.minimum(0, np.floor(y - B)) * A * (B - y) / B \
           - np.minimum(0, np.floor(C - y)) * (1 - A) * (C - y) / (1 - C)


def _wfg_bias_param(y, u, A, B, C):
    return y ** (B + (C - B) * (u ** (A - 2 * (1 - u) * (A - 2))))


def _wfg_shift_linear(y, A):
    return np.abs(y - A) / np.abs(np.floor(A - y) + A)


def _wfg_shift_deceptive(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    base = np.clip(1 - B - y, 0.0, None)
    return r * (0.1 * (1 - y) * base ** C + 0.1 * y * (A - y))


def _wfg_shift_multi(y, A, B, C):
    r = np.abs(y - A) / np.abs(np.floor(A - y) + A)
    return (1 + np.cos((4 * A + 2) * np.pi * (0.5 - r))) / (2 * (1 + B))


def _wfg_reduce_sum(y, w):
    return np.sum(w * y) / np.sum(w)


def _wfg_reduce_nonsep(y, A):
    n = len(y)
    if A < 2:
        A = 2
    result = 0.0
    for j in range(n):
        for k in range(A - 1):
            result += np.abs(y[j] - y[(j + k + 1) % n])
    return (result + np.sum(y)) / (n * A)


def _shape_linear(x, m):
    return x / m


def _shape_convex(x, m):
    return (1 - np.cos(x * np.pi / 2)) / m


def _shape_concave(x, m):
    return np.sin(x * np.pi / 2) / m


def _shape_mixed(x, m):
    return (1 - x - np.cos(2 * np.pi * x + np.pi / 2) / (2 * np.pi)) / m


def _shape_disc(x, m):
    return (1 - x - np.cos(np.pi * x)) / m


class WFGBase(FuzzyBenchmark):
    def __init__(self, n_obj=3, n_var=12):
        self.n_obj = n_obj
        self.k = n_obj - 1
        self.l = n_var - self.k
        self.n_var = n_var

    def evaluate_crisp(self, z):
        z = np.asarray(z, dtype=float) * np.arange(1, self.n_var + 1) * 2.0 / self.n_var
        return self._shape(self._transform(z))


class FWFG1(WFGBase):
    name, n_var, n_obj = "FWFG1", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = z.copy()
        y[:k] = _wfg_bias_poly(y[:k], 0.02)
        y[k:] = _wfg_bias_flat(y[k:], 0.8, 0.75, 0.85)
        y = _wfg_shift_linear(y, 0.35)
        w = np.arange(2, 2 * n + 1, 2)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m) if i == 0 else _shape_mixed(x[i], m)
        f[-1] = _shape_mixed(x[-1], m)
        return f


class FWFG2(WFGBase):
    name, n_var, n_obj = "FWFG2", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[k:], len(y[k:]))
        return t

    def _shape(self, x):
        m = self.n_obj
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = _shape_convex(x[i], m)
        f[-1] = _shape_disc(x[-1], m)
        return f


class FWFG3(WFGBase):
    name, n_var, n_obj = "FWFG3", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], 2)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y[self.k:]))
        return t

    def _shape(self, x):
        return np.array([_shape_linear(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG4(WFGBase):
    name, n_var, n_obj = "FWFG4", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_multi(z, 30, 10, 0.35)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG5(WFGBase):
    name, n_var, n_obj = "FWFG5", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        y = _wfg_shift_deceptive(z, 0.35, 0.001, 0.05)
        w = np.ones(n)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG6(WFGBase):
    name, n_var, n_obj = "FWFG6", 12, 3
    def _transform(self, z):
        y = _wfg_shift_linear(z, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = self.k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[self.k:], len(y) - self.k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG7(WFGBase):
    name, n_var, n_obj = "FWFG7", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k):
            u = _wfg_reduce_sum(z[k:], w[k:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_sum(y[i * k_seg:(i + 1) * k_seg], w[:k_seg])
        t[-1] = _wfg_reduce_sum(y[k:], w[k:])
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG8(WFGBase):
    name, n_var, n_obj = "FWFG8", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(k, n):
            u = _wfg_reduce_sum(z[:k], w[:k])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_linear(y, 0.35)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FWFG9(WFGBase):
    name, n_var, n_obj = "FWFG9", 12, 3
    def _transform(self, z):
        n, k = len(z), self.k
        w = np.ones(n)
        y = z.copy()
        for i in range(n - 1):
            u = _wfg_reduce_sum(z[i + 1:], w[i + 1:])
            y[i] = _wfg_bias_param(z[i], u, 0.98 / 49.98, 0.02, 50)
        y = _wfg_shift_deceptive(y, 0.35, 0.001, 0.05)
        t = np.zeros(self.n_obj)
        k_seg = k // (self.n_obj - 1)
        for i in range(self.n_obj - 1):
            t[i] = _wfg_reduce_nonsep(y[i * k_seg:(i + 1) * k_seg], k_seg)
        t[-1] = _wfg_reduce_nonsep(y[k:], n - k)
        return t

    def _shape(self, x):
        return np.array([_shape_concave(x[i], self.n_obj)
                         for i in range(self.n_obj)])


class FUF1(FuzzyBenchmark):
    name, n_var, n_obj = "FUF1", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(
            (x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            (x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))) ** 2)
        return np.array([f1, f2])


class FUF2(FuzzyBenchmark):
    name, n_var, n_obj = "FUF2", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        f1 = x[0] + 2.0 / len(J1) * np.sum(np.abs(x[J1] - 0.5 * (x[0] + 1)) ** 2)
        f2 = 1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(
            np.abs(x[J2] - 0.5 * (1 - np.sqrt(x[0]))) ** 2)
        return np.array([f1, f2])


class FUF3(FuzzyBenchmark):
    name, n_var, n_obj = "FUF3", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - x[0] ** (0.5 * (1 + 3 * (J1 - 2) / (len(x) - 2)))
        Y2 = x[J2] - x[0] ** (0.5 * (1 + 3 * (J2 - 2) / (len(x) - 2)))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - np.sqrt(x[0]) + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF4(FuzzyBenchmark):
    name, n_var, n_obj = "FUF4", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + 2.0 / len(J1) * np.sum(Y1 ** 2),
                         1 - x[0] ** 2 + 2.0 / len(J2) * np.sum(Y2 ** 2)])


class FUF5(FuzzyBenchmark):
    name, n_var, n_obj = "FUF5", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.cos(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(np.abs(Y1)),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(np.abs(Y2))])


class FUF6(FuzzyBenchmark):
    name, n_var, n_obj = "FUF6", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        Y1 = x[J1] - np.sin(6 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        return np.array([x[0] + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         1 - x[0] + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF7(FuzzyBenchmark):
    name, n_var, n_obj = "FUF7", 30, 2
    def evaluate_crisp(self, x):
        J1 = np.arange(2, len(x), 2)
        J2 = np.arange(1, len(x), 2)
        y1 = x[0] ** 0.2
        y2 = 1 - x[0] ** 0.2
        Y1 = x[J1] - np.sin(6 * np.pi * y1 + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - np.sin(6 * np.pi * y2 + (J2 + 1) * np.pi / len(x))
        return np.array([y1 + (2.0 / len(J1)) * np.sum(Y1 ** 2),
                         y2 + (2.0 / len(J2)) * np.sum(Y2 ** 2)])


class FUF8(FuzzyBenchmark):
    name, n_var, n_obj = "FUF8", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF9(FuzzyBenchmark):
    name, n_var, n_obj = "FUF9", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J1)) * np.sum(Y1 ** 2),
            0.5 * (np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) + 0.5) +
            (2.0 / len(J2)) * np.sum(Y2 ** 2),
            0.5 * (np.sin(0.5 * np.pi * x[0]) + 0.5) +
            (2.0 / len(J3)) * np.sum(Y3 ** 2)])


class FUF10(FuzzyBenchmark):
    name, n_var, n_obj = "FUF10", 30, 3
    def evaluate_crisp(self, x):
        J1 = np.arange(3, len(x), 3)
        J2 = np.arange(1, len(x), 3)
        J3 = np.arange(2, len(x), 3)
        Y1 = x[J1] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J1 + 1) * np.pi / len(x))
        Y2 = x[J2] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J2 + 1) * np.pi / len(x))
        Y3 = x[J3] - 2 * x[1] * np.sin(2 * np.pi * x[0] + (J3 + 1) * np.pi / len(x))
        return np.array([
            np.cos(0.5 * np.pi * x[0]) * np.cos(0.5 * np.pi * x[1]) +
            (2.0 / len(J1)) * np.sum(4 * Y1 ** 2 - np.cos(8 * np.pi * Y1) + 1),
            np.cos(0.5 * np.pi * x[0]) * np.sin(0.5 * np.pi * x[1]) +
            (2.0 / len(J2)) * np.sum(4 * Y2 ** 2 - np.cos(8 * np.pi * Y2) + 1),
            np.sin(0.5 * np.pi * x[0]) +
            (2.0 / len(J3)) * np.sum(4 * Y3 ** 2 - np.cos(8 * np.pi * Y3) + 1)])


class FUF11(FuzzyBenchmark):
    name, n_var, n_obj = "FUF11", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FUF12(FuzzyBenchmark):
    name, n_var, n_obj = "FUF12", 30, 2
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j] ** 4)
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1] ** 4)
        return f


def _maf_dtlz_base(x, n_obj):
    m = n_obj
    g = np.sum((x[m - 1:] - 0.5) ** 2)
    f = np.zeros(m)
    for i in range(m):
        f[i] = (1 + g)
        for j in range(m - i - 1):
            f[i] *= np.cos(0.5 * np.pi * x[j])
        if i > 0:
            f[i] *= np.sin(0.5 * np.pi * x[m - i - 1])
    return f


class FMaF1(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF1", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m):
            f[i] = (1 + g)
            for j in range(m - i - 1):
                f[i] *= x[j]
            if i > 0:
                f[i] *= (1 - x[m - i - 1])
        return 1 - f


class FMaF2(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF2", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj)


class FMaF3(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF3", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF4(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF4", 12, 3
    def evaluate_crisp(self, x):
        return 1 - _maf_dtlz_base(x, self.n_obj)


class FMaF5(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF5", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF6(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF6", 12, 3
    def evaluate_crisp(self, x):
        m = self.n_obj
        g = np.sum((x[m - 1:] - 0.5) ** 2)
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF7(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF7", 22, 3
    def evaluate_crisp(self, x):
        n, m = len(x), self.n_obj
        g = 1 + 9.0 / (n - m + 1) * np.sum(x[m - 1:])
        f = np.zeros(m)
        for i in range(m - 1):
            f[i] = x[i]
        f[m - 1] = (1 + g) * (m - np.sum(
            f[:m - 1] / (1 + g) * (1 + np.sin(3 * np.pi * f[:m - 1]))))
        return f


class FMaF8(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF8", 2, 3
    def evaluate_crisp(self, x):
        x1, x2 = x[0], x[1]
        return np.array([(x1 - 2) ** 2 + (x2 - 1) ** 2,
                         x1 ** 2 + (x2 + 1) ** 2,
                         (x1 - 1) ** 2 + x2 ** 2])


class FMaF9(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF9", 12, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 4


class FMaF10(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF10", 12, 3
    def evaluate_crisp(self, x):
        return FWFG1().evaluate_crisp(x)


class FMaF11(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF11", 12, 3
    def evaluate_crisp(self, x):
        return FWFG2().evaluate_crisp(x)


class FMaF12(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF12", 12, 3
    def evaluate_crisp(self, x):
        return FWFG3().evaluate_crisp(x)


class FMaF13(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF13", 5, 3
    def evaluate_crisp(self, x):
        return _maf_dtlz_base(x, self.n_obj) ** 2


class FMaF14(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF14", 12, 3
    def evaluate_crisp(self, x):
        return FDTLZ1().evaluate_crisp(x)


class FMaF15(FuzzyBenchmark):
    name, n_var, n_obj = "FMaF15", 12, 3
    def evaluate_crisp(self, x):
        return 1 - FDTLZ1().evaluate_crisp(x)


class FScal(FuzzyBenchmark):
    name = "FScal"
    def __init__(self, n_var=500, n_obj=3):
        self.n_var = n_var
        self.n_obj = n_obj
    def evaluate_crisp(self, x):
        g = np.sum((x[self.n_obj - 1:] - 0.5) ** 2)
        f = np.zeros(self.n_obj)
        for i in range(self.n_obj):
            f[i] = (1 + g)
            for j in range(self.n_obj - i - 1):
                f[i] *= np.cos(0.5 * np.pi * x[j])
            if i > 0:
                f[i] *= np.sin(0.5 * np.pi * x[self.n_obj - i - 1])
        return f


class FScal5000(FuzzyBenchmark):
    name = "FScal5000"
    def __init__(self):
        self.n_var = 5000
        self.n_obj = 3
    def evaluate_crisp(self, x):
        return FScal(5000, 3).evaluate_crisp(x)


class FScal10000(FuzzyBenchmark):
    name = "FScal10000"
    def __init__(self):
        self.n_var = 10000
        self.n_obj = 3
    def evaluate_crisp(self, x):
        return FScal(10000, 3).evaluate_crisp(x)


class F2Base(FuzzyBenchmark):
    def __init__(self, base_cls=None, _base=None):
        if base_cls is None and _base is None:
            raise TypeError("F2Base requires either base_cls or _base")
        self.base = _base if _base is not None else base_cls()
        if getattr(type(self), "name", "base") == "base":
            self.name = "F2" + self.base.name
        self.n_var = self.base.n_var
        self.n_obj = self.base.n_obj

    def evaluate_crisp(self, x):
        return self.base.evaluate_crisp(x)

    def evaluate_fuzzy(self, x, delta=0.30):
        sigma = float(np.clip(x[0], 0.0, 1.0))
        eta = 0.15 + 0.30 * sigma
        crisp = self.evaluate_crisp(x)
        fuzzy = np.empty((len(crisp), 4))
        for i, val in enumerate(crisp):
            trfn = crisp_to_trapezoidal(val, delta=delta, eta=eta)
            fuzzy[i] = (trfn.a1, trfn.a2, trfn.a3, trfn.a4)
        return fuzzy


def _make_f2_subclass(base_cls):
    cls_name = f"F2{base_cls.__name__}"

    def _init(self, _b=base_cls, _name=cls_name):
        F2Base.__init__(self, _base=_b())
        self.name = _name

    return type(
        cls_name,
        (F2Base,),
        {
            "__init__": _init,
            "name": cls_name,
            "n_var": base_cls.n_var,
            "n_obj": base_cls.n_obj,
        },
    )


_F2_BASE_CLASSES = [FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
                    FDTLZ2, FDTLZ4, FDTLZ7,
                    FUF1, FUF2, FUF3, FUF4,
                    FMaF1, FMaF2, FMaF3]

F2_PROBLEMS = [_make_f2_subclass(cls) for cls in _F2_BASE_CLASSES]
F2_NAMES = [f"F2{cls.__name__}" for cls in _F2_BASE_CLASSES]


# ============================================================================
# PART 4: PROBLEM REGISTRY
# ============================================================================

ALL_PROBLEMS = [
    FZDT1, FZDT2, FZDT3, FZDT4, FZDT6,
    FDTLZ1, FDTLZ2, FDTLZ3, FDTLZ4, FDTLZ7,
    FWFG1, FWFG2, FWFG3, FWFG4, FWFG5, FWFG6, FWFG7, FWFG8, FWFG9,
    FUF1, FUF2, FUF3, FUF4, FUF5, FUF6, FUF7, FUF8, FUF9, FUF10, FUF11, FUF12,
    FMaF1, FMaF2, FMaF3, FMaF4, FMaF5, FMaF6, FMaF7, FMaF8, FMaF9,
    FMaF10, FMaF11, FMaF12, FMaF13, FMaF14, FMaF15,
    FScal, FScal5000, FScal10000,
] + F2_PROBLEMS

FAMILIES = {
    'ZDT': ['FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6'],
    'DTLZ': ['FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7'],
    'WFG': [f'FWFG{i}' for i in range(1, 10)],
    'CEC09': [f'FUF{i}' for i in range(1, 13)],
    'MaF': [f'FMaF{i}' for i in range(1, 16)],
    'Scalability': ['FScal', 'FScal5000', 'FScal10000'],
    'F2': F2_NAMES,
}

REGISTRY = {cls.name: cls for cls in ALL_PROBLEMS}

_missing_f2 = [n for n in F2_NAMES if n not in REGISTRY]
if _missing_f2:
    raise RuntimeError(
        f"F2 family not registered correctly. Missing from REGISTRY: "
        f"{_missing_f2}."
    )
del _missing_f2


PARETO_DEFAULT_PROBLEMS = [
    'FZDT1', 'FZDT2', 'FZDT3', 'FZDT4', 'FZDT6',
    'FDTLZ1', 'FDTLZ2', 'FDTLZ3', 'FDTLZ4', 'FDTLZ7',
    'FWFG1', 'FWFG2', 'FWFG3', 'FWFG4', 'FWFG5', 'FWFG6', 'FWFG7',
    'FWFG8', 'FWFG9',
    'FUF1', 'FUF2', 'FUF3', 'FUF4', 'FUF5', 'FUF6', 'FUF7', 'FUF8',
    'FUF9', 'FUF10', 'FUF11', 'FUF12',
    'FMaF1', 'FMaF2', 'FMaF3', 'FMaF4', 'FMaF5', 'FMaF6', 'FMaF7',
    'FMaF8', 'FMaF9', 'FMaF10', 'FMaF11', 'FMaF12', 'FMaF13', 'FMaF14',
    'FMaF15',
    'FScal', 'FScal5000', 'FScal10000',
] + F2_NAMES


def get_extended_benchmark(name):
    if name in REGISTRY:
        return REGISTRY[name]()
    if name.startswith("FScal"):
        return FScal(n_var=int(name.replace("FScal", "")))
    raise ValueError(f"Unknown benchmark: {name}")


def get_all_problems():
    problems = []
    for probs in FAMILIES.values():
        problems.extend(probs)
    return problems


# ============================================================================
# PART 5: INDIVIDUAL
# ============================================================================

@dataclass
class Individual:
    x: np.ndarray
    alpha: float
    fuzzy_obj: np.ndarray = field(default=None)
    rank: int = 0
    crowding: float = 0.0
    _weighted: np.ndarray = field(default=None, repr=False)

    def evaluate(self, benchmark, delta=0.30):
        fuzzy = benchmark.evaluate_fuzzy(self.x, delta=delta)
        M = fuzzy.shape[0]
        self.fuzzy_obj = np.empty((M, 3))
        for i in range(M):
            a1, a2, a3, a4 = fuzzy[i]
            aL = min(a1, a2, a3, a4)
            aU = max(a1, a2, a3, a4)
            aM = 0.5 * (a2 + a3)
            if aM < aL:
                aM = aL
            if aM > aU:
                aM = aU
            self.fuzzy_obj[i] = (aL, aM, aU)
        self._weighted = None

    def alpha_weighted_obj(self):
        if self._weighted is None:
            L = self.fuzzy_obj[:, 0]
            M = self.fuzzy_obj[:, 1]
            U = self.fuzzy_obj[:, 2]
            self._weighted = self.alpha * M + (1 - self.alpha) * (L + U) / 2.0
        return self._weighted


# ============================================================================
# PART 5A/5B/5C: ALGORITHMS
# ============================================================================

class AAlphaNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 p_local_search=0.3, alpha_bounds=(0.5, 1.0),
                 alpha_diversity_freq=10, delta=0.30, seed=42,
                 track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation if p_mutation is not None else 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.p_local_search = p_local_search
        self.alpha_L, self.alpha_U = alpha_bounds
        self.alpha_diversity_freq = alpha_diversity_freq
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
            ind = Individual(x=x, alpha=alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def blx_alpha_crossover(self, a1, a2):
        gamma = self.rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1), self.alpha_L, self.alpha_U))

    def polynomial_mutation(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def gaussian_alpha_mutation(self, alpha, gen):
        sigma = 0.05 * (1 - gen / self.n_gen)
        return float(np.clip(alpha + self.rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))

    def _fuzzy_dominates(self, ind1, ind2):
        return _tfn_dominates_arrays(ind1.fuzzy_obj, ind2.fuzzy_obj)

    def local_search(self, ind, gen):
        if self.rng.random() > self.p_local_search:
            return ind
        if self.rng.random() < 0.2:
            x_new = np.clip(ind.x + self.rng.normal(0, 0.05, size=len(ind.x)),
                            self.xl, self.xu)
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        if self.rng.random() < 0.2:
            for da in (-0.05, -0.02, 0.02, 0.05):
                a_new = float(np.clip(ind.alpha + da, self.alpha_L, self.alpha_U))
                cand = Individual(x=ind.x.copy(), alpha=a_new)
                cand.evaluate(self.benchmark, self.delta)
                if self._fuzzy_dominates(cand, ind):
                    ind = cand
                    break
        if self.rng.random() < 0.2:
            x_new = ind.x.copy()
            j = int(self.rng.integers(len(x_new)))
            x_new[j] = self.rng.uniform(self.xl[j], self.xu[j])
            cand = Individual(x=x_new, alpha=ind.alpha)
            cand.evaluate(self.benchmark, self.delta)
            if self._fuzzy_dominates(cand, ind):
                ind = cand
        return ind

    def alpha_diversity_maintenance(self):
        alphas = np.array([ind.alpha for ind in self.population])
        bins = np.linspace(self.alpha_L, self.alpha_U, 6)
        counts, _ = np.histogram(alphas, bins=bins)
        for i in range(len(counts)):
            if counts[i] > 0.5 * self.pop_size:
                mask = (alphas >= bins[i]) & (alphas < bins[i + 1])
                idx = np.where(mask)[0]
                n_reset = len(idx) // 2
                for j in idx[:n_reset]:
                    self.population[j].alpha = self.rng.uniform(self.alpha_L, self.alpha_U)
                    self.population[j]._weighted = None
                    self.population[j].evaluate(self.benchmark, self.delta)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class AAlphaNSGA2AdaptiveMutation(AAlphaNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(1, self.n_gen + 1):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c2_a = self.blx_alpha_crossover(p1.alpha, p2.alpha)
                c1_x = self.polynomial_mutation(c1_x, p_mutation=self.mutation_rate)
                c2_x = self.polynomial_mutation(c2_x, p_mutation=self.mutation_rate)
                c1_a = self.gaussian_alpha_mutation(c1_a, gen)
                c2_a = self.gaussian_alpha_mutation(c2_a, gen)
                c1 = Individual(x=c1_x, alpha=c1_a)
                c2 = Individual(x=c2_x, alpha=c2_a)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                c1 = self.local_search(c1, gen)
                c2 = self.local_search(c2, gen)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen % self.alpha_diversity_freq == 0:
                self.alpha_diversity_maintenance()
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class BaselineNSGA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=0.75, delta=0.30, seed=42, track_history=False):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.population = []

    def initialize(self):
        self.population = []
        for _ in range(self.pop_size):
            x = self.rng.uniform(self.xl, self.xu)
            ind = Individual(x=x, alpha=self.alpha)
            ind.evaluate(self.benchmark, self.delta)
            self.population.append(ind)

    def tournament_selection(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx_crossover(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def polynomial_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def assign_rank_and_crowding(self, population):
        F = np.array([ind.fuzzy_obj for ind in population])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                population[idx].rank = r
            dist = fuzzy_crowding_distance(F, front)
            for k, idx in enumerate(front):
                population[idx].crowding = dist[k]

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self.polynomial_mutation(c1_x)
                c2_x = self.polynomial_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class BaselineNSGA2HighMutation(BaselineNSGA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_mutation = 0.05


class BaselineSelfAdaptiveMutation(BaselineNSGA2):
    def initialize(self):
        super().initialize()
        self.mutation_rate = 0.05

    def _adaptive_mutation(self, x):
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < self.mutation_rate
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.assign_rank_and_crowding(self.population)
        for gen in range(self.n_gen):
            offspring = []
            gen_success, gen_total = 0, 0
            while len(offspring) < self.pop_size:
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                c1_x, c2_x = self.sbx_crossover(p1, p2)
                c1_x = self._adaptive_mutation(c1_x)
                c2_x = self._adaptive_mutation(c2_x)
                c1 = Individual(x=c1_x, alpha=self.alpha)
                c2 = Individual(x=c2_x, alpha=self.alpha)
                c1.evaluate(self.benchmark, self.delta)
                c2.evaluate(self.benchmark, self.delta)
                if c1.rank <= p1.rank and c1.rank <= p2.rank:
                    gen_success += 1
                if c2.rank <= p1.rank and c2.rank <= p2.rank:
                    gen_success += 1
                gen_total += 2
                offspring.extend([c1, c2])
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.assign_rank_and_crowding(combined)
            combined.sort(key=lambda ind: (ind.rank, -ind.crowding))
            self.population = combined[:self.pop_size]
            if gen_total > 0:
                sr = gen_success / gen_total
                self.mutation_rate *= 1.1 if sr > 0.2 else 0.9
                self.mutation_rate = float(np.clip(self.mutation_rate, 0.001, 0.5))
        return self.population


class SPEA2:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 archive_size=None, k_nn=SPEA2_KNN_K):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.archive_size = archive_size or pop_size
        self.k_nn = k_nn
        self.population = []
        self.archive = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def _spea2_dominance_matrix(self, F_all):
        L = F_all[:, :, 0]
        U = F_all[:, :, 2]
        W = U - L
        num = U[None, :, :] - L[:, None, :]
        den = W[:, None, :] + W[None, :, :]
        safe_den = np.where(den > EPS, den, 1.0)
        with np.errstate(divide='ignore', invalid='ignore'):
            cred = np.clip(num / safe_den, 0.0, 1.0)
        cred = np.where(den <= EPS,
                        (U[:, None, :] <= L[None, :, :]).astype(float),
                        cred)
        weak_all = np.all(cred >= 0.5, axis=2)
        strict_any = np.any(L[:, None, :] > U[None, :, :], axis=2)
        D = weak_all & strict_any
        np.fill_diagonal(D, False)
        return D

    def _spea2_fitness(self, combined):
        n = len(combined)
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        D = self._spea2_dominance_matrix(F_all)
        strength = D.sum(axis=1)
        raw = (D.T @ strength).astype(int)
        W = np.array([ind.alpha_weighted_obj() for ind in combined])
        dist = np.sqrt(((W[:, None, :] - W[None, :, :]) ** 2).sum(axis=2))
        np.fill_diagonal(dist, np.inf)
        k = min(self.k_nn, n - 1)
        sorted_d = np.sort(dist, axis=1)
        density = 1.0 / (sorted_d[:, k - 1] + 2.0) if k > 0 else np.zeros(n)
        fitness = raw + density
        return fitness

    def _environmental_selection(self, combined):
        fitness = self._spea2_fitness(combined)
        order = np.argsort(fitness)
        return [combined[i] for i in order[:self.archive_size]]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r

        self.population = self.archive
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class MOEAD:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 neighbourhood_T=MOEAD_NEIGHBORHOOD_T,
                 n_neighbors=None, divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.T = neighbourhood_T

        M = benchmark.n_obj
        H = (divisions or MOEAD_DIVISIONS).get(M, 6)
        weights = das_dennis_points(M, H)
        if len(weights) >= pop_size:
            self.weights = weights[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(weights)))
            self.weights = np.tile(weights, (reps, 1))[:pop_size]
        self.n_neighbors = n_neighbors or min(self.T, pop_size - 1)

        D = np.sqrt(((self.weights[:, None, :] - self.weights[None, :, :]) ** 2).sum(axis=2))
        self.neighbors = np.argsort(D, axis=1)[:, :self.n_neighbors]

        self.population = []
        self.ideal = np.full(M, np.inf)

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _scalarized(self, ind, weight):
        F = ind.alpha_weighted_obj()
        return float(np.max(weight * np.abs(F - self.ideal)))

    def _update_ideal(self, ind):
        F = ind.alpha_weighted_obj()
        self.ideal = np.minimum(self.ideal, F)

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1 = self._make(c1x)
                c2 = self._make(c2x)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class NSGA3:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 divisions=None):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)

        M = benchmark.n_obj
        H = (divisions or NSGA3_DIVISIONS).get(M, 6)
        ref = das_dennis_points(M, H)
        if len(ref) >= pop_size:
            self.ref_points = ref[:pop_size]
        else:
            reps = int(np.ceil(pop_size / len(ref)))
            self.ref_points = np.tile(ref, (reps, 1))[:pop_size]
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def tournament(self, k=2):
        cand = self.rng.choice(self.population, size=k, replace=False)
        return min(cand, key=lambda ind: (ind.rank, -ind.crowding))

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _niche_selection(self, combined, n_select):
        if n_select <= 0:
            return []
        F_all = np.array([ind.fuzzy_obj for ind in combined])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                combined[idx].rank = r
        norms = np.linalg.norm(self.ref_points, axis=1)
        norms = np.where(norms < EPS, 1.0, norms)
        ref_unit = self.ref_points / norms[:, None]
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= n_select:
                selected.extend(combined[i] for i in front)
            else:
                remaining = n_select - len(selected)
                W = np.array([combined[i].alpha_weighted_obj() for i in front])
                proj = W @ ref_unit.T
                dist_perp = np.sqrt(np.clip(
                    (W ** 2).sum(axis=1)[:, None] - proj ** 2, 0.0, None))
                nearest = np.argmin(dist_perp, axis=1)
                niche_count = np.zeros(len(self.ref_points), dtype=int)
                if len(selected) > 0:
                    W_sel = np.array([ind.alpha_weighted_obj() for ind in selected])
                    proj_sel = W_sel @ ref_unit.T
                    dist_sel = np.sqrt(np.clip(
                        (W_sel ** 2).sum(axis=1)[:, None] - proj_sel ** 2,
                        0.0, None))
                    nearest_sel = np.argmin(dist_sel, axis=1)
                    for k in nearest_sel:
                        niche_count[k] += 1
                order = np.argsort([niche_count[nearest[k]]
                                    for k in range(len(front))])
                chosen = [combined[front[k]] for k in order[:remaining]]
                selected.extend(chosen)
                break
        return selected

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class cMLSGA:
    def __init__(self, benchmark, pop_size=60, n_gen=300,
                 p_crossover=0.9, eta_crossover=20.0,
                 p_mutation=None, eta_mutation=20.0,
                 alpha=FAIR_ALPHA, delta=0.30, seed=42,
                 n_subpops=CMLSGA_N_SUBPOPS,
                 migration_freq=CMLSGA_MIGRATION_FREQ):
        self.benchmark = benchmark
        self.pop_size = pop_size
        self.n_gen = n_gen
        self.p_crossover = p_crossover
        self.eta_crossover = eta_crossover
        self.p_mutation = p_mutation or 1.0 / benchmark.n_var
        self.eta_mutation = eta_mutation
        self.alpha = alpha
        self.delta = delta
        self.rng = np.random.default_rng(seed)
        self.xl = np.zeros(benchmark.n_var)
        self.xu = np.ones(benchmark.n_var)
        self.n_subpops = max(1, n_subpops)
        self.migration_freq = migration_freq
        self.deme_size = max(2, pop_size // self.n_subpops)
        self.population = []

    def _make(self, x):
        ind = Individual(x=x, alpha=self.alpha)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def initialize(self):
        self.demes = [self._make_deme() for _ in range(self.n_subpops)]

    def tournament(self, pool, k=2):
        cand = self.rng.choice(pool, size=min(k, len(pool)), replace=False)
        return min(cand, key=lambda ind: ind.rank)

    def sbx(self, p1, p2):
        if self.rng.random() > self.p_crossover:
            return p1.x.copy(), p2.x.copy()
        u = self.rng.random(self.benchmark.n_var)
        beta = np.where(u <= 0.5,
                        (2 * u) ** (1.0 / (self.eta_crossover + 1)),
                        (1.0 / (2 * (1 - u + EPS))) ** (1.0 / (self.eta_crossover + 1)))
        c1 = 0.5 * ((1 + beta) * p1.x + (1 - beta) * p2.x)
        c2 = 0.5 * ((1 - beta) * p1.x + (1 + beta) * p2.x)
        return np.clip(c1, self.xl, self.xu), np.clip(c2, self.xl, self.xu)

    def poly_mut(self, x, p_mutation=None):
        if p_mutation is None:
            p_mutation = self.p_mutation
        y = x.copy()
        mask = self.rng.random(y.shape[0]) < p_mutation
        if not mask.any():
            return y
        u = self.rng.random(int(mask.sum()))
        delta = np.where(u < 0.5,
                         (2 * u) ** (1.0 / (self.eta_mutation + 1)) - 1.0,
                         1.0 - (2 * (1 - u + EPS)) ** (1.0 / (self.eta_mutation + 1)))
        y[mask] += delta * (self.xu[mask] - self.xl[mask])
        return np.clip(y, self.xl, self.xu)

    def _rank_deme(self, deme):
        F = np.array([ind.fuzzy_obj for ind in deme])
        fronts = fast_nondominated_sort_fuzzy(F)
        for r, front in enumerate(fronts):
            for idx in front:
                deme[idx].rank = r

    def _evolve_deme(self, deme, n_gen):
        for _ in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                offspring.append(self._make(c1x))
                offspring.append(self._make(c2x))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def _migrate(self):
        bests = [min(d, key=lambda ind: ind.rank) for d in self.demes]
        for i, deme in enumerate(self.demes):
            j = (i + 1) % self.n_subpops
            k = int(self.rng.integers(len(self.demes[j])))
            self.demes[j][k] = bests[i]

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(
                    self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population

    def get_pareto_front(self):
        return [ind for ind in self.population if ind.rank == 0]

    def get_alphas(self):
        return np.array([ind.alpha for ind in self.population])


class _AlphaMixin:
    def _init_alpha(self):
        self.alpha_L = 0.5
        self.alpha_U = 1.0
        self._alpha_rng = self.rng

    def _random_alpha(self):
        return float(self._alpha_rng.uniform(self.alpha_L, self.alpha_U))

    def _blx_alpha(self, a1, a2):
        gamma = self._alpha_rng.uniform(-0.5, 1.5)
        return float(np.clip(a1 + gamma * (a2 - a1),
                             self.alpha_L, self.alpha_U))

    def _gauss_alpha(self, alpha, gen):
        sigma = 0.05 * max(0.0, 1.0 - gen / max(1, self.n_gen))
        return float(np.clip(alpha + self._alpha_rng.normal(0, sigma),
                             self.alpha_L, self.alpha_U))


class AlphaWrappedSPEA2(_AlphaMixin, SPEA2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        self.archive = []

    def run(self):
        self.initialize()
        self.archive = self._environmental_selection(self.population)
        F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
        fronts = fast_nondominated_sort_fuzzy(F_arch)
        for r, front in enumerate(fronts):
            for idx in front:
                self.archive[idx].rank = r

        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(self.archive)
                p2 = self.tournament(self.archive)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._blx_alpha(p1.alpha, p2.alpha)
                c2a = self._blx_alpha(p1.alpha, p2.alpha)
                c1a = self._gauss_alpha(c1a, gen)
                c2a = self._gauss_alpha(c2a, gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.archive + offspring
            self.archive = self._environmental_selection(combined)
            F_arch = np.array([ind.fuzzy_obj for ind in self.archive])
            fronts = fast_nondominated_sort_fuzzy(F_arch)
            for r, front in enumerate(fronts):
                for idx in front:
                    self.archive[idx].rank = r
        self.population = self.archive
        return self.population


class AlphaWrappedMOEAD(_AlphaMixin, MOEAD):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]
        for ind in self.population:
            self._update_ideal(ind)

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            for i in range(self.pop_size):
                nbrs = self.neighbors[i]
                k1, k2 = self.rng.choice(nbrs, size=2, replace=False)
                p1, p2 = self.population[k1], self.population[k2]
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c1 = self._make(c1x, c1a)
                c2 = self._make(c2x, c2a)
                for child in (c1, c2):
                    self._update_ideal(child)
                    for j in nbrs:
                        w = self.weights[j]
                        if (self._scalarized(child, w) <
                                self._scalarized(self.population[j], w)):
                            self.population[j] = child
        F_all = np.array([ind.fuzzy_obj for ind in self.population])
        fronts = fast_nondominated_sort_fuzzy(F_all)
        for r, front in enumerate(fronts):
            for idx in front:
                self.population[idx].rank = r
        return self.population


class AlphaWrappedNSGA3(_AlphaMixin, NSGA3):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def initialize(self):
        self.population = [
            self._make(self.rng.uniform(self.xl, self.xu))
            for _ in range(self.pop_size)
        ]

    def run(self):
        self.initialize()
        for gen in range(self.n_gen):
            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament()
                p2 = self.tournament()
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.pop_size]
            combined = self.population + offspring
            self.population = self._niche_selection(combined, self.pop_size)
        return self.population


class AlphaWrappedcMLSGA(_AlphaMixin, cMLSGA):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._init_alpha()
        self.alpha = None

    def _make(self, x, alpha=None):
        a = self._random_alpha() if alpha is None else alpha
        ind = Individual(x=x, alpha=a)
        ind.evaluate(self.benchmark, self.delta)
        return ind

    def _make_deme(self):
        return [self._make(self.rng.uniform(self.xl, self.xu))
                for _ in range(self.deme_size)]

    def _evolve_deme(self, deme, n_gen):
        for gen in range(n_gen):
            offspring = []
            while len(offspring) < self.deme_size:
                p1 = self.tournament(deme)
                p2 = self.tournament(deme)
                c1x, c2x = self.sbx(p1, p2)
                c1x = self.poly_mut(c1x)
                c2x = self.poly_mut(c2x)
                c1a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                c2a = self._gauss_alpha(self._blx_alpha(p1.alpha, p2.alpha), gen)
                offspring.append(self._make(c1x, c1a))
                offspring.append(self._make(c2x, c2a))
            offspring = offspring[:self.deme_size]
            combined = deme + offspring
            self._rank_deme(combined)
            combined.sort(key=lambda ind: ind.rank)
            deme[:] = combined[:self.deme_size]
        return deme

    def run(self):
        self.initialize()
        generations_per_migration = max(1, self.migration_freq)
        n_migrations = max(1, self.n_gen // generations_per_migration)
        gens_per_round = max(1, self.n_gen // n_migrations)
        for _ in range(n_migrations):
            for i in range(self.n_subpops):
                self.demes[i] = self._evolve_deme(self.demes[i], gens_per_round)
            self._migrate()
        self.population = [ind for deme in self.demes for ind in deme]
        self._rank_deme(self.population)
        return self.population


# ============================================================================
# PART 6: METRICS
# ============================================================================

def hypervolume(F, ref):
    if len(F) == 0:
        return 0.0
    mask = np.all(F <= ref, axis=1)
    F = F[mask]
    if len(F) == 0:
        return 0.0
    if F.shape[1] == 2:
        return _hv_2d(F, ref)
    return _hv_mc(F, ref, n_samples=3000)


def _hv_2d(F, ref):
    idx = np.argsort(F[:, 0])
    F = F[idx]
    hv = 0.0
    prev_y = ref[1]
    for x, y in F:
        if y < prev_y:
            hv += (ref[0] - x) * (prev_y - y)
            prev_y = y
    return float(hv)


_MC_RNG = np.random.default_rng(0)


def _hv_mc(F, ref, n_samples=3000):
    lows = np.min(F, axis=0)
    samples = _MC_RNG.uniform(lows, ref, size=(n_samples, F.shape[1]))
    dominated = np.zeros(n_samples, dtype=bool)
    for f in F:
        dominated |= np.all(samples >= f, axis=1)
    return float(np.mean(dominated) * np.prod(ref - lows))


def _fixed_alpha_objectives(population, alpha=FAIR_ALPHA, force=False):
    pf = [ind for ind in population if ind.rank == 0]
    if not pf:
        return np.empty((0, 0))
    out = np.empty((len(pf), pf[0].fuzzy_obj.shape[0]))
    for i, ind in enumerate(pf):
        if force:
            a = float(alpha)
        elif ind.alpha is None:
            a = float(alpha)
        else:
            a = float(ind.alpha)
        out[i] = a * ind.fuzzy_obj[:, 1] + (1 - a) * (ind.fuzzy_obj[:, 0] + ind.fuzzy_obj[:, 2]) / 2.0
    return out


def compute_pareto_hv(population, fixed_alpha=FAIR_ALPHA,
                      alpha_mode='per_individual'):
    force = (alpha_mode == 'fixed')
    F = _fixed_alpha_objectives(population, alpha=fixed_alpha, force=force)
    if F.size == 0:
        return 0.0, np.array([])
    ref = np.max(F, axis=0) * 1.1 + 1.0
    return hypervolume(F, ref), F


def _igd_gd_against_reference(F, ref_front):
    F = np.asarray(F, dtype=float)
    ref_front = np.asarray(ref_front, dtype=float)
    if F.size == 0 or ref_front.size == 0:
        return float('nan'), float('nan')
    D = np.sqrt(((ref_front[:, None, :] - F[None, :, :]) ** 2).sum(axis=2))
    igd = float(np.mean(np.min(D, axis=1)))
    gd = float(np.mean(np.min(D, axis=0)))
    return igd, gd


def _ref_front_diagonal(ref_front):
    ref_front = np.asarray(ref_front, dtype=float)
    if ref_front.size == 0:
        return 1.0
    span = ref_front.max(axis=0) - ref_front.min(axis=0)
    diag = float(np.sqrt((span ** 2).sum()))
    return diag if diag > EPS else 1.0


def build_empirical_reference_front(all_fronts,
                                    k=REF_FRONT_MAX_OUTPUT,
                                    seed=0,
                                    max_sort_size=REF_FRONT_MAX_INPUT):
    if not all_fronts:
        return np.empty((0, 0))
    stacked = np.vstack([F for F in all_fronts if F.size > 0])
    if stacked.size == 0:
        return np.empty((0, 0))
    if len(stacked) > max_sort_size:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(stacked), size=max_sort_size, replace=False)
        stacked = stacked[idx]
    uniq = np.unique(np.round(stacked, decimals=10), axis=0)
    M = uniq.shape[1]
    F_tfn = np.stack([uniq, uniq, uniq], axis=2)
    fronts = fast_nondominated_sort_fuzzy(F_tfn, chunk_size=256)
    front0 = fronts[0] if fronts else []
    nd = uniq[front0] if len(front0) > 0 else uniq
    if len(nd) > k:
        order = np.argsort(nd[:, 0])
        idx = np.linspace(0, len(nd) - 1, k).astype(int)
        nd = nd[order][idx]
    return nd


# ============================================================================
# PART 6B: Σ̂ PROXY
# ============================================================================

def compute_sigma_hat(prob_name, alphas=SIGMA_HAT_ALPHAS,
                      seed=2026, pop_size=60, n_gen=300,
                      n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA,
                      ref_quantile=SIGMA_HAT_REF_QUANTILE):
    """v8.8 fix: uses the SAME population to evaluate all three α values."""
    benchmark = get_extended_benchmark(prob_name)
    algo = AAlphaNSGA2AdaptiveMutation

    per_alpha_fronts = {a: [] for a in alphas}
    rng = np.random.default_rng(seed)

    for r in range(n_runs_per_alpha):
        s = int(rng.integers(0, 2**31 - 1))
        algo_obj = algo(benchmark, pop_size=pop_size, n_gen=n_gen, seed=s)
        pop = algo_obj.run()
        for a in alphas:
            F = _fixed_alpha_objectives(pop, alpha=a, force=True)
            if F.size > 0:
                per_alpha_fronts[a].append(F)

    all_F = [F for a in alphas for F in per_alpha_fronts[a]]
    if not all_F:
        return float('nan'), [float('nan')] * len(alphas)
    stacked = np.vstack(all_F)
    ref = np.quantile(stacked, ref_quantile, axis=0) * 1.05 + 1e-6

    mean_hvs = []
    for a in alphas:
        hvs_a = []
        for F in per_alpha_fronts[a]:
            mask = np.all(F <= ref, axis=1)
            F_clipped = F[mask]
            hvs_a.append(hypervolume(F_clipped, ref)
                         if F_clipped.size > 0 else 0.0)
        mean_hvs.append(float(np.mean(hvs_a)) if hvs_a else 0.0)

    mean_hv = float(np.mean(mean_hvs))
    sigma = (max(mean_hvs) - min(mean_hvs)) / mean_hv if mean_hv > EPS else 0.0
    return float(sigma), mean_hvs


def _probe_f2_override():
    print("\n  [probe] F2 modulation check:")
    x_lo = np.array([0.0] + [0.5] * 29)
    x_hi = np.array([1.0] + [0.5] * 29)
    for name, xs in (('FZDT1', x_lo), ('F2FZDT1 @x0=0', x_lo),
                     ('F2FZDT1 @x0=1', x_hi)):
        b = get_extended_benchmark(name if name == 'FZDT1' else 'F2FZDT1')
        fz = b.evaluate_fuzzy(xs, delta=0.30)
        widths = fz[:, 3] - fz[:, 0]
        print(f"    {name:<18} f1 TFN=[{fz[0,0]:+.3f},{fz[0,1]:+.3f},"
              f"{fz[0,2]:+.3f},{fz[0,3]:+.3f}]  width={widths[0]:.3f}")


def verify_sigma_hat_families(sigma_dir):
    csv_path = os.path.join(sigma_dir, 'sigma_hat.csv')
    if not os.path.exists(csv_path):
        return
    try:
        df = pd.read_csv(csv_path)
        f1 = df[df['family'] == 'F1']['sigma_hat'].values
        f2 = df[df['family'] == 'F2']['sigma_hat'].values
        if len(f1) and len(f2):
            m1, m2 = float(np.mean(f1)), float(np.mean(f2))
            ratio = m2 / m1 if m1 > EPS else float('inf')
            print(f"\n  Σ̂ sanity: mean F1 = {m1:.4f}  mean F2 = {m2:.4f}  "
                  f"ratio = {ratio:.2f}")
            if m1 > 0.015:
                print(f"  ⚠ NOTE: F1 mean Σ̂ = {m1:.4f} is not ≈ 0. "
                      f"This is expected because F1's trapezoidal geometry "
                      f"has centroid ≠ midpoint, so α-weighting still "
                      f"changes HV even for a fixed population.")
            if ratio < 1.5:
                print(f"  ⚠ WARNING: F2/F1 Σ̂ ratio = {ratio:.2f} is "
                      f"too small. F2 modulation may not be active. "
                      f"Check the [probe] output at startup.")
            else:
                print(f"  ✓ F2/F1 Σ̂ ratio = {ratio:.2f} — F2 is "
                      f"correctly more α-sensitive than F1.")
    except Exception as e:
        print(f"  Σ̂ sanity check failed: {e}")


# ============================================================================
# PART 7: CONFIG
# ============================================================================

CONFIG = {
    'problems': get_all_problems(),
    'families': FAMILIES,
    'n_runs': 10,
    'pop_size': 60,
    'n_gen': 300,
    'seed_base': 2026,
    'n_workers': None,
    'use_parallel': True,
    'quiet': False,
    'results_dir':       os.path.join(OUTPUT_ROOT, 'results'),
    'figures_dir':       os.path.join(OUTPUT_ROOT, 'figures'),
    'latex_dir':         os.path.join(OUTPUT_ROOT, 'latex'),
    'supp_latex_dir':    os.path.join(OUTPUT_ROOT, 'supplementary_latex'),
    'sigma_hat_dir':     os.path.join(OUTPUT_ROOT, 'sigma_hat'),
    'external_dir':      os.path.join(OUTPUT_ROOT, 'external'),
    'reference_algo':    'Aα-NSGA-II (adaptive mutation)',
    'bootstrap_samples': 5000,
    'sigma_hat_seed':    2026,
    'sigma_hat_problems': F2_NAMES
                          + ['FZDT1', 'FZDT2', 'FDTLZ2',
                             'FUF1', 'FUF2', 'FMaF1', 'FMaF2'],
}

INTERNAL_ALGO_NAMES = [
    'Aα-NSGA-II (adaptive mutation)',
    'Aα-NSGA-II',
    'NSGA-II',
    'NSGA-II (high mutation)',
    'Self-Adaptive Mutation',
]

EXTERNAL_ALGO_NAMES = [
    'SPEA2+α',
    'MOEA/D+α',
    'NSGA-III+α',
    'cMLSGA+α',
]

EXTERNAL_ALGO_NAMES_STRICT = [
    'SPEA2',
    'MOEA/D',
    'NSGA-III',
    'cMLSGA',
]

ALGO_NAMES = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT

# ============================================================================
# HIGH-CONTRAST COLOR & MARKER SETTINGS
# ============================================================================
ALGO_COLORS = {
    # Hero algorithm for strict comparison
    'Aα-NSGA-II (adaptive mutation)': '#1f77b4',  # Strong Blue
    
    # Strict competitors (C1-C4)
    'SPEA2': '#d62728',                           # Strong Red
    'MOEA/D': '#2ca02c',                          # Strong Green
    'NSGA-III': '#ff7f0e',                        # Strong Orange
    'cMLSGA': '#9467bd',                          # Strong Purple
    
    # Ablation algorithms
    'Aα-NSGA-II': '#8c564b',
    'NSGA-II': '#e377c2',
    'NSGA-II (high mutation)': '#7f7f7f',
    'Self-Adaptive Mutation': '#bcbd22',
    
    # Fair (α-wrapped) algorithms
    'SPEA2+α': '#17becf',
    'MOEA/D+α': '#aec7e8',
    'NSGA-III+α': '#ffbb78',
    'cMLSGA+α': '#98df8a',
}

ALGO_MARKERS = {
    'Aα-NSGA-II (adaptive mutation)': 'o',
    'SPEA2': 's',
    'MOEA/D': '^',
    'NSGA-III': 'D',
    'cMLSGA': 'v',
    'Aα-NSGA-II': 'p',
    'NSGA-II': 'X',
    'NSGA-II (high mutation)': '*',
    'Self-Adaptive Mutation': 'h',
    'SPEA2+α': 'o',
    'MOEA/D+α': 's',
    'NSGA-III+α': '^',
    'cMLSGA+α': 'D',
}


def get_algo_config(pop_size, n_gen, external_mode=None):
    if external_mode is None:
        external_mode = EXTERNAL_COMPARISON_MODE

    common = {'pop_size': pop_size, 'n_gen': n_gen}
    algos = [
        ('Aα-NSGA-II (adaptive mutation)',
         AAlphaNSGA2AdaptiveMutation, {**common}),
        ('Aα-NSGA-II',
         AAlphaNSGA2, {**common}),
        ('NSGA-II',
         BaselineNSGA2, {**common, 'alpha': FAIR_ALPHA}),
        ('NSGA-II (high mutation)',
         BaselineNSGA2HighMutation, {**common, 'alpha': FAIR_ALPHA}),
        ('Self-Adaptive Mutation',
         BaselineSelfAdaptiveMutation, {**common, 'alpha': FAIR_ALPHA}),
    ]

    if external_mode in ('fair', 'both'):
        algos.extend([
            ('SPEA2+α',
             AlphaWrappedSPEA2, {**common,
                                 'archive_size': SPEA2_ARCHIVE_SIZE,
                                 'k_nn': SPEA2_KNN_K}),
            ('MOEA/D+α',
             AlphaWrappedMOEAD, {**common,
                                 'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                                 'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III+α',
             AlphaWrappedNSGA3, {**common,
                                 'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA+α',
             AlphaWrappedcMLSGA, {**common,
                                  'n_subpops': CMLSGA_N_SUBPOPS,
                                  'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    if external_mode in ('strict', 'both'):
        algos.extend([
            ('SPEA2',
             SPEA2, {**common, 'alpha': FAIR_ALPHA,
                     'archive_size': SPEA2_ARCHIVE_SIZE,
                     'k_nn': SPEA2_KNN_K}),
            ('MOEA/D',
             MOEAD, {**common, 'alpha': FAIR_ALPHA,
                     'neighbourhood_T': MOEAD_NEIGHBORHOOD_T,
                     'divisions': MOEAD_DIVISIONS}),
            ('NSGA-III',
             NSGA3, {**common, 'alpha': FAIR_ALPHA,
                     'divisions': NSGA3_DIVISIONS}),
            ('cMLSGA',
             cMLSGA, {**common, 'alpha': FAIR_ALPHA,
                      'n_subpops': CMLSGA_N_SUBPOPS,
                      'migration_freq': CMLSGA_MIGRATION_FREQ}),
        ])

    return algos


# ============================================================================
# PART 8: WORKER + IGD/GD PASS
# ============================================================================

def _worker_init():
    try:
        os.environ.setdefault('OMP_NUM_THREADS', '1')
        os.environ.setdefault('MKL_NUM_THREADS', '1')
        os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
    except Exception:
        pass


def _save_checkpoint(results, path):
    if path is None:
        return
    try:
        tmp = path + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(results, f)
        os.replace(tmp, path)
    except Exception:
        pass


def run_single_problem(args):
    prob_name, n_runs, pop_size, n_gen, seed_base = args
    try:
        benchmark = get_extended_benchmark(prob_name)
        algos = get_algo_config(pop_size, n_gen)
        results = {}
        for algo_name, algo_cls, kwargs in algos:
            hvs_raw, alphas = [], []
            errors = 0
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    errors += 1
                    continue
                hv_raw, F = compute_pareto_hv(pop)
                if len(F) == 0:
                    continue
                hvs_raw.append(hv_raw)
                if hasattr(algo, 'get_alphas'):
                    try:
                        alphas.append(float(np.mean(algo.get_alphas())))
                    except Exception:
                        pass
            results[algo_name] = {
                'hv_raw_mean': float(np.mean(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_std': float(np.std(hvs_raw)) if hvs_raw else 0.0,
                'hv_raw_all': hvs_raw,
                'alpha_mean': float(np.mean(alphas)) if alphas else np.nan,
                'alpha_mean_all': alphas,
                'n_errors': errors,
                'igd_mean': np.nan, 'igd_std': np.nan, 'igd_all': [],
                'gd_mean': np.nan, 'gd_std': np.nan, 'gd_all': [],
                'igd_norm_mean': np.nan, 'gd_norm_mean': np.nan,
            }
        return (prob_name, results)
    except Exception as e:
        return (prob_name, {
            '_error': f"{type(e).__name__}: {e}",
            '_traceback': traceback.format_exc(),
        })


def _igd_gd_checkpoint_path():
    return os.path.join(CONFIG['results_dir'], 'igd_gd_checkpoint.pkl')


def _load_igd_gd_checkpoint():
    p = _igd_gd_checkpoint_path()
    if os.path.exists(p):
        try:
            with open(p, 'rb') as f:
                return pickle.load(f)
        except Exception:
            return {}
    return {}


def _save_igd_gd_checkpoint(state):
    p = _igd_gd_checkpoint_path()
    try:
        tmp = p + '.tmp'
        with open(tmp, 'wb') as f:
            pickle.dump(state, f)
        os.replace(tmp, p)
    except Exception:
        pass


def compute_igd_gd_pass(all_results, problems, pop_size, n_gen,
                        seed_base, n_runs, verbose=True):
    if verbose:
        print("\n" + "=" * 78)
        print("IGD/GD POST-PROCESSING PASS (v8.8)")
        print("=" * 78)

    state = _load_igd_gd_checkpoint()
    if verbose and state:
        print(f"  Resuming: {len(state)} problems already processed")

    algos = get_algo_config(pop_size, n_gen)
    n_cached, n_computed, n_skipped = 0, 0, 0

    for prob_idx, prob_name in enumerate(problems, 1):
        if prob_name in state:
            cached = state[prob_name]
            if isinstance(cached, dict) and cached.get('__skipped__'):
                if verbose:
                    print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                          f"(skipped, cached)")
                n_skipped += 1
                continue
            for algo_name, vals in cached.items():
                if algo_name in all_results.get(prob_name, {}):
                    all_results[prob_name][algo_name].update(vals)
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"(cached)")
            n_cached += 1
            continue

        if prob_name in IGD_SKIP:
            if verbose:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                      f"IGD/GD skipped (reference front unstable)")
            state[prob_name] = {'__skipped__': True}
            _save_igd_gd_checkpoint(state)
            n_skipped += 1
            continue

        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue

        try:
            benchmark = get_extended_benchmark(prob_name)
        except Exception:
            continue

        per_algo_fronts = {a: [] for a, _, _ in algos}
        for algo_name, algo_cls, kwargs in algos:
            if algo_name not in res:
                continue
            for run_idx in range(n_runs):
                seed = seed_base + run_idx
                try:
                    algo = algo_cls(benchmark, seed=seed, **kwargs)
                    pop = algo.run()
                except Exception:
                    continue
                F = _fixed_alpha_objectives(pop, alpha=FAIR_ALPHA,
                                             force=False)
                if F.size > 0:
                    per_algo_fronts[algo_name].append(F)
                del algo, pop

        balanced_fronts = []
        for algo_name, fronts in per_algo_fronts.items():
            if not fronts:
                continue
            algo_points = np.vstack(fronts)
            if len(algo_points) > IGD_PER_ALGO_CAP:
                rng_local = np.random.default_rng(
                    seed_base + (abs(hash(algo_name)) % (2**31)))
                idx = rng_local.choice(len(algo_points),
                                       size=IGD_PER_ALGO_CAP,
                                       replace=False)
                algo_points = algo_points[idx]
            balanced_fronts.append(algo_points)

        ref_front = build_empirical_reference_front(
            balanced_fronts, k=REF_FRONT_MAX_OUTPUT)
        if ref_front.size == 0:
            del per_algo_fronts, balanced_fronts
            gc.collect()
            continue
        ref_diag = _ref_front_diagonal(ref_front)

        cached_for_problem = {}
        for algo_name, fronts in per_algo_fronts.items():
            if algo_name not in res:
                continue
            igds, gds, igds_n, gds_n = [], [], [], []
            for F in fronts:
                igd, gd = _igd_gd_against_reference(F, ref_front)
                if not np.isnan(igd):
                    igds.append(igd)
                    igds_n.append(igd / ref_diag)
                if not np.isnan(gd):
                    gds.append(gd)
                    gds_n.append(gd / ref_diag)
            vals = {
                'igd_mean': float(np.mean(igds)) if igds else np.nan,
                'igd_std':  float(np.std(igds))  if igds else np.nan,
                'igd_all':  igds,
                'gd_mean':  float(np.mean(gds))  if gds else np.nan,
                'gd_std':   float(np.std(gds))   if gds else np.nan,
                'gd_all':   gds,
                'igd_norm_mean': float(np.mean(igds_n)) if igds_n else np.nan,
                'gd_norm_mean':  float(np.mean(gds_n))  if gds_n else np.nan,
            }
            res[algo_name].update(vals)
            cached_for_problem[algo_name] = vals

        del per_algo_fronts, balanced_fronts, ref_front, benchmark
        gc.collect()

        state[prob_name] = cached_for_problem
        _save_igd_gd_checkpoint(state)
        n_computed += 1

        if verbose:
            ref_name = CONFIG['reference_algo']
            igd_dbg = cached_for_problem.get(ref_name, {}).get(
                'igd_mean', float('nan'))
            print(f"  [{prob_idx}/{len(problems)}] {prob_name:<14} "
                  f"ref=|{len(cached_for_problem)}|  "
                  f"IGD({ref_name})={igd_dbg:.4f}")

    if verbose:
        print(f"\n  IGD/GD summary: "
              f"{n_cached} cached, {n_computed} computed, "
              f"{n_skipped} skipped")

    if verbose:
        n_missing = 0
        for prob_name, res in all_results.items():
            if not isinstance(res, dict) or '_error' in res:
                continue
            for algo_name in res:
                if algo_name.startswith('_'):
                    continue
                igd = res[algo_name].get('igd_mean', np.nan)
                if np.isnan(igd):
                    n_missing += 1
        if n_missing > 0:
            print(f"  ⚠ {n_missing} (problem, algorithm) pairs still have "
                  f"NaN IGD. This is expected for FScal5000/FScal10000 "
                  f"and for any problem in IGD_SKIP.")

    return all_results


# ============================================================================
# PART 9: SERIAL / PARALLEL EXECUTION
# ============================================================================

def _wait_some(futures, timeout=300):
    from concurrent.futures import wait, FIRST_COMPLETED
    done, not_done = wait(futures, timeout=timeout,
                          return_when=FIRST_COMPLETED)
    return list(done), list(not_done)


def run_serial(problems, n_runs, pop_size, n_gen, seed_base,
               checkpoint_path=None, verbose=True, monitor=None):
    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}
    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if verbose:
        print(f"\n[SERIAL] {len(todo)} problems to run "
              f"({len(all_results)} cached)")
    t0 = time.time()
    for i, task in enumerate(tasks, 1):
        prob_name = task[0]
        if monitor:
            monitor.start_task(prob_name)
        if verbose and not CONFIG['quiet']:
            print(f"  [{i}/{len(tasks)}] Running {prob_name}...", flush=True)
        _, results = run_single_problem(task)
        if '_error' in results:
            if verbose:
                print(f"    ✗ {results['_error']}")
            if monitor:
                monitor.task_done(prob_name, success=False)
            continue
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


def run_parallel(problems, n_runs, pop_size, n_gen, seed_base,
                 n_workers=None, checkpoint_path=None, verbose=True,
                 monitor=None):
    if IN_JUPYTER and not FORCE_PARALLEL_FROM_JUPYTER:
        if verbose:
            print("\n[JUPYTER MODE] Using serial execution for stability.")
            print("    (Set FORCE_PARALLEL_FROM_JUPYTER = True to override.)")
        return run_serial(problems, n_runs, pop_size, n_gen, seed_base,
                          checkpoint_path, verbose, monitor)

    if n_workers is None:
        n_workers = max(1, mp.cpu_count() // 2)
    n_workers = max(1, min(n_workers, len(problems)))

    all_results = {}
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            if verbose:
                print(f"  Loaded {len(all_results)} finished problems")
        except Exception:
            all_results = {}

    todo = [p for p in problems if p not in all_results]
    tasks = [(p, n_runs, pop_size, n_gen, seed_base + i * 10000)
             for i, p in enumerate(todo)]
    if not todo:
        if verbose:
            print("  All problems already cached.")
        return all_results

    if verbose:
        print(f"\n[PARALLEL] Workers: {n_workers} | "
              f"{len(todo)} problems to run ({len(all_results)} cached)\n")

    t0 = time.time()
    quiet = CONFIG.get('quiet', False)
    in_flight = {}

    def _track_start(name):
        in_flight[name] = time.time()
        if monitor:
            monitor.start_task(name)

    def _track_done(prob_name, results, completed):
        in_flight.pop(prob_name, None)
        if '_error' in results:
            if verbose:
                print(f"  ✗ {prob_name}: {results['_error']}", flush=True)
            if monitor:
                monitor.task_done(prob_name, success=False)
            return
        all_results[prob_name] = results
        _save_checkpoint(all_results, checkpoint_path)
        if monitor:
            monitor.task_done(prob_name, success=True)
        elif verbose and not quiet:
            elapsed = time.time() - t0
            rate = completed / max(elapsed, 1e-6)
            eta = (len(tasks) - completed) / max(rate, 1e-6)
            print(f"  [{completed}/{len(tasks)}] {prob_name} done "
                  f"(ETA: {eta / 60:.1f} min)", flush=True)

    try:
        pe_kwargs = dict(
            max_workers=n_workers,
            mp_context=mp.get_context('spawn'),
            initializer=_worker_init,
        )
        try:
            pe = ProcessPoolExecutor(max_tasks_per_child=1, **pe_kwargs)
        except TypeError:
            pe = ProcessPoolExecutor(**pe_kwargs)
        with pe as executor:
            futures = {}
            for t in tasks:
                _track_start(t[0])
                futures[executor.submit(run_single_problem, t)] = t
            completed = 0
            pending = list(futures.keys())
            while pending:
                done, pending = _wait_some(pending, timeout=300)
                for fut in done:
                    try:
                        prob_name, results = fut.result(timeout=60)
                    except Exception as e:
                        if verbose:
                            print(f"  ✗ Worker crashed: {e}", flush=True)
                        continue
                    completed += 1
                    _track_done(prob_name, results, completed)
    except Exception as e:
        if verbose:
            print(f"\n⚠ Parallel pool failed ({e}). Falling back to serial.")
        remaining = [p for p in problems if p not in all_results]
        serial_results = run_serial(
            remaining, n_runs, pop_size, n_gen, seed_base,
            checkpoint_path, verbose, monitor,
        )
        all_results.update(serial_results)

    if verbose:
        print(f"\nAll done in {(time.time() - t0) / 60:.1f} min")
    return all_results


# ============================================================================
# PART 10: STATISTICS
# ============================================================================

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return 0.0
    count = 0
    for xi in x:
        count += np.sum(xi > y) - np.sum(xi < y)
    return count / (nx * ny)


def cohens_d(x, y):
    x, y = np.asarray(x), np.asarray(y)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return 0.0
    pooled_var = ((nx - 1) * np.var(x, ddof=1) +
                  (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2)
    if pooled_var <= 0:
        return 0.0
    return float((np.mean(x) - np.mean(y)) / np.sqrt(pooled_var))


def bootstrap_ci(data, n_samples=5000, confidence=0.95):
    data = np.asarray(data)
    if len(data) < 2:
        return (float(data[0]) if len(data) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(42)
    n = len(data)
    idx = rng.integers(0, n, size=(n_samples, n))
    means = data[idx].mean(axis=1)
    lo = np.percentile(means, (1 - confidence) / 2 * 100)
    hi = np.percentile(means, (1 + confidence) / 2 * 100)
    return float(np.mean(data)), float(lo), float(hi)


def run_statistical_tests(all_results, problems, algo_names,
                          reference='Aα-NSGA-II (adaptive mutation)',
                          internal_names=None, external_names=None,
                          label=''):
    print("\n" + "=" * 90)
    print(f"STATISTICAL ANALYSIS {label}")
    print("=" * 90)
    if internal_names is None:
        internal_names = INTERNAL_ALGO_NAMES
    if external_names is None:
        external_names = EXTERNAL_ALGO_NAMES

    agg_rank = {a: [] for a in algo_names}
    agg_hv = {a: [] for a in algo_names}
    agg_igd = {a: [] for a in algo_names}
    agg_gd = {a: [] for a in algo_names}
    agg_igd_n = {a: [] for a in algo_names}
    agg_gd_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        if '_error' in res:
            continue
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            agg_rank[a].append(r)
        best_hv = vals[0][1]
        if best_hv > 0:
            for a, hv in vals:
                agg_hv[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                agg_igd[a].append(igd)
            if not np.isnan(gd):
                agg_gd[a].append(gd)
            if not np.isnan(igd_n):
                agg_igd_n[a].append(igd_n)
            if not np.isnan(gd_n):
                agg_gd_n[a].append(gd_n)

    print(f"\n--- Shapiro-Wilk ---")
    for algo in algo_names:
        if len(agg_rank[algo]) < 3:
            continue
        W, p = stats.shapiro(agg_rank[algo])
        print(f"{algo:<38} W={W:.4f}  p={p:.4e}")

    print(f"\n--- Friedman Test ---")
    try:
        sets = [agg_rank[a] for a in algo_names if len(agg_rank[a]) > 0]
        if len(sets) >= 3 and all(len(s) == len(sets[0]) for s in sets):
            stat, p = stats.friedmanchisquare(*sets)
            print(f"Chi-square = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Friedman failed: {e}")

    print(f"\n--- Kruskal-Wallis ---")
    try:
        groups = [agg_hv[a] for a in algo_names if len(agg_hv[a]) > 0]
        if len(groups) >= 2:
            stat, p = stats.kruskal(*groups)
            print(f"H = {stat:.4f}, p = {p:.6e}")
    except Exception as e:
        print(f"Kruskal-Wallis failed: {e}")

    pairwise_results = {}
    for family_label, family_names in (
            ('INTERNAL ABLATION', internal_names),
            (f'EXTERNAL ({label})', external_names)):
        print(f"\n--- Pairwise Wilcoxon vs. reference ({family_label}) ---")
        n_comparisons = max(1, len(family_names) - 1)
        threshold = 0.05 / n_comparisons
        if reference not in agg_rank or len(agg_rank[reference]) < 2:
            continue
        ref_data = np.array(agg_rank[reference])
        for algo in family_names:
            if algo == reference or len(agg_rank[algo]) < 2:
                continue
            if len(agg_rank[algo]) != len(ref_data):
                continue
            try:
                _, p = stats.wilcoxon(ref_data, np.array(agg_rank[algo]))
            except Exception:
                p = np.nan
            d = cohens_d(ref_data, np.array(agg_rank[algo]))
            cl = cliffs_delta(ref_data, np.array(agg_rank[algo]))
            sig = "YES" if (not np.isnan(p) and p < threshold) else "no"
            pairwise_results[algo] = {
                'family': family_label,
                'p': p, 'cohens_d': d, 'cliffs_delta': cl,
                'significant': sig,
            }
            print(f"{algo:<38} p={p:.3e}  d={d:+.3f}  delta={cl:+.3f}  {sig}")
        print(f"Bonferroni threshold ({family_label}): {threshold:.6f}")

    print(f"\n--- Bootstrap 95% CIs (HV) ---")
    ci_results = {}
    for algo in algo_names:
        if len(agg_hv[algo]) < 2:
            continue
        mean, lo, hi = bootstrap_ci(agg_hv[algo], CONFIG['bootstrap_samples'])
        ci_results[algo] = (mean, lo, hi)
        print(f"{algo:<38} {mean:.4f}  [{lo:.4f}, {hi:.4f}]")

    print(f"\n--- Mean IGD / GD (raw, lower is better) ---")
    igd_summary, gd_summary = {}, {}
    igd_norm_summary, gd_norm_summary = {}, {}
    for algo in algo_names:
        if agg_igd[algo]:
            igd_summary[algo] = float(np.mean(agg_igd[algo]))
        if agg_gd[algo]:
            gd_summary[algo] = float(np.mean(agg_gd[algo]))
        if agg_igd_n[algo]:
            igd_norm_summary[algo] = float(np.mean(agg_igd_n[algo]))
        if agg_gd_n[algo]:
            gd_norm_summary[algo] = float(np.mean(agg_gd_n[algo]))
        if algo in igd_summary:
            print(f"{algo:<38} IGD={igd_summary[algo]:.4f}  "
                  f"GD={gd_summary.get(algo, float('nan')):.4f}  "
                  f"IGD_n={igd_norm_summary.get(algo, float('nan')):.4f}")
    if not igd_summary:
        print("  (no IGD/GD values available — did "
              "compute_igd_gd_pass run?)")

    print(f"\n--- Mean Ranks ---")
    for algo in algo_names:
        if agg_rank[algo]:
            print(f"{algo:<38} {np.mean(agg_rank[algo]):.2f}  "
                  f"std={np.std(agg_rank[algo]):.2f}")

    return {
        'agg_rank': agg_rank, 'agg_hv': agg_hv,
        'agg_igd': agg_igd, 'agg_gd': agg_gd,
        'agg_igd_norm': agg_igd_n, 'agg_gd_norm': agg_gd_n,
        'igd_summary': igd_summary, 'gd_summary': gd_summary,
        'igd_norm_summary': igd_norm_summary,
        'gd_norm_summary': gd_norm_summary,
        'pairwise': pairwise_results,
        'bootstrap_cis': ci_results,
    }


def report_framing(stats_results, algo_names):
    print("\n" + "=" * 90)
    print("FRAMING REPORT — What the data actually supports")
    print("=" * 90)

    ref = 'Aα-NSGA-II (adaptive mutation)'
    full_hv = np.mean(stats_results['agg_hv'].get(ref, [np.nan]))
    full_rank = np.mean(stats_results['agg_rank'].get(ref, [np.nan]))

    print(f"\nReference (full mechanism): {ref}")
    print(f"  Normalized HV = {full_hv:.4f}")
    print(f"  Mean rank      = {full_rank:.2f}")

    print("\nComponent ablations (all vs. reference):")
    for algo in algo_names:
        if algo == ref or not stats_results['agg_hv'].get(algo):
            continue
        hv = np.mean(stats_results['agg_hv'][algo])
        rank = np.mean(stats_results['agg_rank'][algo])
        pw = stats_results['pairwise'].get(algo, {})
        sig = pw.get('significant', 'n/a')
        p = pw.get('p', float('nan'))
        print(f"  {algo:<38} HV={hv:.4f}  rank={rank:5.2f}  "
              f"p={p:.2e}  sig={sig}")

    tied = []
    for algo in algo_names:
        if algo == ref:
            continue
        pw = stats_results['pairwise'].get(algo, {})
        if pw.get('significant') == 'no':
            tied.append(algo)
    print("\n--- Recommended claim language ---")
    if tied:
        print("The following algorithms are STATISTICALLY TIED with the")
        print("reference under Bonferroni-corrected Wilcoxon:")
        for a in tied:
            print(f"  - {a}")
        print("\nRecommendation: the contribution is the COMBINATION of")
        print("adaptive-α and adaptive-mutation, not adaptive-α alone.")
    else:
        print("The reference is statistically significantly better than all")
        print("ablations. The contribution can be attributed to the full")
        print("combined mechanism.")
    print("=" * 90)


# ============================================================================
# PART 11: MAIN VISUALIZATIONS
# ============================================================================

def _valid_results(all_results):
    return {k: v for k, v in all_results.items() if '_error' not in v}


def plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family heatmap...")
    all_results = _valid_results(all_results)
    prob_order = []
    family_bounds = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_bounds.append((family, len(prob_order),
                              len(prob_order) + len(probs_in)))
        prob_order.extend(probs_in)

    matrix = np.zeros((len(prob_order), len(algo_names)))
    for i, prob_name in enumerate(prob_order):
        res = all_results[prob_name]
        best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                      if a in res and res[a]['hv_raw_mean'] > 0)
        if best_hv <= 0:
            continue
        for j, algo in enumerate(algo_names):
            if algo in res:
                matrix[i, j] = res[algo]['hv_raw_mean'] / best_hv

    fig, ax = plt.subplots(figsize=(14, max(8, len(prob_order) * 0.25)))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.05)
    ax.set_xticks(np.arange(len(algo_names)))
    ax.set_xticklabels(algo_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(prob_order)))
    ax.set_yticklabels(prob_order, fontsize=7)
    for family, start, end in family_bounds:
        ax.axhline(y=start - 0.5, color='black', linewidth=1.5)
        ax.axhline(y=end - 0.5, color='black', linewidth=1.5)
        mid = (start + end) / 2 - 0.5
        ax.text(-0.6, mid, family, fontsize=11, fontweight='bold',
                rotation=90, va='center', ha='center',
                transform=ax.get_yaxis_transform())
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            color = 'white' if val < 0.5 else 'black'
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=6, color=color)
    ax.set_title(f'Normalized α-weighted HV (α = {FAIR_ALPHA}) per Problem',
                 fontsize=13, pad=15)
    plt.colorbar(im, ax=ax, shrink=0.6)
    plt.tight_layout()
    path = os.path.join(save_dir, 'heatmap_per_problem.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_per_family_bars(all_results, problems, algo_names, families, save_dir):
    print("  Generating per-family bar chart...")
    all_results = _valid_results(all_results)
    families_present = []
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if probs_in:
            families_present.append((family, probs_in))
    if not families_present:
        return
    fig, ax = plt.subplots(figsize=(14, 5))
    n_fam = len(families_present)
    n_algos = len(algo_names)
    width = 0.8 / n_algos
    x = np.arange(n_fam)
    for j, algo in enumerate(algo_names):
        means = []
        for family, probs in families_present:
            vals = []
            for prob_name in probs:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            means.append(np.mean(vals) if vals else 0.0)
        ax.bar(x + j * width - (n_algos - 1) * width / 2,
               means, width, label=algo,
               color=ALGO_COLORS.get(algo, None),
               edgecolor='black', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f[0] for f in families_present], fontsize=10)
    ax.set_ylabel('Mean Normalized α-weighted HV', fontsize=11)
    ax.set_title(f'Performance by Family (α = {FAIR_ALPHA})', fontsize=13)
    ax.legend(fontsize=8, loc='lower right', ncol=2)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    path = os.path.join(save_dir, 'family_bars.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_rank_distribution(all_results, problems, algo_names, save_dir):
    print("  Generating rank distribution...")
    all_results = _valid_results(all_results)
    ranks = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        vals.sort(key=lambda x: -x[1])
        for r, (a, _) in enumerate(vals, 1):
            ranks[a].append(r)
    if not any(ranks[a] for a in algo_names):
        return
    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(algo_names))
    bp = ax.boxplot([ranks[a] for a in algo_names],
                    positions=positions, widths=0.6,
                    patch_artist=True, showmeans=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(ALGO_COLORS.get(algo_names[i], '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xticks(positions)
    ax.set_xticklabels(algo_names, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Rank (1 = best)', fontsize=11)
    ax.set_title('Algorithm Rank Distribution', fontsize=13)
    ax.invert_yaxis()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'rank_distribution.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_analysis(all_results, problems, algo_names, save_dir):
    print("  Generating alpha analysis...")
    all_results = _valid_results(all_results)
    alpha_by_problem = []
    family_labels = []
    for family, probs in FAMILIES.items():
        for prob_name in probs:
            if (prob_name in all_results and
                    'Aα-NSGA-II (adaptive mutation)' in all_results[prob_name]):
                a = all_results[prob_name]['Aα-NSGA-II (adaptive mutation)']['alpha_mean']
                if not np.isnan(a):
                    alpha_by_problem.append(a)
                    family_labels.append(family)
    if not alpha_by_problem:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.hist(alpha_by_problem, bins=15, color='steelblue',
            edgecolor='black', alpha=0.7)
    ax.axvline(np.mean(alpha_by_problem), color='red', linestyle='--',
               linewidth=2, label=f'Mean = {np.mean(alpha_by_problem):.3f}')
    ax.axvline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xlabel('Converged α', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title('Converged α of Aα-NSGA-II (adaptive mutation)',
                 fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax = axes[1]
    family_alphas = {}
    for a, f in zip(alpha_by_problem, family_labels):
        family_alphas.setdefault(f, []).append(a)
    families_list = list(family_alphas.keys())
    positions = np.arange(len(families_list))
    bp = ax.boxplot([family_alphas[f] for f in families_list],
                    positions=positions, widths=0.6, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)
    ax.axhline(0.75, color='green', linestyle=':', linewidth=2,
               label='Fixed α = 0.75')
    ax.set_xticks(positions)
    ax.set_xticklabels(families_list, fontsize=10)
    ax.set_ylabel('Converged α', fontsize=11)
    ax.set_title('Converged α by Family', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_analysis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_scalability(all_results, problems, algo_names, save_dir):
    all_results = _valid_results(all_results)
    scal_probs = [p for p in problems if p.startswith('FScal')]
    if not scal_probs:
        return
    print("  Generating scalability plot...")
    sizes = []
    for p in scal_probs:
        try:
            s = int(p.replace('FScal', ''))
        except ValueError:
            s = 500
        sizes.append(s)
    order = np.argsort(sizes)
    sizes = [sizes[i] for i in order]
    scal_probs = [scal_probs[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    any_plotted = False
    for algo in algo_names:
        hvs = []
        for prob_name in scal_probs:
            if prob_name not in all_results:
                hvs.append(np.nan)
                continue
            if algo in all_results[prob_name]:
                best = max(all_results[prob_name][a]['hv_raw_mean']
                           for a in algo_names
                           if a in all_results[prob_name]
                           and all_results[prob_name][a]['hv_raw_mean'] > 0)
                hvs.append(all_results[prob_name][algo]['hv_raw_mean'] / best
                           if best > 0 else 0)
            else:
                hvs.append(np.nan)
        if not np.all(np.isnan(hvs)):
            ax.plot(sizes, hvs, 'o-', label=algo,
                    color=ALGO_COLORS.get(algo, None), linewidth=1.5)
            any_plotted = True
    if not any_plotted:
        plt.close()
        return
    ax.set_xlabel('Number of Variables', fontsize=11)
    ax.set_ylabel('Normalized α-weighted HV', fontsize=11)
    ax.set_title('Scalability: Hypervolume', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xscale('log')
    plt.tight_layout()
    path = os.path.join(save_dir, 'scalability.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_bootstrap_cis(ci_results, algo_names, save_dir):
    if not ci_results:
        return
    print("  Generating bootstrap CI plot...")
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, algo in enumerate(algo_names):
        if algo not in ci_results:
            continue
        mean, lo, hi = ci_results[algo]
        ax.errorbar(mean, i, xerr=[[mean - lo], [hi - mean]],
                    fmt='o', color=ALGO_COLORS.get(algo, 'black'),
                    capsize=6, capthick=2, markersize=10)
    ax.set_yticks(np.arange(len(algo_names)))
    ax.set_yticklabels(algo_names, fontsize=9)
    ax.set_xlabel('Mean Normalized α-weighted HV (95% Bootstrap CI)',
                  fontsize=11)
    ax.set_title('Bootstrap 95% Confidence Intervals', fontsize=13)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'bootstrap_cis.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_igd_gd_bars(stats_results, algo_names, save_dir):
    igd = stats_results.get('igd_summary', {})
    gd = stats_results.get('gd_summary', {})
    igd_n = stats_results.get('igd_norm_summary', {})
    gd_n = stats_results.get('gd_norm_summary', {})
    if not igd and not gd:
        print("  (skipping IGD/GD bars — no values)")
        return
    print("  Generating IGD/GD bar chart...")
    algos = [a for a in algo_names if a in igd or a in gd]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for ax, source, title in (
            (axes[0, 0], igd, 'IGD (raw)'),
            (axes[0, 1], gd, 'GD (raw)'),
            (axes[1, 0], igd_n, 'IGD (normalized)'),
            (axes[1, 1], gd_n, 'GD (normalized)')):
        vals = [source.get(a, np.nan) for a in algos]
        ypos = np.arange(len(algos))
        colors = [ALGO_COLORS.get(a, '#cccccc') for a in algos]
        ax.barh(ypos, vals, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(ypos)
        ax.set_yticklabels(algos, fontsize=8)
        ax.set_title(title, fontsize=11)
        ax.grid(axis='x', alpha=0.3)
        ax.invert_yaxis()
    plt.tight_layout()
    path = os.path.join(save_dir, 'igd_gd.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def plot_alpha_adaptation_effect(all_results, problems, save_dir):
    pairs = [('SPEA2', 'SPEA2+α'),
             ('MOEA/D', 'MOEA/D+α'),
             ('NSGA-III', 'NSGA-III+α'),
             ('cMLSGA', 'cMLSGA+α')]
    probs = [p for p in problems
             if p in all_results and '_error' not in all_results[p]]
    if not probs:
        return
    print("  Generating α-adaptation effect plot...")
    fig, ax = plt.subplots(figsize=(12, 5))
    n = len(pairs)
    w = 0.8 / n
    x = np.arange(len(probs))
    for j, (strict, fair) in enumerate(pairs):
        vals = []
        for p in probs:
            r = all_results[p]
            if (strict in r and fair in r
                    and r[strict].get('hv_raw_mean', 0) > 0):
                vals.append(r[fair]['hv_raw_mean'] /
                            r[strict]['hv_raw_mean'])
            else:
                vals.append(np.nan)
        ax.bar(x + j * w - (n - 1) * w / 2, vals, w,
               label=f'{strict} → {fair}',
               color=ALGO_COLORS.get(fair, None),
               edgecolor='black', linewidth=0.4)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(probs, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('HV(fair) / HV(strict)   (>1 ⇒ α-adaptation helps)')
    ax.set_title('Effect of α-adaptation on each competitor algorithm',
                 fontsize=13)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, 'alpha_adaptation_effect.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"    Saved: {path}")


def generate_all_visualizations(all_results, problems, algo_names, families,
                                ci_results, save_dir, stats_results=None):
    print("\n" + "=" * 78)
    print("GENERATING MAIN VISUALIZATIONS")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    plot_per_family_heatmap(all_results, problems, algo_names, families, save_dir)
    plot_per_family_bars(all_results, problems, algo_names, families, save_dir)
    plot_rank_distribution(all_results, problems, algo_names, save_dir)
    plot_alpha_analysis(all_results, problems, algo_names, save_dir)
    plot_scalability(all_results, problems, algo_names, save_dir)
    plot_bootstrap_cis(ci_results, algo_names, save_dir)
    plot_alpha_adaptation_effect(all_results, problems, save_dir)
    if stats_results is not None:
        plot_igd_gd_bars(stats_results, algo_names, save_dir)


# ============================================================================
# PART 11B: PARETO FRONT PLOTS (v8.8: HIGH CONTRAST, Z-ORDER, MARKERS)
# ============================================================================

_CACHE_DIR_NAME = '_cache'


def _pareto_cache_path(save_dir, prob_name, tag, algo_name):
    cache = os.path.join(save_dir, _CACHE_DIR_NAME)
    safe = (algo_name.replace('α', 'alpha').replace('/', '_')
            .replace('\\', '_').replace(' ', '_'))
    return os.path.join(cache, f"{prob_name}__{tag}__{safe}.npy")


def _load_cached_front(save_dir, prob_name, tag, algo_name):
    p = _pareto_cache_path(save_dir, prob_name, tag, algo_name)
    if os.path.exists(p):
        try:
            return np.load(p, allow_pickle=False)
        except Exception:
            return None
    return None


def _save_cached_front(save_dir, prob_name, tag, algo_name, F):
    cache = os.path.join(save_dir, _CACHE_DIR_NAME)
    os.makedirs(cache, exist_ok=True)
    p = _pareto_cache_path(save_dir, prob_name, tag, algo_name)
    try:
        np.save(p, np.asarray(F, dtype=float))
    except Exception:
        pass


def _downsample_front(F, max_points=60):
    if len(F) <= max_points:
        return F
    order = np.argsort(F[:, 0])
    idx = np.linspace(0, len(F) - 1, max_points).astype(int)
    return F[order][idx]


def _plot_fronts_from_dict(prob_name, fronts, alpha, save_dir,
                          make_3d=True, make_2d_projections=True,
                          max_points=60, n_obj=None):
    if n_obj is None:
        if not fronts:
            return
        for F in fronts.values():
            if F.size:
                n_obj = F.shape[1]
                break
    if not n_obj or not fronts:
        return

    # Z-Order layering so the hero algorithm is plotted on top
    hero_name = 'Aα-NSGA-II (adaptive mutation)'
    
    def get_z_order(algo_name):
        if algo_name == hero_name: return 10
        if algo_name.startswith('SPEA2'): return 4
        if algo_name.startswith('MOEA/D'): return 3
        if algo_name.startswith('NSGA-III'): return 2
        if algo_name.startswith('cMLSGA'): return 1
        return 5
        
    sorted_algos = sorted(fronts.keys(), key=get_z_order)

    # --- 2D Plot ---
    if n_obj == 2:
        fig, ax = plt.subplots(figsize=(10, 7))
        for algo_name in sorted_algos:
            F = fronts[algo_name]
            if len(F) == 0: continue
            Fd = _downsample_front(F, max_points)
            order = np.argsort(Fd[:, 0])
            
            color = ALGO_COLORS.get(algo_name, 'black')
            marker = ALGO_MARKERS.get(algo_name, 'o')
            z = get_z_order(algo_name)
            
            ax.plot(Fd[order, 0], Fd[order, 1], marker=marker, markersize=7,
                    label=f"{algo_name}", color=color, linewidth=1.5,
                    alpha=0.9, markeredgecolor='black', markeredgewidth=0.5,
                    zorder=z)
                    
        ax.set_xlabel('\(f_1\) (α-weighted)', fontsize=12)
        ax.set_ylabel('\(f_2\) (α-weighted)', fontsize=12)
        ax.set_title(f'{prob_name} — Pareto Fronts (α={alpha})', fontsize=14)
        ax.legend(fontsize=10, framealpha=0.9, edgecolor='black')
        ax.grid(alpha=0.3, linestyle='--')
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2d.png')
        plt.savefig(path, dpi=250, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")
        return

    # --- 3D Projections ---
    pairs = [(0, 1), (0, 2), (1, 2)]
    proj_titles = [('\(f_1\)', '\(f_2\)'), ('\(f_1\)', '\(f_3\)'), ('\(f_2\)', '\(f_3\)')]
    
    if make_2d_projections:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        for ax, (i, j), (xl, yl) in zip(axes, pairs, proj_titles):
            for algo_name in sorted_algos:
                F = fronts[algo_name]
                if len(F) == 0: continue
                Fd = _downsample_front(F, max_points)
                
                color = ALGO_COLORS.get(algo_name, 'black')
                marker = ALGO_MARKERS.get(algo_name, 'o')
                z = get_z_order(algo_name)
                
                ax.scatter(Fd[:, i], Fd[:, j], s=45, color=color, marker=marker,
                           alpha=0.85, edgecolors='black', linewidths=0.6,
                           label=algo_name if i == 0 and j == 1 else None,
                           zorder=z)
                           
            ax.set_xlabel(xl, fontsize=12)
            ax.set_ylabel(yl, fontsize=12)
            ax.grid(alpha=0.3, linestyle='--')
            
        axes[0].legend(fontsize=10, framealpha=0.9, edgecolor='black', 
                       loc='best', markerscale=1.2)
        fig.suptitle(f'{prob_name} — 2D Projections (α={alpha})', fontsize=15, y=1.02)
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_2dproj.png')
        plt.savefig(path, dpi=250, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")

    # --- 3D Scatter ---
    if make_3d:
        from mpl_toolkits.mplot3d import Axes3D  # noqa
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        for algo_name in sorted_algos:
            F = fronts[algo_name]
            if len(F) == 0: continue
            Fd = _downsample_front(F, max_points)
            
            color = ALGO_COLORS.get(algo_name, 'black')
            marker = ALGO_MARKERS.get(algo_name, 'o')
            z = get_z_order(algo_name)
            
            # Hero gets slightly larger markers and full opacity
            size = 45 if algo_name == hero_name else 35
            opac = 1.0 if algo_name == hero_name else 0.75
            
            ax.scatter(Fd[:, 0], Fd[:, 1], Fd[:, 2],
                       label=f"{algo_name} (|PF|={len(F)})", 
                       s=size, color=color, marker=marker,
                       alpha=opac, edgecolors='black', linewidths=0.6,
                       zorder=z)
                       
        ax.set_xlabel('$f_1$', fontsize=12, labelpad=10)
        ax.set_ylabel('$f_2$', fontsize=12, labelpad=10)
        ax.set_zlabel('$f_3$', fontsize=12, labelpad=10)
        ax.set_title(f'{prob_name} — Pareto Fronts (α={alpha})', fontsize=14)
        
        ax.legend(fontsize=9, loc='upper left', framealpha=0.9, 
                  edgecolor='black', markerscale=1.3)
                  
        # Clean up the 3D panes for higher contrast
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.grid(color='grey', linestyle=':', linewidth=0.5, alpha=0.5)
        
        plt.tight_layout()
        path = os.path.join(save_dir, f'pareto_{prob_name}_3d.png')
        plt.savefig(path, dpi=250, bbox_inches='tight')
        plt.close()
        print(f"    Saved: {path}")


def _tag_algo_configs(ps, ng):
    all_cfg = get_algo_config(ps, ng, external_mode='both')
    internal_set = set(INTERNAL_ALGO_NAMES)
    fair_set = set(EXTERNAL_ALGO_NAMES)
    strict_set = set(EXTERNAL_ALGO_NAMES_STRICT)
    return {
        'ablation': [c for c in all_cfg if c[0] in internal_set],
        'fair':     [c for c in all_cfg
                     if c[0] in ({'Aα-NSGA-II (adaptive mutation)'}
                                 | fair_set)],
        'strict':   [c for c in all_cfg
                     if c[0] in ({'Aα-NSGA-II (adaptive mutation)'}
                                 | strict_set)],
    }


def generate_all_pareto_fronts(problems, save_dir, alpha=FAIR_ALPHA,
                                quick=False, algo_subset='all',
                                force_recompute=False):
    print("\n" + "=" * 78)
    print(f"GENERATING PARETO FRONT PLOTS (fixed α = {alpha})")
    print(f"  quick={quick}  algo_subset={algo_subset}  "
          f"force_recompute={force_recompute}")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    ps, ng = (40, 150) if quick else (60, 300)

    tags_cfg = _tag_algo_configs(ps, ng)
    if algo_subset == 'all':
        tags = ['ablation', 'fair', 'strict']
    elif algo_subset in tags_cfg:
        tags = [algo_subset]
    else:
        raise ValueError(
            f"algo_subset must be one of 'all', 'ablation', 'fair', "
            f"'strict' — got {algo_subset!r}"
        )

    for tag in tags:
        cfg = tags_cfg[tag]
        out_dir = os.path.join(save_dir, tag)
        os.makedirs(out_dir, exist_ok=True)

        for prob_idx, prob_name in enumerate(problems, 1):
            try:
                benchmark = get_extended_benchmark(prob_name)
            except Exception as e:
                print(f"  [{prob_idx}/{len(problems)}] {prob_name}: "
                      f"get_benchmark failed: {e}")
                continue

            n_obj = benchmark.n_obj
            fronts = {}
            print(f"  [{prob_idx}/{len(problems)}] {prob_name} "
                  f"({tag}, n_obj={n_obj}) ...", flush=True)

            for algo_name, algo_cls, kwargs in cfg:
                F = None
                if not force_recompute:
                    F = _load_cached_front(save_dir, prob_name, tag,
                                           algo_name)
                if F is None:
                    try:
                        algo = algo_cls(benchmark, seed=2026, **kwargs)
                        pop = algo.run()
                        F = _fixed_alpha_objectives(pop, alpha=alpha,
                                                    force=False)
                        _save_cached_front(save_dir, prob_name, tag,
                                           algo_name, F)
                        print(f"      {algo_name}: |PF|={len(F)}",
                              flush=True)
                        del algo, pop
                    except Exception as e:
                        print(f"      {algo_name}: FAILED "
                              f"({type(e).__name__}: {e})", flush=True)
                        traceback.print_exc()
                        F = None
                else:
                    print(f"      {algo_name}: cached |PF|={len(F)}",
                          flush=True)
                if F is not None and F.size > 0:
                    fronts[algo_name] = F

            if not fronts:
                print(f"      (no fronts; skipping plot)", flush=True)
                continue

            try:
                _plot_fronts_from_dict(
                    prob_name, fronts, alpha, out_dir,
                    make_3d=True,
                    make_2d_projections=True,
                    max_points=40,
                    n_obj=n_obj,
                )
            except Exception as e:
                print(f"      plot failed for {prob_name} ({tag}): "
                      f"{type(e).__name__}: {e}", flush=True)
                traceback.print_exc()
            finally:
                del fronts
                gc.collect()


# ============================================================================
# PART 11C: Σ̂ TABLE
# ============================================================================

def generate_sigma_hat_table(all_results, problems, save_dir,
                             n_runs_per_alpha=SIGMA_HAT_RUNS_PER_ALPHA):
    print("\n" + "=" * 78)
    print("COMPUTING α-SENSITIVITY PROXY Σ̂  (v8.8: same population per α)")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)

    rows = []
    for prob_name in problems:
        try:
            sigma, hvs = compute_sigma_hat(
                prob_name,
                alphas=SIGMA_HAT_ALPHAS,
                seed=CONFIG['sigma_hat_seed'],
                pop_size=CONFIG['pop_size'],
                n_gen=CONFIG['n_gen'],
                n_runs_per_alpha=n_runs_per_alpha,
            )
            rows.append({
                'problem': prob_name,
                'family': 'F2' if prob_name.startswith('F2') else 'F1',
                'sigma_hat': sigma,
                'hv_0.50': hvs[0],
                'hv_0.75': hvs[1],
                'hv_1.00': hvs[2],
            })
            print(f"  {prob_name:<14}  Σ̂ = {sigma:.4f}  "
                  f"HV(0.5)={hvs[0]:.3f}  HV(0.75)={hvs[1]:.3f}  "
                  f"HV(1.0)={hvs[2]:.3f}")
        except Exception as e:
            print(f"  ✗ {prob_name}: {type(e).__name__}: {e}")

    if not rows:
        return

    df = pd.DataFrame(rows)
    csv_path = os.path.join(save_dir, 'sigma_hat.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"\n  CSV saved to {csv_path}")

    df = df.sort_values(['family', 'problem']).reset_index(drop=True)

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Empirical α-sensitivity proxy $\widehat{\Sigma}$ '
        r'per problem. $\widehat{\Sigma}$ is the normalized spread of the '
        r'fixed-$\alpha$ hypervolume across $\alpha \in \{0.5, 0.75, 1.0\}$, '
        r'averaged over '
        f'{n_runs_per_alpha} seeds and computed against a '
        r'single reference point (union of all fronts). '
        r'Each population is evaluated under all three $\alpha$ values '
        r'to isolate sensitivity from cross-seed variability.}',
        r'\label{tab:sigma_hat}',
        r'\small',
        r'\begin{tabular}{@{}llccc@{}}', r'\toprule',
        r'Problem & Family & $\widehat{\Sigma}$ & '
        r'$\overline{\mathrm{HV}}(0.5)$ & '
        r'$\overline{\mathrm{HV}}(1.0)$ \\',
        r'\midrule',
    ]
    for _, row in df.iterrows():
        lines.append(
            f"{row['problem']} & {row['family']} & "
            f"{row['sigma_hat']:.4f} & {row['hv_0.50']:.3f} & "
            f"{row['hv_1.00']:.3f} \\\\"
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    tex_path = os.path.join(save_dir, 'table_sigma_hat.tex')
    with open(tex_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  LaTeX table saved to {tex_path}")

    verify_sigma_hat_families(save_dir)


# ============================================================================
# PART 12: LATEX TABLES
# ============================================================================

def _build_external_table(all_results, problems, algo_names,
                          stats_results, save_dir, filename, caption, label):
    norm = {a: [] for a in algo_names}
    ranks = {a: [] for a in algo_names}
    igds = {a: [] for a in algo_names}
    gds = {a: [] for a in algo_names}
    igds_n = {a: [] for a in algo_names}
    gds_n = {a: [] for a in algo_names}
    for prob_name in problems:
        if prob_name not in all_results:
            continue
        res = all_results[prob_name]
        vals = [(a, res[a]['hv_raw_mean']) for a in algo_names if a in res]
        if not vals:
            continue
        vals.sort(key=lambda x: -x[1])
        best_hv = vals[0][1]
        for r, (a, hv) in enumerate(vals, 1):
            ranks[a].append(r)
            if best_hv > 0:
                norm[a].append(hv / best_hv)
        for a in algo_names:
            if a not in res:
                continue
            igd = res[a].get('igd_mean', np.nan)
            gd = res[a].get('gd_mean', np.nan)
            igd_n = res[a].get('igd_norm_mean', np.nan)
            gd_n = res[a].get('gd_norm_mean', np.nan)
            if not np.isnan(igd):
                igds[a].append(igd)
            if not np.isnan(gd):
                gds[a].append(gd)
            if not np.isnan(igd_n):
                igds_n[a].append(igd_n)
            if not np.isnan(gd_n):
                gds_n[a].append(gd_n)

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{' + caption + '}',
        r'\label{' + label + '}',
        r'\begin{tabular}{@{}lccccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Rank & Wins & IGD & GD & '
        r'IGD$_n$ \\',
        r'\midrule',
    ]
    for algo in algo_names:
        if not norm[algo]:
            continue
        mean_hv = np.mean(norm[algo])
        std_hv = np.std(norm[algo])
        mean_rank = np.mean(ranks[algo])
        wins = sum(1 for i in range(len(norm[algo])) if ranks[algo][i] == 1)
        igd_s = (f'{np.mean(igds[algo]):.4f}' if igds[algo] else '--')
        gd_s = (f'{np.mean(gds[algo]):.4f}' if gds[algo] else '--')
        igd_n_s = (f'{np.mean(igds_n[algo]):.4f}' if igds_n[algo] else '--')
        lines.append(f'{algo} & {mean_hv:.4f} & {std_hv:.4f} & '
                     f'{mean_rank:.2f} & {wins} & {igd_s} & {gd_s} & '
                     f'{igd_n_s} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, filename), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))


def generate_latex_tables(all_results, problems, algo_names, families,
                          stats_strict, stats_fair, save_dir):
    print("\n" + "=" * 78)
    print("GENERATING MAIN LATEX TABLES")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)

    strict_algos = ['Aα-NSGA-II (adaptive mutation)'] + EXTERNAL_ALGO_NAMES_STRICT
    _build_external_table(
        all_results, problems, strict_algos, stats_strict, save_dir,
        filename='table_external_strict.tex',
        caption=(
            r'External comparison against standard state-of-the-art '
            r'algorithms with fixed $\alpha = 0.75$. This is the primary '
            r'external comparison; it tests whether adaptive-$\alpha$ is '
            r'valuable. IGD/GD are computed against a memory-safe, '
            r'per-algorithm-balanced empirical reference front; lower is '
            r'better. IGD$_n$ is IGD divided by the reference front '
            r'bounding-box diagonal for cross-problem comparability.'
        ),
        label='tab:external_strict',
    )

    fair_algos = ['Aα-NSGA-II (adaptive mutation)'] + EXTERNAL_ALGO_NAMES
    _build_external_table(
        all_results, problems, fair_algos, stats_fair, save_dir,
        filename='table_external_fair.tex',
        caption=(
            r'Ablation: competitors are $\alpha$-wrapped so they receive the '
            r'same $\alpha$-adaptation capability as A$\alpha$-NSGA-II. This '
            r'isolates the contribution of the selection and mutation '
            r'mechanisms when the $\alpha$ advantage is neutralized.'
        ),
        label='tab:external_fair',
    )

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Internal ablation: overall performance on all problems. '
        r'Normalized α-weighted HV (fixed $\alpha = 0.75$) and raw IGD/GD.}',
        r'\label{tab:overall_internal}',
        r'\begin{tabular}{@{}lccccccc@{}}', r'\toprule',
        r'Algorithm & Norm.\ HV & Std. & Rank & Wins & IGD & GD & '
        r'IGD$_n$ \\',
        r'\midrule',
    ]
    for algo in INTERNAL_ALGO_NAMES:
        hv_all = []
        ranks_all = []
        igd_all = []
        gd_all = []
        igd_n_all = []
        for prob_name in problems:
            if prob_name not in all_results:
                continue
            res = all_results[prob_name]
            vals = [(a, res[a]['hv_raw_mean']) for a in INTERNAL_ALGO_NAMES
                    if a in res]
            if not vals:
                continue
            vals.sort(key=lambda x: -x[1])
            best = vals[0][1]
            if best > 0 and algo in res:
                hv_all.append(res[algo]['hv_raw_mean'] / best)
            for r, (a, _) in enumerate(vals, 1):
                if a == algo:
                    ranks_all.append(r)
            if algo in res:
                if not np.isnan(res[algo].get('igd_mean', np.nan)):
                    igd_all.append(res[algo]['igd_mean'])
                if not np.isnan(res[algo].get('gd_mean', np.nan)):
                    gd_all.append(res[algo]['gd_mean'])
                if not np.isnan(res[algo].get('igd_norm_mean', np.nan)):
                    igd_n_all.append(res[algo]['igd_norm_mean'])
        if not hv_all:
            continue
        lines.append(
            f'{algo} & {np.mean(hv_all):.4f} & {np.std(hv_all):.4f} & '
            f'{np.mean(ranks_all):.2f} & '
            f'{sum(1 for r in ranks_all if r == 1)} & '
            f'{(np.mean(igd_all) if igd_all else float("nan")):.4f} & '
            f'{(np.mean(gd_all) if gd_all else float("nan")):.4f} & '
            f'{(np.mean(igd_n_all) if igd_n_all else float("nan")):.4f} \\\\'
        )
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_overall.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Mean Normalized α-weighted HV by Family}',
        r'\label{tab:family}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'Family & ' + ' & '.join(algo_names) + r' \\', r'\midrule',
    ]
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        row = [family]
        for algo in algo_names:
            vals = []
            for prob_name in probs_in:
                res = all_results[prob_name]
                best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                              if a in res and res[a]['hv_raw_mean'] > 0)
                if best_hv > 0 and algo in res:
                    vals.append(res[algo]['hv_raw_mean'] / best_hv)
            row.append(f'{np.mean(vals):.3f}' if vals else '--')
        lines.append(' & '.join(row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_family.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))

    for tag, stats_res, ext_names in (
            ('strict', stats_strict, EXTERNAL_ALGO_NAMES_STRICT),
            ('fair', stats_fair, EXTERNAL_ALGO_NAMES)):
        lines = [
            r'\begin{table}[htbp]', r'\centering',
            r'\caption{Pairwise Wilcoxon (Bonferroni) on ranks — '
            + tag + r' external comparison.}',
            r'\label{tab:statistics_' + tag + '}',
            r'\begin{tabular}{@{}llcccc@{}}', r'\toprule',
            r'Family & Comparison & $p$ & Cohen $d$ & Cliff $\delta$ & Sig. \\',
            r'\midrule',
        ]
        ref = CONFIG['reference_algo']
        for algo, res in stats_res.get('pairwise', {}).items():
            if algo not in ext_names:
                continue
            lines.append(
                f"{res.get('family', '--')} & {ref} vs. {algo} & "
                f"{res['p']:.2e} & {res['cohens_d']:.3f} & "
                f"{res['cliffs_delta']:.3f} & {res['significant']} \\\\"
            )
        lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
        with open(os.path.join(save_dir,
                               f'table_statistics_{tag}.tex'), 'w',
                  encoding='utf-8') as f:
            f.write('\n'.join(lines))

    lines = [
        r'\begin{table}[htbp]', r'\centering',
        r'\caption{Bootstrap 95\% Confidence Intervals (Normalized HV)}',
        r'\label{tab:bootstrap}',
        r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
        r'Algorithm & Mean & CI Lower & CI Upper \\', r'\midrule',
    ]
    for algo in algo_names:
        if algo in stats_strict['bootstrap_cis']:
            mean, lo, hi = stats_strict['bootstrap_cis'][algo]
            lines.append(f'{algo} & {mean:.4f} & {lo:.4f} & {hi:.4f} \\\\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{table}'])
    with open(os.path.join(save_dir, 'table_bootstrap.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  Generated LaTeX tables in {save_dir}/")


def generate_per_problem_table(all_results, problems, algo_names, families,
                                save_dir):
    print("\n" + "=" * 78)
    print("GENERATING PER-PROBLEM TABLE")
    print("=" * 78)
    os.makedirs(save_dir, exist_ok=True)
    all_results = _valid_results(all_results)
    lines = [
        r'\begin{table*}[htbp]', r'\centering',
        r'\caption{Per-Problem Normalized α-weighted HV (best in bold)}',
        r'\label{tab:per_problem}',
        r'\scriptsize', r'\setlength{\tabcolsep}{3pt}',
        r'\begin{tabular}{@{}l' + 'c' * len(algo_names) + r'@{}}',
        r'\toprule',
        r'\textbf{Problem} & ' + ' & '.join([f'\\textbf{{{a}}}'
                                              for a in algo_names]) + r' \\',
        r'\midrule',
    ]
    family_rows = {}
    for family, probs in families.items():
        probs_in = [p for p in probs if p in all_results]
        if not probs_in:
            continue
        family_rows[family] = []
        for prob_name in probs_in:
            res = all_results[prob_name]
            best_hv = max(res[a]['hv_raw_mean'] for a in algo_names
                          if a in res and res[a]['hv_raw_mean'] > 0)
            if best_hv <= 0:
                continue
            row_vals = []
            for algo in algo_names:
                if algo in res:
                    row_vals.append((algo, res[algo]['hv_raw_mean'] / best_hv))
                else:
                    row_vals.append((algo, np.nan))
            valid = [(a, v) for a, v in row_vals if not np.isnan(v)]
            best_val = max(v for _, v in valid) if valid else 0
            family_rows[family].append((prob_name, row_vals, best_val))
    for family, rows in family_rows.items():
        lines.append(f'\\multicolumn{{{len(algo_names) + 1}}}{{l}}'
                     f'{{\\textit{{{family}}}}} \\\\')
        lines.append(r'\midrule')
        for prob_name, row_vals, best_val in rows:
            cells = []
            for algo, val in row_vals:
                if np.isnan(val):
                    cells.append('--')
                elif abs(val - best_val) < 1e-6:
                    cells.append(f'\\textbf{{{val:.3f}}}')
                else:
                    cells.append(f'{val:.3f}')
            lines.append(f'{prob_name} & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    if lines[-1] == r'\midrule':
        lines[-1] = r'\bottomrule'
    lines.extend([r'\end{tabular}', r'\end{table*}'])
    with open(os.path.join(save_dir, 'table_per_problem.tex'), 'w',
              encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"    Saved: {os.path.join(save_dir, 'table_per_problem.tex')}")

    csv_rows = []
    for family, rows in family_rows.items():
        for prob_name, row_vals, best_val in rows:
            row = {'family': family, 'problem': prob_name}
            for algo, val in row_vals:
                row[algo] = val
            csv_rows.append(row)
    if csv_rows:
        df = pd.DataFrame(csv_rows)
        df.to_csv(os.path.join(save_dir, 'per_problem_results.csv'),
                  index=False, encoding='utf-8')


# ============================================================================
# PART 13: SETTINGS AND MAIN
# ============================================================================

def apply_settings(mode=None, use_parallel=None, n_workers=None,
                   runs=None, pop=None, gen=None, quiet=None,
                   external_mode=None):
    if mode == 'test':
        CONFIG['problems'] = ['FZDT1', 'FZDT2', 'FDTLZ2',
                              'F2FZDT1', 'F2FUF1']
        CONFIG['n_runs'] = 2
        CONFIG['pop_size'] = 30
        CONFIG['n_gen'] = 50
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'test_results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'test_figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'test_latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'test_supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'test_sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'test_external')
    else:
        CONFIG['results_dir']    = os.path.join(OUTPUT_ROOT, 'results')
        CONFIG['figures_dir']    = os.path.join(OUTPUT_ROOT, 'figures')
        CONFIG['latex_dir']      = os.path.join(OUTPUT_ROOT, 'latex')
        CONFIG['supp_latex_dir'] = os.path.join(OUTPUT_ROOT,
                                                'supplementary_latex')
        CONFIG['sigma_hat_dir']  = os.path.join(OUTPUT_ROOT, 'sigma_hat')
        CONFIG['external_dir']   = os.path.join(OUTPUT_ROOT, 'external')
    if runs is not None:
        CONFIG['n_runs'] = runs
    if pop is not None:
        CONFIG['pop_size'] = pop
    if gen is not None:
        CONFIG['n_gen'] = gen
    if use_parallel is not None:
        CONFIG['use_parallel'] = use_parallel
    if n_workers is not None:
        CONFIG['n_workers'] = n_workers
    if quiet is not None:
        CONFIG['quiet'] = quiet
    if external_mode is not None:
        global EXTERNAL_COMPARISON_MODE
        EXTERNAL_COMPARISON_MODE = external_mode
    if CONFIG['n_workers'] is None:
        CONFIG['n_workers'] = max(1, mp.cpu_count() // 2)


def run_pareto_only(problems=None, quick=False,
                    pop=None, gen=None,
                    algo_subset='all',
                    force_recompute=False):
    apply_settings(mode='full', pop=pop, gen=gen)
    if problems is None:
        problems = PARETO_DEFAULT_PROBLEMS
    ps = CONFIG['pop_size']
    ng = CONFIG['n_gen']
    if quick:
        ps, ng = 40, 150

    out = os.path.join(CONFIG['figures_dir'], 'pareto_fronts')
    os.makedirs(out, exist_ok=True)
    for tag in ('ablation', 'fair', 'strict'):
        os.makedirs(os.path.join(out, tag), exist_ok=True)

    print("=" * 78)
    print(f"PARETO-ONLY MODE (v8.8) — {len(problems)} problems, "
          f"pop={ps}, gen={ng}, α={FAIR_ALPHA}")
    print(f"  algo_subset   = {algo_subset}")
    print(f"  quick         = {quick}")
    print(f"  recompute     = {force_recompute}")
    print(f"Output: {out}")
    print("=" * 78)

    generate_all_pareto_fronts(
        problems, out, alpha=FAIR_ALPHA, quick=quick,
        algo_subset=algo_subset,
        force_recompute=force_recompute,
    )

    print("\nPareto-only mode complete.")


def main(mode=None, use_parallel=None, n_workers=None,
         runs=None, pop=None, gen=None, quiet=None,
         pareto_quick=True, compute_sigma=True,
         include_external=True, external_mode=None,
         skip_main_runs=False, skip_igd_gd=False,
         pareto_only=False, pareto_problems=None,
         pareto_algos='all', pareto_force=False):
    apply_settings(mode=mode, use_parallel=use_parallel, n_workers=n_workers,
                   runs=runs, pop=pop, gen=gen, quiet=quiet,
                   external_mode=external_mode)

    if pareto_only:
        return run_pareto_only(
            problems=pareto_problems,
            quick=pareto_quick,
            pop=pop, gen=gen,
            algo_subset=pareto_algos,
            force_recompute=pareto_force,
        )

    print("=" * 90)
    print("Aα-NSGA-II: COMPLETE PIPELINE (v8.8)")
    print(f"Problems: {len(CONFIG['problems'])} | "
          f"Runs: {CONFIG['n_runs']} | "
          f"Pop: {CONFIG['pop_size']} | Gen: {CONFIG['n_gen']} | "
          f"Workers: {CONFIG['n_workers']} | "
          f"Mode: {'parallel' if CONFIG['use_parallel'] else 'serial'}")
    print(f"CPU count: {mp.cpu_count()}")
    print(f"Environment: {'Jupyter/Colab' if IN_JUPYTER else 'Terminal'}")
    print(f"External comparison mode: {EXTERNAL_COMPARISON_MODE}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Fair HV metric uses α = {FAIR_ALPHA}")
    print("=" * 90)

    for d in [CONFIG['results_dir'], CONFIG['figures_dir'],
              CONFIG['latex_dir'], CONFIG['supp_latex_dir'],
              CONFIG['sigma_hat_dir'], CONFIG['external_dir']]:
        os.makedirs(d, exist_ok=True)
        print(f"  Ensured: {d}")

    if compute_sigma:
        _probe_f2_override()

    _cfg_strict = get_algo_config(CONFIG['pop_size'], CONFIG['n_gen'],
                                   external_mode='strict')
    for name, _, kwargs in _cfg_strict:
        if name in EXTERNAL_ALGO_NAMES_STRICT:
            assert kwargs.get('alpha', FAIR_ALPHA) == FAIR_ALPHA, (
                f"Strict-mode {name} uses alpha={kwargs.get('alpha')}, "
                f"expected FAIR_ALPHA={FAIR_ALPHA}"
            )

    checkpoint_path = os.path.join(CONFIG['results_dir'], 'checkpoint.pkl')

    if not skip_main_runs:
        if CONFIG['use_parallel']:
            all_results = run_parallel(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                n_workers=CONFIG['n_workers'],
                checkpoint_path=checkpoint_path,
                verbose=not CONFIG['quiet'],
                monitor=None,
            )
        else:
            all_results = run_serial(
                CONFIG['problems'], CONFIG['n_runs'], CONFIG['pop_size'],
                CONFIG['n_gen'], CONFIG['seed_base'],
                checkpoint_path=checkpoint_path,
                verbose=True,
                monitor=None,
            )
    else:
        if os.path.exists(checkpoint_path):
            with open(checkpoint_path, 'rb') as f:
                all_results = pickle.load(f)
            print(f"\n[skip_main_runs] Loaded {len(all_results)} problems "
                  f"from checkpoint")
        else:
            all_results = {}
            print("\n[skip_main_runs] No checkpoint found — nothing to do.")

    if all_results and not skip_igd_gd:
        all_results = compute_igd_gd_pass(
            all_results, CONFIG['problems'],
            CONFIG['pop_size'], CONFIG['n_gen'],
            CONFIG['seed_base'], CONFIG['n_runs'],
            verbose=True,
        )
        _save_checkpoint(all_results, checkpoint_path)
    elif skip_igd_gd:
        print("\n[skip_igd_gd] IGD/GD pass skipped.")

    # Pre-flight check to ensure IGD/GD data is present before LaTeX
    if not skip_igd_gd and all_results:
        has_igd = any(
            not np.isnan(res[a].get('igd_mean', np.nan))
            for res in all_results.values() if isinstance(res, dict) and '_error' not in res
            for a in res if not a.startswith('_')
        )
        if not has_igd:
            print("  ⚠ WARNING: No IGD/GD values found in results. LaTeX "
                  "tables will show '--' for IGD/GD columns. Re-run without "
                  "--skip-igd-gd to compute them.")

    stats_fair = run_statistical_tests(
        all_results, CONFIG['problems'], ALGO_NAMES,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES if include_external else [],
        label='[FAIR = ABLATION]',
    )

    strict_algo_names = INTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT
    stats_strict = run_statistical_tests(
        all_results, CONFIG['problems'], strict_algo_names,
        reference=CONFIG['reference_algo'],
        internal_names=INTERNAL_ALGO_NAMES,
        external_names=EXTERNAL_ALGO_NAMES_STRICT if include_external else [],
        label='[STRICT = PRIMARY]',
    )

    report_framing(stats_strict, strict_algo_names)

    generate_all_visualizations(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_strict['bootstrap_cis'], CONFIG['figures_dir'],
        stats_results=stats_strict,
    )

    generate_latex_tables(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        stats_strict, stats_fair, CONFIG['latex_dir']
    )

    generate_per_problem_table(
        all_results, CONFIG['problems'], ALGO_NAMES, CONFIG['families'],
        CONFIG['supp_latex_dir']
    )

    if compute_sigma:
        try:
            generate_sigma_hat_table(
                all_results,
                CONFIG['sigma_hat_problems'],
                CONFIG['sigma_hat_dir'],
            )
        except Exception as e:
            print(f"  Σ̂ computation failed: {e}")
            traceback.print_exc()

    try:
        generate_all_pareto_fronts(
            CONFIG['problems'],
            os.path.join(CONFIG['figures_dir'], 'pareto_fronts'),
            alpha=FAIR_ALPHA,
            quick=pareto_quick,
        )
    except Exception as e:
        print(f"  Pareto front generation failed: {e}")
        traceback.print_exc()

    external_rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo in (EXTERNAL_ALGO_NAMES + EXTERNAL_ALGO_NAMES_STRICT):
            if algo in results:
                external_rows.append({
                    'problem': prob_name,
                    'algorithm': algo,
                    'hv_raw_mean': results[algo]['hv_raw_mean'],
                    'hv_raw_std': results[algo]['hv_raw_std'],
                    'igd_mean': results[algo].get('igd_mean', np.nan),
                    'gd_mean': results[algo].get('gd_mean', np.nan),
                    'igd_norm_mean': results[algo].get('igd_norm_mean', np.nan),
                    'gd_norm_mean': results[algo].get('gd_norm_mean', np.nan),
                })
    if external_rows:
        pd.DataFrame(external_rows).to_csv(
            os.path.join(CONFIG['external_dir'], 'external_results.csv'),
            index=False, encoding='utf-8')

    rows = []
    for prob_name, results in all_results.items():
        if '_error' in results:
            continue
        for algo, res in results.items():
            rows.append({
                'problem': prob_name, 'algorithm': algo,
                'hv_raw_mean': res['hv_raw_mean'],
                'hv_raw_std': res['hv_raw_std'],
                'alpha_mean': res.get('alpha_mean', np.nan),
                'igd_mean': res.get('igd_mean', np.nan),
                'gd_mean': res.get('gd_mean', np.nan),
                'igd_norm_mean': res.get('igd_norm_mean', np.nan),
                'gd_norm_mean': res.get('gd_norm_mean', np.nan),
            })
    if rows:
        pd.DataFrame(rows).to_csv(
            os.path.join(CONFIG['results_dir'], 'results.csv'),
            index=False, encoding='utf-8')

    print("\n" + "=" * 90)
    print("EXPERIMENT COMPLETE")
    print("=" * 90)


def _cli_entry():
    try:
        mp.freeze_support()
    except Exception:
        pass
    import argparse
    parser = argparse.ArgumentParser(
        description='Aα-NSGA-II Pipeline v8.8')
    parser.add_argument('--test', action='store_true')
    parser.add_argument('--serial', action='store_true')
    parser.add_argument('--quiet', action='store_true')
    parser.add_argument('--no-pareto', action='store_true')
    parser.add_argument('--full-pareto', action='store_true')
    parser.add_argument('--no-sigma', action='store_true')
    parser.add_argument('--no-external', action='store_true')
    parser.add_argument('--no-igd-gd', action='store_true')
    parser.add_argument('--external-mode', choices=['fair', 'strict', 'both'],
                        default='both')
    parser.add_argument('--force-parallel', action='store_true')
    parser.add_argument('--workers', type=int, default=None)
    parser.add_argument('--runs', type=int, default=None)
    parser.add_argument('--pop', type=int, default=None)
    parser.add_argument('--gen', type=int, default=None)
    parser.add_argument('--resume', action='store_true')

    parser.add_argument('--pareto-only', action='store_true',
                        help='Generate ONLY the Pareto front figures.')
    parser.add_argument('--pareto-problems', type=str, default=None,
                        help='Comma-separated problems for --pareto-only.')
    parser.add_argument('--pareto-algos',
                        choices=['all', 'ablation', 'fair', 'strict'],
                        default='all')
    parser.add_argument('--pareto-force', action='store_true')
    args, _ = parser.parse_known_args()

    if args.force_parallel:
        globals()['FORCE_PARALLEL_FROM_JUPYTER'] = True

    if args.pareto_only:
        probs = None
        if args.pareto_problems:
            if args.pareto_problems.strip().lower() == 'all':
                probs = PARETO_DEFAULT_PROBLEMS
            else:
                probs = [s.strip() for s in args.pareto_problems.split(',')
                         if s.strip()]
        run_pareto_only(
            problems=probs,
            quick=not args.full_pareto,
            pop=args.pop, gen=args.gen,
            algo_subset=args.pareto_algos,
            force_recompute=args.pareto_force,
        )
        return

    mode = 'test' if args.test else 'full'
    use_parallel = not args.serial

    if args.no_pareto:
        globals()['generate_all_pareto_fronts'] = lambda *a, **kw: None

    main(mode=mode, use_parallel=use_parallel,
         n_workers=args.workers, runs=args.runs,
         pop=args.pop, gen=args.gen,
         quiet=args.quiet,
         pareto_quick=not args.full_pareto,
         compute_sigma=not args.no_sigma,
         include_external=not args.no_external,
         external_mode=args.external_mode,
         skip_main_runs=args.resume,
         skip_igd_gd=args.no_igd_gd,
         pareto_only=False,
         pareto_problems=None,
         pareto_algos='all',
         pareto_force=False)


if __name__ == "__main__":
    _cli_entry()

Aα-NSGA-II: COMPLETE PIPELINE (v8.8)
Problems: 64 | Runs: 10 | Pop: 60 | Gen: 300 | Workers: 8 | Mode: parallel
CPU count: 16
Environment: Jupyter/Colab
External comparison mode: both
Output root: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3
Fair HV metric uses α = 0.75
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\results
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\figures
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt CTRouting Problem as a M2-M PUD\Fuzzy_Aα_NSGA_II-Results_v8_3\supplementary_latex
  Ensured: D:\Container Transportation Routing Problems\Manuscript5-Opt C